In [ ]:
# =============================================================================
#  TDNN-HMM ASR – Bahasa Jawa  [v4 – Perbaikan Bug Kritis]
#
#  PERBAIKAN DARI v3:
#
#  [FIX-1] Decoder insertion masif (I=15533 vs N=2388):
#          Tambah CTC collapse (hapus repetisi frame) SEBELUM lookup lexicon
#          Tambah estimasi panjang kata untuk membatasi output
#
#  [FIX-2] n_hmm_states tidak digunakan:
#          Implementasi HMM DP sebenarnya dengan self-loop + advance transition
#          Setiap fonem punya N state, min-duration = N frame
#
#  [FIX-3] _fonem_seq_ke_kata output tidak terbatas:
#          Batasi max kata output = estimasi dari total fonem / rata fonem per kata
#          Beam pruning lebih ketat
#
#  [FIX-4] TDNNLayer forward tidak konsisten train vs inference:
#          Training frame-level menggunakan (B,1,D) agar melewati jalur (B,T,D)
#          BatchNorm menerima (B, C, T) yang konsisten
#
#  [FIX-5] Alignment per-frame O(T) append sangat lambat:
#          Kumpulkan per-utterance → vstack sekali per fonem
#
#  [FIX-6] Early stopping dihapus:
#          Training berjalan penuh TDNN_EPOCHS, checkpoint best tetap disimpan
#
#  [FIX-7] Class weights di-clip agar fonem langka tidak mendominasi loss
# =============================================================================

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"

CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)

DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\hybrid\src\TDNN-HMM\out_v4"
)

COL_SPEAKER = "speaker_id"

# ── GMM ───────────────────────────────────────────────────────────────────────
N_COMPONENTS_GMM = 16
N_ITER_GMM       = 100
TOL_GMM          = 1e-4
MIN_FRAME_GMM    = 10

# ── TDNN ──────────────────────────────────────────────────────────────────────
TDNN_HIDDEN     = 512
TDNN_DROPOUT    = 0.1
TDNN_EPOCHS     = 50      # [FIX-6] training penuh, tanpa early stopping
TDNN_LR         = 1e-3
TDNN_BATCH_SIZE = 512     # naikkan dari 256 → lebih efisien
TDNN_CONTEXTS   = [
    [-1, 0, 1],
    [-1, 0, 1],
    [-3, 0, 3],
    [-3, 0, 3],
    [-6, 0, 6],
]

# ── HMM ──────────────────────────────────────────────────────────────────────
N_HMM_STATES  = 3
BEAM_WIDTH    = 10

# ── LM ───────────────────────────────────────────────────────────────────────
N_GRAM        = 3
VAL_RATIO     = 0.2
SPEAKER_SPLIT = True

# ── Numerik ───────────────────────────────────────────────────────────────────
LOG_ZERO  = -1e15
LOG_EPS   = 1e-300
COV_REG   = 1e-6
FEAT_EPS  = 1e-8

# =============================================================================
#  IMPORT
# =============================================================================
import os, re, glob, math, time, random, warnings
from functools import lru_cache
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    _TORCH = True
except ImportError:
    raise ImportError("PyTorch diperlukan: pip install torch")

try:
    import librosa; _LIBROSA = True
except ImportError:
    _LIBROSA = False

try:
    from scipy.io import wavfile as _wavfile; _SCIPY = True
except ImportError:
    _SCIPY = False

warnings.filterwarnings("ignore")

# =============================================================================
#  SET FONEM BAHASA JAWA
# =============================================================================
FONEM_JAWA: List[str] = [
    "a", "i", "u", "e", "o", "eu", "ae",
    "p", "b", "t", "d", "k", "g", "dh", "th",
    "m", "n", "ng", "ny",
    "s", "h", "f",
    "j", "c",
    "l", "r",
    "w", "y",
    "SIL",
]
FONEM2ID: Dict[str, int] = {f: i for i, f in enumerate(FONEM_JAWA)}
N_FONEM: int = len(FONEM_JAWA)
SIL_ID: int  = FONEM2ID["SIL"]

_G2P_DIGRAF: Dict[str, str] = {
    "dh": "dh", "th": "th", "ng": "ng", "ny": "ny",
}

@lru_cache(maxsize=8192)
def kata_ke_fonem(kata: str) -> List[str]:
    fonem, i = [], 0
    while i < len(kata):
        if i + 1 < len(kata) and kata[i:i+2] in _G2P_DIGRAF:
            fonem.append(_G2P_DIGRAF[kata[i:i+2]]); i += 2
        elif kata[i] in FONEM2ID:
            fonem.append(kata[i]); i += 1
        else:
            i += 1
    return fonem if fonem else ["SIL"]

@lru_cache(maxsize=4096)
def normalisasi(teks: str) -> str:
    teks = teks.lower().strip()
    teks = re.sub(r"[^a-z\s]", "", teks)
    return re.sub(r"\s+", " ", teks).strip()

def buat_lexicon(transkrip_list: List[str]) -> Dict[str, List[str]]:
    lex: Dict[str, List[str]] = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lex:
                lex[kata] = kata_ke_fonem(kata)
    return lex

# =============================================================================
#  BACA CSV
# =============================================================================
def baca_transkrip_csv(
    csv_path: str,
    col_speaker: Optional[str] = COL_SPEAKER,
) -> Tuple[Dict[str, str], Dict[str, str]]:
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(csv_path)
    df = None
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(csv_path, encoding=enc); break
        except Exception:
            continue
    if df is None:
        raise ValueError(f"Gagal membaca CSV: {csv_path}")
    df.columns = [c.strip() for c in df.columns]
    col_path = next((c for c in ["Path_Data","path_data","path","audio_path","filename"]
                     if c in df.columns), None)
    col_text = next((c for c in ["sentences","sentence","text","transcription","label"]
                     if c in df.columns), None)
    if not col_path: raise ValueError(f"Kolom path tidak ada. Kolom: {list(df.columns)}")
    if not col_text: raise ValueError(f"Kolom teks tidak ada. Kolom: {list(df.columns)}")
    dict_transkrip, dict_speaker = {}, {}
    has_spk = col_speaker and col_speaker in df.columns
    for _, row in df.iterrows():
        pv = str(row[col_path]).strip(); tv = str(row[col_text]).strip()
        if pv in ("nan","") or tv in ("nan",""): continue
        aid = os.path.splitext(os.path.basename(pv))[0]
        dict_transkrip[aid] = tv.lower()
        if has_spk:
            dict_speaker[aid] = str(row[col_speaker]).strip()
    print(f"  CSV dibaca: {len(dict_transkrip)} entri, "
          f"{'dengan' if has_spk else 'tanpa'} kolom speaker")
    return dict_transkrip, dict_speaker

# =============================================================================
#  SPLIT SPEAKER
# =============================================================================
def split_by_speaker(audio_ids, dict_speaker, val_ratio=VAL_RATIO):
    if not dict_speaker:
        random.shuffle(audio_ids)
        n_val = max(1, int(len(audio_ids) * val_ratio))
        return audio_ids[n_val:], audio_ids[:n_val]
    spk2ids = defaultdict(list)
    for aid in audio_ids:
        spk2ids[dict_speaker.get(aid, "unknown")].append(aid)
    speakers = list(spk2ids.keys()); random.shuffle(speakers)
    n_val_spk = max(1, int(len(speakers) * val_ratio))
    val_spk   = set(speakers[:n_val_spk])
    train_ids, val_ids = [], []
    for spk, ids in spk2ids.items():
        (val_ids if spk in val_spk else train_ids).extend(ids)
    print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")
    return train_ids, val_ids

# =============================================================================
#  EKSTRAKSI MFCC + CMVN (39-dim)
# =============================================================================
_SR=16000; _N_MFCC=13; _N_FFT=512; _HOP_LENGTH=160
_WIN_LENGTH=400; _N_MELS=26; _PRE_EMPH=0.97
_FMIN=0.0; _FMAX=4000.0; _DELTA_W=2
FEAT_DIM = _N_MFCC * 3

def apply_cmvn(f: np.ndarray) -> np.ndarray:
    return (f - f.mean(0, keepdims=True)) / (f.std(0, keepdims=True) + FEAT_EPS)

def _delta_numpy(feat: np.ndarray, w: int = _DELTA_W) -> np.ndarray:
    T = feat.shape[0]; den = 2.0 * sum(n*n for n in range(1, w+1))
    pad = np.pad(feat, ((w,w),(0,0)), mode="edge"); d = np.zeros_like(feat)
    for n in range(1, w+1):
        d += n * (pad[w+n:w+n+T] - pad[w-n:w-n+T])
    return d / den

def extract_features(wav_path: str, cmvn: bool = True) -> np.ndarray:
    if not os.path.isfile(wav_path): raise FileNotFoundError(wav_path)
    feat = _mfcc_librosa(wav_path) if _LIBROSA else _mfcc_scipy(wav_path)
    return apply_cmvn(feat) if cmvn else feat

def _mfcc_librosa(wav_path: str) -> np.ndarray:
    y, _ = librosa.load(wav_path, sr=_SR, mono=True)
    if len(y) == 0: raise ValueError(f"Audio kosong: {wav_path}")
    y  = np.append(y[0], y[1:] - _PRE_EMPH * y[:-1])
    m  = librosa.feature.mfcc(y=y, sr=_SR, n_mfcc=_N_MFCC, n_fft=_N_FFT,
                                hop_length=_HOP_LENGTH, win_length=_WIN_LENGTH,
                                n_mels=_N_MELS, fmin=_FMIN, fmax=_FMAX, window="hamming")
    d1 = librosa.feature.delta(m, width=_DELTA_W*2+1)
    d2 = librosa.feature.delta(m, order=2, width=_DELTA_W*2+1)
    return np.vstack([m, d1, d2]).T.astype(np.float32)

def _mfcc_scipy(wav_path: str) -> np.ndarray:
    sr, audio = _wavfile.read(wav_path)
    if audio.dtype == np.int16:   audio = audio.astype(np.float32)/32768.0
    elif audio.dtype == np.int32: audio = audio.astype(np.float32)/2147483648.0
    else:                         audio = audio.astype(np.float32)
    if audio.ndim > 1: audio = audio.mean(1)
    if sr != _SR:
        n_new = int(len(audio)*_SR/sr)
        audio = np.interp(np.linspace(0,len(audio)-1,n_new), np.arange(len(audio)), audio)
    y = np.append(audio[0], audio[1:] - _PRE_EMPH*audio[:-1])
    nf  = max(1, 1+(len(y)-_WIN_LENGTH)//_HOP_LENGTH)
    idx = np.arange(_WIN_LENGTH)[None,:] + _HOP_LENGTH*np.arange(nf)[:,None]
    idx = np.clip(idx, 0, len(y)-1)
    fr  = y[idx].astype(np.float32) * np.hamming(_WIN_LENGTH).astype(np.float32)
    mag = np.abs(np.fft.rfft(fr, n=_N_FFT))
    h2m = lambda f: 1127.0*np.log1p(f/700.0)
    m2h = lambda m: 700.0*(np.exp(m/1127.0)-1.0)
    mp  = np.linspace(h2m(_FMIN), h2m(_FMAX), _N_MELS+2)
    bins= np.floor((_N_FFT+1)*m2h(mp)/_SR).astype(int)
    fb  = np.zeros((_N_MELS,_N_FFT//2+1), dtype=np.float32)
    for m in range(1,_N_MELS+1):
        for k in range(bins[m-1],bins[m]):   fb[m-1,k]=(k-bins[m-1])/max(bins[m]-bins[m-1],1)
        for k in range(bins[m],bins[m+1]):   fb[m-1,k]=(bins[m+1]-k)/max(bins[m+1]-bins[m],1)
    loge = np.log10(np.maximum(mag**2 @ fb.T, 1e-10))
    N    = _N_MELS
    dct  = np.cos(np.pi*np.arange(_N_MFCC)[:,None]*(np.arange(N)+0.5)[None,:]/N)
    mfcc = (loge @ dct.T)*np.sqrt(2.0/N)
    d1=_delta_numpy(mfcc); d2=_delta_numpy(d1)
    return np.concatenate([mfcc,d1,d2],axis=1).astype(np.float32)

def build_mfcc_cache(sampled_root, dict_transkrip):
    wf  = glob.glob(os.path.join(sampled_root,"**/*.wav"), recursive=True)
    wf += glob.glob(os.path.join(sampled_root,"**/*.flac"), recursive=True)
    cache, meta = {}, []
    print(f"  Mengekstrak MFCC dari {len(wf)} file ...")
    t0 = time.time()
    for idx,wp in enumerate(wf,1):
        aid = os.path.splitext(os.path.basename(wp))[0]
        if not dict_transkrip.get(aid,""): continue
        try:
            feat = extract_features(wp, cmvn=True)
            cache[aid] = feat
            meta.append({"Id":aid,"Frames":feat.shape[0],"Dim":feat.shape[1]})
        except Exception as e:
            print(f"    [SKIP] {os.path.basename(wp)}: {e}")
        if idx%100==0:
            print(f"    [{idx}/{len(wf)}] {time.time()-t0:.0f}s elapsed")
    print(f"  Selesai: {len(cache)} audio")
    return cache, meta

# =============================================================================
#  GMM PER FONEM
# =============================================================================
class GaussianMixtureModel:
    def __init__(self, n_components=4, n_iter=100, tol=1e-3,
                 reg_covar=COV_REG, random_state=None):
        self.n_components=n_components; self.n_iter=n_iter
        self.tol=tol; self.reg_covar=reg_covar; self.random_state=random_state
        self.weights_=self.means_=self.covariances_=None
        self.converged_=False; self.n_iter_done_=0
        self.log_likelihood_history_: List[float]=[]

    def _init_params(self, X):
        rng=np.random.default_rng(self.random_state); T,D=X.shape; K=self.n_components
        idx=rng.choice(T,size=K,replace=False)
        self.means_=X[idx].copy(); self.weights_=np.full(K,1.0/K)
        self.covariances_=np.tile(np.var(X,axis=0)+self.reg_covar,(K,1))

    def _log_gaussian_all(self, X):
        mu=self.means_; var=self.covariances_
        log_det=np.sum(np.log(var),axis=1)
        diff=X[:,None,:]-mu[None,:,:]
        maha=np.sum(diff**2/var[None,:,:],axis=2)
        return (-0.5*(X.shape[1]*np.log(2*np.pi)+log_det))[None,:]-0.5*maha

    def _e_step(self, X):
        lw=np.log(self.weights_+LOG_EPS)+self._log_gaussian_all(X)
        lse=np.max(lw,axis=1,keepdims=True)
        ls=lse.squeeze()+np.log(np.sum(np.exp(lw-lse),axis=1))
        return lw-ls[:,None], ls

    def _m_step(self, X, lr):
        r=np.exp(lr); Nk=r.sum(0)+LOG_EPS
        self.weights_=Nk/X.shape[0]; self.means_=(r.T@X)/Nk[:,None]
        diff=X[:,None,:]-self.means_[None,:,:]
        self.covariances_=(np.sum(r[:,:,None]*diff**2,axis=0)/Nk[:,None]+self.reg_covar)

    def fit(self, X):
        assert X.ndim==2; self.n_components=min(self.n_components,X.shape[0])
        self._init_params(X); self.log_likelihood_history_=[]; prev=-np.inf
        for it in range(self.n_iter):
            lr,ls=self._e_step(X); ll=float(ls.mean())
            self.log_likelihood_history_.append(ll)
            if abs(ll-prev)<self.tol: self.converged_=True; self.n_iter_done_=it+1; break
            prev=ll; self._m_step(X,lr)
        else: self.converged_=False; self.n_iter_done_=self.n_iter
        return self

    def log_prob_per_frame(self, X) -> np.ndarray:
        assert X.ndim==2
        lw=np.log(self.weights_+LOG_EPS)+self._log_gaussian_all(X)
        lse=np.max(lw,axis=1,keepdims=True)
        return lse.squeeze()+np.log(np.sum(np.exp(lw-lse),axis=1))

# =============================================================================
#  FORCED ALIGNMENT FONEM
# =============================================================================
def alignment_fonem_viterbi(fitur, fonem_seq, gmm_per_fonem):
    T=fitur.shape[0]; n=len(fonem_seq)
    if n==0 or T==0: return np.full(T,SIL_ID,dtype=np.int32)
    skor=np.full((T,n),LOG_ZERO)
    for j,f in enumerate(fonem_seq):
        if f in gmm_per_fonem: skor[:,j]=gmm_per_fonem[f].log_prob_per_frame(fitur)
        elif f=="SIL": skor[:,j]=0.0
    dp=np.full((T+1,n+1),LOG_ZERO); bp=np.zeros((T+1,n+1),dtype=np.int8)
    dp[0,0]=0.0
    for t in range(1,T+1):
        for j in range(n):
            stay=dp[t-1,j]+skor[t-1,j] if dp[t-1,j]>LOG_ZERO else LOG_ZERO
            adv =(dp[t-1,j-1]+skor[t-1,j]
                  if j>0 and dp[t-1,j-1]>LOG_ZERO else LOG_ZERO)
            if stay>=adv: dp[t,j]=stay; bp[t,j]=0
            else:         dp[t,j]=adv;  bp[t,j]=1
    path=np.zeros(T,dtype=int); j=n-1
    for t in range(T,0,-1):
        path[t-1]=j
        if bp[t,j]==1: j=max(j-1,0)
    return np.array([FONEM2ID.get(fonem_seq[j],SIL_ID) for j in path],dtype=np.int32)


def kumpulkan_frame_fonem_naif(fitur, fonem_seq):
    """Alignment naif (sebelum GMM tersedia)."""
    T,n=fitur.shape[0],len(fonem_seq)
    if n==0: return {}
    nf=(np.ones(n)/n*T).astype(int); nf[:T-nf.sum()]+=1
    hasil={}; idx=0
    for i,f in enumerate(fonem_seq):
        seg=fitur[idx:idx+nf[i]]
        if seg.shape[0]>0:
            hasil[f]=np.vstack([hasil[f],seg]) if f in hasil else seg
        idx+=nf[i]
    return hasil


def kumpulkan_frame_per_fonem(
    cache_mfcc, dict_transkrip, lexicon,
    gmm_per_fonem=None, audio_ids=None,
):
    """
    [FIX-5] Kumpulkan per-utterance, bukan per-frame.
    Menghindari list 798K elemen yang lambat di-vstack.
    """
    fonem_frames: Dict[str, List[np.ndarray]] = defaultdict(list)
    use_gmm = gmm_per_fonem is not None and len(gmm_per_fonem) > 0
    ids = audio_ids if audio_ids else list(cache_mfcc.keys())
    print(f"  Mengumpulkan frame per fonem ({'Viterbi-GMM' if use_gmm else 'naif'}) ...")

    for audio_id in ids:
        fitur = cache_mfcc.get(audio_id)
        label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label: continue
        fonem_seq = []
        for kata in normalisasi(label).split():
            fonem_seq.extend(lexicon.get(kata, kata_ke_fonem(kata)))

        if use_gmm:
            # [FIX-5] kumpulkan segmen per-utterance, bukan per-frame
            labels = alignment_fonem_viterbi(fitur, fonem_seq, gmm_per_fonem)
            # Kelompokkan frame berurutan dengan label sama → satu segment
            t = 0
            while t < len(labels):
                fid = labels[t]; j = t + 1
                while j < len(labels) and labels[j] == fid:
                    j += 1
                fonem_frames[FONEM_JAWA[fid]].append(fitur[t:j])
                t = j
        else:
            pecahan = kumpulkan_frame_fonem_naif(fitur, fonem_seq)
            for f, seg in pecahan.items():
                fonem_frames[f].append(seg)

    # Stack semua segmen per fonem sekali
    result: Dict[str, np.ndarray] = {}
    for f, parts in fonem_frames.items():
        result[f] = np.vstack(parts)

    print(f"  Fonem aktif: {len(result)}")
    for f, arr in sorted(result.items(), key=lambda x: -x[1].shape[0])[:5]:
        print(f"    {f}: {arr.shape[0]} frame")
    return result


def latih_gmm_per_fonem(frame_per_fonem, n_components=N_COMPONENTS_GMM,
                         n_iter=N_ITER_GMM, tol=TOL_GMM, min_frame=MIN_FRAME_GMM,
                         verbose=True):
    gmm_per_fonem = {}; t0 = time.time()
    for i,(fonem,X) in enumerate(sorted(frame_per_fonem.items())):
        if X.shape[0] < min_frame:
            if verbose: print(f"    SKIP {fonem}: hanya {X.shape[0]} frame")
            continue
        gmm = GaussianMixtureModel(
            n_components=min(n_components, X.shape[0]//2),
            n_iter=n_iter, tol=tol, reg_covar=COV_REG,
            random_state=hash(fonem)%100_000,
        )
        gmm.fit(X); gmm_per_fonem[fonem]=gmm
        if verbose:
            ll=round(gmm.log_likelihood_history_[-1],3)
            print(f"    [{i+1:2d}] {fonem:4s}: {X.shape[0]:6d} frame, "
                  f"K={gmm.n_components}, iter={gmm.n_iter_done_}, "
                  f"konvergen={gmm.converged_}, LL={ll}")
    print(f"  GMM selesai: {len(gmm_per_fonem)} fonem dalam {time.time()-t0:.1f}s")
    return gmm_per_fonem

# =============================================================================
#  [FIX-4] TDNN LAYER — KONSISTEN TRAIN & INFERENCE
#  Selalu menggunakan jalur (B, T, D) dengan T=1 saat frame-level
# =============================================================================
class TDNNLayer(nn.Module):
    def __init__(self, input_dim, output_dim, context, dropout=0.0):
        super().__init__()
        self.context = sorted(context)
        self.linear  = nn.Linear(input_dim * len(context), output_dim)
        self.bn      = nn.BatchNorm1d(output_dim)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        [FIX-4] Selalu gunakan jalur sequence (B, T, D).
        Jika input (B, D), reshape ke (B, 1, D) terlebih dahulu.
        """
        squeezed = False
        if x.dim() == 2:
            x = x.unsqueeze(1)  # (B, D) → (B, 1, D)
            squeezed = True

        B, T, D = x.shape
        ctx_min  = min(self.context)
        ctx_max  = max(self.context)
        pad_l    = max(0, -ctx_min)
        pad_r    = max(0,  ctx_max)
        # pad dimensi waktu: F.pad bekerja dari dimensi terakhir
        x_pad    = F.pad(x.permute(0,2,1), (pad_l, pad_r)).permute(0,2,1)  # (B, T+pad, D)

        frames = []
        for c in self.context:
            shift = c - ctx_min
            frames.append(x_pad[:, shift:shift+T, :])
        spliced = torch.cat(frames, dim=-1)   # (B, T, D*C)

        out = self.linear(spliced)            # (B, T, out_dim)
        # BatchNorm1d butuh (B, C, T)
        out = self.bn(out.permute(0,2,1)).permute(0,2,1)
        out = F.relu(out)
        out = self.drop(out)

        if squeezed:
            out = out.squeeze(1)  # (B, 1, out_dim) → (B, out_dim)
        return out


class TDNNAcousticModel(nn.Module):
    """
    TDNN memprediksi posterior FONEM per frame.
    Output: log-softmax (N_FONEM).
    """
    def __init__(self, input_dim=FEAT_DIM, n_classes=N_FONEM,
                 hidden_dim=TDNN_HIDDEN, contexts=None, dropout=TDNN_DROPOUT):
        super().__init__()
        if contexts is None: contexts = TDNN_CONTEXTS
        layers=[]; in_dim=input_dim
        for ctx in contexts:
            layers.append(TDNNLayer(in_dim, hidden_dim, ctx, dropout))
            in_dim=hidden_dim
        self.tdnn_layers=nn.ModuleList(layers)
        self.classifier =nn.Linear(hidden_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.tdnn_layers:
            x = layer(x)
        return F.log_softmax(self.classifier(x), dim=-1)

    def get_posteriors_sequence(self, x: torch.Tensor) -> torch.Tensor:
        """x: (1, T, D) → (T, N_FONEM)."""
        self.eval()
        with torch.no_grad():
            return self.forward(x)[0]

# =============================================================================
#  DATASET FRAME-LEVEL
# =============================================================================
class FrameLevelDataset(Dataset):
    def __init__(self, audio_ids, cache_mfcc, dict_transkrip, lexicon, gmm_per_fonem):
        self.frames: List[torch.Tensor] = []
        self.labels: List[int]          = []
        n_skip = 0
        for audio_id in audio_ids:
            fitur = cache_mfcc.get(audio_id)
            label = dict_transkrip.get(audio_id,"")
            if fitur is None or not label: n_skip+=1; continue
            fonem_seq=[]
            for kata in normalisasi(label).split():
                fonem_seq.extend(lexicon.get(kata, kata_ke_fonem(kata)))
            if not fonem_seq: n_skip+=1; continue
            lbls=alignment_fonem_viterbi(fitur, fonem_seq, gmm_per_fonem)
            for t in range(fitur.shape[0]):
                self.frames.append(torch.from_numpy(fitur[t]))
                self.labels.append(int(lbls[t]))
        print(f"  FrameLevelDataset: {len(self.frames):,} frame "
              f"dari {len(audio_ids)-n_skip} audio ({n_skip} skip)")

    def __len__(self): return len(self.frames)
    def __getitem__(self, i): return self.frames[i], self.labels[i]

# =============================================================================
#  [FIX-6] TRAINING TDNN — TANPA EARLY STOPPING
#  Training penuh TDNN_EPOCHS, best checkpoint tetap disimpan
# =============================================================================
def latih_tdnn_frame_level(
    train_ids, val_ids, cache_mfcc, dict_transkrip,
    lexicon, gmm_per_fonem, dir_save,
    epochs=TDNN_EPOCHS, lr=TDNN_LR, batch_size=TDNN_BATCH_SIZE,
    hidden_dim=TDNN_HIDDEN, contexts=None, dropout=TDNN_DROPOUT,
):
    """
    [FIX-6] Early stopping DIHAPUS.
    Training berjalan penuh `epochs` epoch.
    Checkpoint best (val_loss terendah) tetap disimpan untuk evaluasi.
    """
    if contexts is None: contexts = TDNN_CONTEXTS
    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = os.path.join(dir_save, "model_tdnn_fonem.pt")

    print(f"\n  Training TDNN (fonem-level) pada: {device}")
    print(f"  N_FONEM={N_FONEM} | hidden={hidden_dim} | epochs={epochs}")
    print(f"  [INFO] Early stopping DINONAKTIFKAN — training penuh {epochs} epoch")

    model = TDNNAcousticModel(
        input_dim=FEAT_DIM, n_classes=N_FONEM,
        hidden_dim=hidden_dim, contexts=contexts, dropout=dropout,
    ).to(device)

    # Cek checkpoint lama
    if os.path.isfile(model_path):
        try:
            model.load_state_dict(torch.load(model_path, map_location=device))
            print(f"  Checkpoint dimuat: {model_path}"); return model
        except Exception as e:
            print(f"  Checkpoint korup ({e}), training ulang ..."); os.remove(model_path)

    print("  Membangun train dataset ...")
    train_ds = FrameLevelDataset(train_ids, cache_mfcc, dict_transkrip,
                                  lexicon, gmm_per_fonem)
    print("  Membangun val dataset ...")
    val_ds   = FrameLevelDataset(val_ids, cache_mfcc, dict_transkrip,
                                  lexicon, gmm_per_fonem)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=0, pin_memory=False)

    # [FIX-7] Class weights di-clip agar fonem langka tidak terlalu dominan
    label_counts = np.zeros(N_FONEM)
    for _, lbl in train_ds:
        label_counts[lbl] += 1
    label_counts = np.maximum(label_counts, 1)
    raw_weights  = 1.0 / label_counts
    # Clip: weight maksimum tidak boleh lebih dari 10x rata-rata
    clip_val     = raw_weights.mean() * 10.0
    clipped_w    = np.clip(raw_weights, 0, clip_val)
    weights      = torch.tensor(clipped_w / clipped_w.sum(), dtype=torch.float32).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, verbose=False)
    loss_fn   = nn.NLLLoss(weight=weights)

    riwayat       = []
    best_val_loss = float("inf")
    t0            = time.time()

    for epoch in range(1, epochs + 1):
        # ── Training ──────────────────────────────────────────────────────────
        model.train()
        total_loss=0.0; n_correct=0; n_total=0
        for feats, labels in train_loader:
            feats  = feats.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            log_probs = model(feats)          # (B, N_FONEM) via [FIX-4]
            loss = loss_fn(log_probs, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * len(labels)
            n_correct  += (log_probs.argmax(1) == labels).sum().item()
            n_total    += len(labels)

        # ── Validasi ──────────────────────────────────────────────────────────
        model.eval()
        val_loss=0.0; val_correct=0; val_total=0
        with torch.no_grad():
            for feats, labels in val_loader:
                feats=feats.to(device); labels=labels.to(device)
                lp = model(feats)
                val_loss    += loss_fn(lp, labels).item() * len(labels)
                val_correct += (lp.argmax(1) == labels).sum().item()
                val_total   += len(labels)

        avg_tr  = total_loss / max(n_total,1)
        avg_val = val_loss   / max(val_total,1)
        acc_tr  = 100*n_correct  / max(n_total,1)
        acc_val = 100*val_correct / max(val_total,1)
        scheduler.step(avg_val)

        riwayat.append({"Epoch":epoch,"Train Loss":round(avg_tr,5),
                         "Val Loss":round(avg_val,5),
                         "Train Acc (%)":round(acc_tr,2),
                         "Val Acc (%)":round(acc_val,2)})
        print(f"  Epoch {epoch:3d}/{epochs}  "
              f"train_loss={avg_tr:.4f} acc={acc_tr:.1f}%  "
              f"val_loss={avg_val:.4f} acc={acc_val:.1f}%  "
              f"lr={optimizer.param_groups[0]['lr']:.2e}  "
              f"{time.time()-t0:.0f}s")

        # Simpan best checkpoint tanpa menghentikan training
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), model_path)
            print(f"    → Best checkpoint disimpan (val_loss={avg_val:.5f})")

    # Muat best model setelah training selesai penuh
    if os.path.isfile(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"  Best model dimuat kembali (val_loss={best_val_loss:.5f})")

    pd.DataFrame(riwayat).to_csv(
        os.path.join(dir_save,"HasilTrainingTDNN_Fonem.csv"), index=False)
    return model

# =============================================================================
#  N-GRAM LM
# =============================================================================
class NGramModel:
    def __init__(self, n=3, discount=0.75):
        self.n=n; self.discount=discount
        self.ngram_count=defaultdict(int); self.context_count=defaultdict(int)
        self.vocab=set(); self.N_total=0; self._ctx_div={}

    def _tok(self, s):
        t=normalisasi(s).split(); return ["<s>"]*(self.n-1)+t+["</s>"]

    def fit(self, korpus):
        cw=defaultdict(set)
        for s in korpus:
            t=self._tok(s); self.vocab.update(t); self.N_total+=len(t)
            for i in range(len(t)-self.n+1):
                ng=tuple(t[i:i+self.n]); c=ng[:-1]
                self.ngram_count[ng]+=1; self.context_count[c]+=1; cw[c].add(ng[-1])
        self._ctx_div={c:len(w) for c,w in cw.items()}; return self

    def log_prob(self, kata, ctx):
        ng=ctx+(kata,); Cng=self.ngram_count.get(ng,0); Cc=self.context_count.get(ctx,0)
        if Cc==0:
            Cu=self.ngram_count.get((kata,),0)
            return math.log((Cu+1)/(self.N_total+len(self.vocab))+LOG_EPS)
        lam=self.discount*self._ctx_div.get(ctx,0)/Cc
        Cu=self.ngram_count.get((kata,),0); pb=(Cu+1)/(self.N_total+len(self.vocab))
        return math.log(max(Cng-self.discount,0)/Cc+lam*pb+LOG_EPS)

    def score_seq(self, words):
        t=["<s>"]*(self.n-1)+words+["</s>"]; lp=0.0
        for i in range(self.n-1,len(t)):
            lp+=self.log_prob(t[i],tuple(t[i-self.n+1:i]))
        return lp

    def rescore(self, hyps): return max(hyps, key=self.score_seq) if hyps else []

# =============================================================================
#  [FIX-1 & FIX-2 & FIX-3] HMM-VITERBI DECODER — PERBAIKAN TOTAL
#
#  v3 BUG: argmax per frame → ribuan fonem → ribuan kata (I=15533)
#  v4 FIX:
#    1. CTC collapse: hapus repetisi frame → fonem sequence jauh lebih pendek
#    2. HMM DP sebenarnya: self-loop + advance, min-duration = n_hmm_states
#    3. Batasi panjang kata output berdasarkan estimasi panjang kalimat
# =============================================================================
class HMMViterbiDecoder:
    def __init__(self, model, lexicon, ngram_model, gmm_per_fonem,
                 device=None, n_hmm_states=N_HMM_STATES, beam_width=BEAM_WIDTH):
        self.model         = model
        self.lexicon       = lexicon
        self.rev_lex: Dict[tuple, List[str]] = defaultdict(list)
        for kata, fonems in lexicon.items():
            self.rev_lex[tuple(fonems)].append(kata)
        # Hitung rata-rata panjang fonem per kata untuk estimasi
        lengths = [len(v) for v in lexicon.values()]
        self.avg_fonem_per_kata = float(np.mean(lengths)) if lengths else 3.0
        self.ngram         = ngram_model
        self.gmm_per_fonem = gmm_per_fonem
        self.device        = device or torch.device("cpu")
        self.n_hmm_states  = n_hmm_states
        self.beam_width    = beam_width
        self.model.to(self.device); self.model.eval()

    def _get_posteriors(self, fitur: np.ndarray) -> np.ndarray:
        t = torch.from_numpy(fitur).unsqueeze(0).to(self.device)  # (1, T, D)
        return self.model.get_posteriors_sequence(t).cpu().numpy()  # (T, N_FONEM)

    def _ctc_collapse(self, fonem_ids: np.ndarray) -> List[str]:
        """
        [FIX-1] CTC collapse: hapus frame repetisi berurutan.
        Ini adalah perbaikan UTAMA — tanpa ini output membengkak ribuan kata.
        Contoh: [a,a,a,b,b,a,a,k,k] → [a,b,a,k]
        """
        collapsed = []
        prev = -1
        for fid in fonem_ids:
            if fid != prev:
                if fid != SIL_ID:
                    collapsed.append(FONEM_JAWA[fid])
                prev = fid
        return collapsed

    def _hmm_viterbi_sequence(self, log_post: np.ndarray) -> np.ndarray:
        """
        [FIX-2] HMM Viterbi sebenarnya dengan N state per fonem.
        State: (fonem_id, state_in_fonem 0..N-1)
        Transisi: self-loop ATAU advance ke state berikutnya dalam fonem
                  ATAU advance ke fonem berikutnya (hanya dari state terakhir)
        Ini memastikan setiap fonem bertahan minimal n_hmm_states frame.

        Untuk decoding bebas (tidak ada fonem sequence yang diketahui):
        kita gunakan N state per fonem aktif secara paralel.
        """
        T     = log_post.shape[0]
        N     = self.n_hmm_states
        F     = N_FONEM
        p_self    = math.log(0.7)   # probabilitas self-loop
        p_advance = math.log(0.3)   # probabilitas maju ke state berikutnya

        # State: (fonem, state_dalam_fonem)  total = F * N state
        n_states = F * N
        def sid(f, s): return f * N + s

        vit = np.full((T, n_states), LOG_ZERO)
        bt  = np.zeros((T, n_states), dtype=np.int32)

        # Inisialisasi: semua fonem bisa jadi state awal (state 0)
        for f in range(F):
            vit[0, sid(f, 0)] = log_post[0, f]

        for t in range(1, T):
            for f in range(F):
                for s in range(N):
                    cur = sid(f, s)
                    # Opsi 1: self-loop dari state yang sama
                    score_stay = (vit[t-1, cur] + p_self + log_post[t, f]
                                  if vit[t-1, cur] > LOG_ZERO else LOG_ZERO)
                    # Opsi 2: advance dari state sebelumnya dalam fonem yang sama
                    score_adv = LOG_ZERO
                    if s > 0:
                        prev_s = sid(f, s-1)
                        if vit[t-1, prev_s] > LOG_ZERO:
                            score_adv = vit[t-1, prev_s] + p_advance + log_post[t, f]
                    # Opsi 3: transisi dari fonem lain (state terakhir fonem lain)
                    score_trans = LOG_ZERO
                    if s == 0:
                        for f2 in range(F):
                            prev_last = sid(f2, N-1)
                            if vit[t-1, prev_last] > LOG_ZERO:
                                cand = vit[t-1, prev_last] + log_post[t, f]
                                if cand > score_trans:
                                    score_trans = cand
                                    bt[t, cur] = prev_last

                    best = max(score_stay, score_adv, score_trans)
                    if best > LOG_ZERO:
                        vit[t, cur] = best
                        if best == score_stay:   bt[t, cur] = cur
                        elif best == score_adv:  bt[t, cur] = sid(f, s-1)

        # Traceback
        last_state = int(np.argmax(vit[T-1]))
        path = [last_state]
        for t in range(T-1, 0, -1):
            path.append(int(bt[t, path[-1]]))
        path.reverse()

        return np.array([p // N for p in path], dtype=np.int32)  # fonem_id per frame

    def _fonem_seq_ke_kata(self, fonem_seq: List[str],
                            max_kata: int = None) -> List[List[str]]:
        """
        [FIX-3] Konversi fonem sequence → kata sequence dengan batas panjang.
        max_kata membatasi panjang output agar tidak membengkak.
        """
        n = len(fonem_seq)
        if n == 0: return [[]]

        # [FIX-3] Estimasi panjang kalimat dari panjang fonem sequence
        if max_kata is None:
            max_kata = max(1, int(n / self.avg_fonem_per_kata) + 2)

        dp: List[List[Tuple[float, List[str]]]] = [[] for _ in range(n+1)]
        dp[0] = [(0.0, [])]

        for i in range(n):
            if not dp[i]: continue
            for j in range(i+1, min(n+1, i+10)):   # max 9 fonem per kata
                sub = tuple(fonem_seq[i:j])
                if sub in self.rev_lex:
                    for kata in self.rev_lex[sub]:
                        for score, words in dp[i]:
                            # [FIX-3] Hanya tambah jika belum melebihi max_kata
                            if len(words) >= max_kata: continue
                            new_words = words + [kata]
                            new_score = score + self.ngram.log_prob(
                                kata, tuple(words[-(self.ngram.n-1):]))
                            dp[j].append((new_score, new_words))
                # Beam pruning ketat
                if len(dp[j]) > self.beam_width * 2:
                    dp[j].sort(key=lambda x: x[0], reverse=True)
                    dp[j] = dp[j][:self.beam_width]

        # Ambil kandidat dari posisi akhir yang paling mendekati fonem sequence
        best_pos = n
        while best_pos > 0 and not dp[best_pos]:
            best_pos -= 1

        if dp[best_pos]:
            dp[best_pos].sort(key=lambda x: x[0], reverse=True)
            return [w for _, w in dp[best_pos][:self.beam_width] if w]

        return [fonem_seq]   # fallback

    def dekode(self, fitur: np.ndarray) -> List[str]:
        """
        [FIX-1,2,3] Pipeline decode v4:
          MFCC → TDNN posterior → HMM Viterbi fonem ID
          → CTC collapse → fonem string sequence
          → kata sequence (bounded) → LM rescore
        """
        if fitur.shape[0] == 0: return []

        # Step 1: TDNN posterior (T, N_FONEM)
        log_post = self._get_posteriors(fitur)

        # Step 2: [FIX-2] HMM Viterbi → fonem_id per frame
        fonem_ids = self._hmm_viterbi_sequence(log_post)

        # Step 3: [FIX-1] CTC collapse → fonem string sequence
        fonem_seq = self._ctc_collapse(fonem_ids)
        if not fonem_seq: return []

        # Step 4: [FIX-3] Fonem → kata (bounded)
        n_kata_est = max(1, int(len(fonem_seq) / self.avg_fonem_per_kata) + 2)
        kata_kandidat = self._fonem_seq_ke_kata(fonem_seq, max_kata=n_kata_est)
        if not kata_kandidat: return fonem_seq

        # Step 5: LM rescore
        return self.ngram.rescore(kata_kandidat)

# =============================================================================
#  WER
# =============================================================================
def compute_wer(ref, hyp):
    R,H=len(ref),len(hyp); dp=[[0]*(H+1) for _ in range(R+1)]
    for i in range(R+1): dp[i][0]=i
    for j in range(H+1): dp[0][j]=j
    for i in range(1,R+1):
        for j in range(1,H+1):
            if ref[i-1]==hyp[j-1]: dp[i][j]=dp[i-1][j-1]
            else: dp[i][j]=1+min(dp[i-1][j-1],dp[i-1][j],dp[i][j-1])
    S=D=I=0; i,j=R,H
    while i>0 or j>0:
        if i>0 and j>0 and ref[i-1]==hyp[j-1]: i-=1;j-=1
        elif i>0 and j>0 and dp[i][j]==dp[i-1][j-1]+1: S+=1;i-=1;j-=1
        elif i>0 and dp[i][j]==dp[i-1][j]+1: D+=1;i-=1
        else: I+=1;j-=1
    N=R; wer=(S+D+I)/N if N>0 else 0.0
    return {"wer_persen":round(wer*100,2),"S":S,"D":D,"I":I,"N":N}


def dekode_semua_audio(cache_mfcc, audio_ids, dict_transkrip, decoder, verbose=True):
    hasil=[]; t0=time.time()
    for idx,audio_id in enumerate(audio_ids,1):
        fitur=cache_mfcc.get(audio_id); label=dict_transkrip.get(audio_id,"")
        if fitur is None or not label: continue
        ref=normalisasi(label).split()
        if not ref: continue
        try:
            hyp=decoder.dekode(fitur)
        except Exception as e:
            if verbose: print(f"    [ERROR] {audio_id}: {e}")
            continue
        wr=compute_wer(ref,hyp)
        hasil.append({"Id":audio_id,"Referensi":" ".join(ref),
                       "Hipotesis":" ".join(hyp),"WER (%)":wr["wer_persen"],
                       "S":wr["S"],"D":wr["D"],"I":wr["I"],"N":wr["N"]})
        if verbose and idx%100==0:
            print(f"    [{idx}/{len(audio_ids)}] {time.time()-t0:.1f}s")
    return hasil


def ringkasan_wer(df, label=""):
    if df.empty: print("  [WARNING] Tidak ada data WER"); return {}
    S=int(df.S.sum()); D=int(df.D.sum()); I=int(df.I.sum()); N=int(df.N.sum())
    wc=(S+D+I)/N*100 if N>0 else 100.0
    print(f"\n  === Ringkasan WER: {label} ===")
    print(f"  Jumlah audio : {len(df)}")
    print(f"  WER Corpus   : {wc:.2f}%  (S={S} D={D} I={I} N={N})")
    print(f"  WER Mean     : {df['WER (%)'].mean():.2f}%")
    print(f"  WER Median   : {df['WER (%)'].median():.2f}%")
    print(f"  WER Std      : {df['WER (%)'].std():.2f}%")
    return {"wer_corpus":round(wc,2),"wer_mean":round(df['WER (%)'].mean(),2),
            "wer_median":round(df['WER (%)'].median(),2),
            "wer_std":round(df['WER (%)'].std(),2),
            "total_S":S,"total_D":D,"total_I":I,"total_N":N}

# =============================================================================
#  MAIN
# =============================================================================
def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    random.seed(42); np.random.seed(42)
    t_start = time.time()

    # [1] Baca data
    print("\n[1] Membaca transkrip ...")
    dict_transkrip, dict_speaker = baca_transkrip_csv(CSV_PATH, COL_SPEAKER)
    semua_transkrip = list(dict_transkrip.values())

    # [2] Lexicon
    print("\n[2] Membangun lexicon Bahasa Jawa ...")
    lexicon = buat_lexicon(semua_transkrip)
    pd.DataFrame([{"Kata":k,"Fonem":" ".join(v)} for k,v in lexicon.items()]
                 ).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)
    print(f"  {len(lexicon)} kata unik | {N_FONEM} fonem")

    # [3] MFCC
    print("\n[3] Ekstraksi MFCC + CMVN (39-dim) ...")
    cache_mfcc, meta_list = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    pd.DataFrame(meta_list).to_csv(f"{DIR_SAVE}/HasilMFCC.csv", index=False)
    if not cache_mfcc: print("  [ERROR] Tidak ada audio."); return

    # [4] Split speaker
    print("\n[4] Split train/val berdasarkan speaker ...")
    all_ids = list(cache_mfcc.keys())
    if SPEAKER_SPLIT and dict_speaker:
        train_ids, val_ids = split_by_speaker(all_ids, dict_speaker, VAL_RATIO)
    else:
        random.shuffle(all_ids)
        n_val = max(1, int(len(all_ids)*VAL_RATIO))
        train_ids, val_ids = all_ids[n_val:], all_ids[:n_val]

    # [5] GMM per fonem (2 iterasi alignment)
    print("\n[5a] Kumpulkan frame per fonem (alignment naif) ...")
    fpf_naif = kumpulkan_frame_per_fonem(
        {a: cache_mfcc[a] for a in train_ids},
        dict_transkrip, lexicon, gmm_per_fonem=None, audio_ids=train_ids)

    print("\n[5b] Latih GMM per fonem (iterasi 1) ...")
    gmm_pk = latih_gmm_per_fonem(fpf_naif)

    print("\n[5c] Re-alignment dengan GMM Viterbi ...")
    fpf_gmm = kumpulkan_frame_per_fonem(
        {a: cache_mfcc[a] for a in train_ids},
        dict_transkrip, lexicon, gmm_per_fonem=gmm_pk, audio_ids=train_ids)

    print("\n[5d] Re-latih GMM per fonem (iterasi 2) ...")
    gmm_pk = latih_gmm_per_fonem(fpf_gmm)

    gmm_stat=[]
    for fonem,gmm in gmm_pk.items():
        X=fpf_gmm.get(fonem)
        gmm_stat.append({"Fonem":fonem,"Frames":X.shape[0] if X is not None else 0,
                          "K":gmm.n_components,"Iter":gmm.n_iter_done_,
                          "Conv":gmm.converged_,
                          "LL":round(gmm.log_likelihood_history_[-1],4)
                          if gmm.log_likelihood_history_ else None})
    pd.DataFrame(gmm_stat).to_csv(f"{DIR_SAVE}/HasilGMM_Fonem.csv", index=False)

    # [6] LM (hanya dari train set)
    print("\n[6] Melatih language model 3-gram ...")
    ngram_model = NGramModel(n=N_GRAM, discount=0.75)
    ngram_model.fit([dict_transkrip[a] for a in train_ids if a in dict_transkrip])
    print(f"  Vocab LM: {len(ngram_model.vocab)} token")

    # [7] Training TDNN — TANPA EARLY STOPPING [FIX-6]
    print("\n[7] Training TDNN akustik (fonem-level, tanpa early stopping) ...")
    tdnn_model = latih_tdnn_frame_level(
        train_ids=train_ids, val_ids=val_ids,
        cache_mfcc=cache_mfcc, dict_transkrip=dict_transkrip,
        lexicon=lexicon, gmm_per_fonem=gmm_pk, dir_save=DIR_SAVE,
    )

    # [8] Decode + WER
    print("\n[8] Dekode val set dengan TDNN-HMM Viterbi (v4) ...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    decoder = HMMViterbiDecoder(
        model=tdnn_model, lexicon=lexicon, ngram_model=ngram_model,
        gmm_per_fonem=gmm_pk, device=device,
        n_hmm_states=N_HMM_STATES, beam_width=BEAM_WIDTH,
    )

    hasil_wer = dekode_semua_audio(cache_mfcc, val_ids, dict_transkrip, decoder)
    df_wer = pd.DataFrame(hasil_wer)
    df_wer.to_csv(f"{DIR_SAVE}/HasilWER_TDNN_HMM_v4.csv", index=False)
    stat = ringkasan_wer(df_wer, label=f"TDNN-HMM v4 ({N_HMM_STATES}-state)")
    pd.DataFrame([stat]).to_csv(f"{DIR_SAVE}/HasilRingkasanWER.csv", index=False)

    # [9] Eksperimen variasi HMM states
    print("\n[9] Eksperimen variasi HMM states (3, 4, 5) ...")
    all_stat = []
    for n_states in [3, 4, 5]:
        dec = HMMViterbiDecoder(
            model=tdnn_model, lexicon=lexicon, ngram_model=ngram_model,
            gmm_per_fonem=gmm_pk, device=device,
            n_hmm_states=n_states, beam_width=BEAM_WIDTH,
        )
        hw  = dekode_semua_audio(cache_mfcc, val_ids, dict_transkrip,
                                  dec, verbose=False)
        df  = pd.DataFrame(hw)
        s   = ringkasan_wer(df, label=f"{n_states}-state HMM")
        s["n_hmm_states"] = n_states
        all_stat.append(s)
    pd.DataFrame(all_stat).to_csv(
        f"{DIR_SAVE}/HasilEksperimenHMMStates.csv", index=False)

    print(f"\n  Total waktu: {(time.time()-t_start)/60:.1f} menit")
    print(f"  Output: {DIR_SAVE}")
    for f in ["HasilLexicon","HasilMFCC","HasilGMM_Fonem","HasilTrainingTDNN_Fonem",
              "HasilWER_TDNN_HMM_v4","HasilRingkasanWER","HasilEksperimenHMMStates"]:
        p=f"{DIR_SAVE}/{f}.csv"
        print(f"  {'OK' if os.path.isfile(p) else '--'}  {f}.csv")


if __name__ == "__main__":
    main()

In [3]:
SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"

CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)

DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\hybrid\src\TDNN-HMM\out"
)

COL_SPEAKER         = "speaker_id"
N_COMPONENTS_GMM    = 16
N_ITER_GMM          = 100
TOL_GMM             = 1e-4
MIN_FRAME_GMM       = 10

TDNN_HIDDEN         = 512
TDNN_DROPOUT        = 0.1
TDNN_EPOCHS         = 50
TDNN_LR             = 1e-3
TDNN_BATCH_SIZE     = 256
TDNN_CONTEXTS       = [
    [-1, 0, 1],
    [-1, 0, 1],
    [-3, 0, 3],
    [-3, 0, 3],
    [-6, 0, 6],
]

N_HMM_STATES        = 3
N_GRAM              = 3
BEAM_WIDTH          = 10
VAL_RATIO           = 0.2
SPEAKER_SPLIT       = True
MAX_FONEM_PER_KATA  = 8
MIN_FONEM_DUR       = 2
AVG_FRAMES_PER_WORD = 30

LOG_ZERO  = -1e15
LOG_EPS   = 1e-300
COV_REG   = 1e-6
FEAT_EPS  = 1e-8

import os, re, glob, math, time, random, warnings
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    _TORCH = True
except ImportError:
    raise ImportError("PyTorch diperlukan: pip install torch")

try:
    import librosa
    _LIBROSA = True
except ImportError:
    _LIBROSA = False

try:
    from scipy.io import wavfile as _wavfile
    _SCIPY = True
except ImportError:
    _SCIPY = False


FONEM_JAWA: List[str] = [
    "a", "i", "u", "e", "o", "eu", "ae",
    "p", "b", "t", "d", "k", "g", "dh", "th",
    "m", "n", "ng", "ny",
    "s", "h", "f",
    "j", "c",
    "l", "r",
    "w", "y",
    "SIL",
]
FONEM2ID: Dict[str, int] = {f: i for i, f in enumerate(FONEM_JAWA)}
N_FONEM: int = len(FONEM_JAWA)
SIL_ID: int  = FONEM2ID["SIL"]

_G2P_DIGRAF: Dict[str, str] = {
    "dh": "dh", "th": "th", "ng": "ng", "ny": "ny",
}

_kata_fonem_cache: Dict[str, Tuple[str, ...]] = {}
_normalisasi_cache: Dict[str, str] = {}


def kata_ke_fonem(kata: str) -> Tuple[str, ...]:
    """
    FIX-8: return tuple (immutable) — cache manual dict menggantikan
    lru_cache yang tidak aman untuk mutable return value.
    """
    if kata in _kata_fonem_cache:
        return _kata_fonem_cache[kata]
    fonem: List[str] = []
    i = 0
    while i < len(kata):
        if i + 1 < len(kata) and kata[i:i+2] in _G2P_DIGRAF:
            fonem.append(_G2P_DIGRAF[kata[i:i+2]])
            i += 2
        elif kata[i] in FONEM2ID:
            fonem.append(kata[i])
            i += 1
        else:
            i += 1
    result = tuple(fonem) if fonem else ("SIL",)
    _kata_fonem_cache[kata] = result
    return result


def normalisasi(teks: str) -> str:
    if teks in _normalisasi_cache:
        return _normalisasi_cache[teks]
    t = teks.lower().strip()
    t = re.sub(r"[^a-z\s]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    _normalisasi_cache[teks] = t
    return t


def buat_lexicon(transkrip_list: List[str]) -> Dict[str, List[str]]:
    lex: Dict[str, List[str]] = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lex:
                lex[kata] = list(kata_ke_fonem(kata))
    return lex


def baca_transkrip_csv(
    csv_path   : str,
    col_speaker: Optional[str] = COL_SPEAKER,
) -> Tuple[Dict[str, str], Dict[str, str]]:
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"CSV tidak ditemukan: {csv_path}")

    df = None
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(csv_path, encoding=enc)
            break
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue
    if df is None:
        raise ValueError(f"Gagal membaca CSV: {csv_path}")

    df.columns = [c.strip() for c in df.columns]

    col_path_cands = ["Path_Data", "path_data", "path", "audio_path",
                      "filename", "file", "wav_path"]
    col_text_cands = ["sentences", "sentence", "text", "transcription",
                      "label", "transcript"]

    col_path = next((c for c in col_path_cands if c in df.columns), None)
    col_text = next((c for c in col_text_cands if c in df.columns), None)
    if col_path is None:
        raise ValueError(f"Kolom path tidak ditemukan. Kolom: {list(df.columns)}")
    if col_text is None:
        raise ValueError(f"Kolom transkrip tidak ditemukan. Kolom: {list(df.columns)}")

    dict_transkrip: Dict[str, str] = {}
    dict_speaker:   Dict[str, str] = {}
    has_speaker = bool(col_speaker and col_speaker in df.columns)

    for _, row in df.iterrows():
        pv = str(row[col_path]).strip()
        tv = str(row[col_text]).strip()
        if pv in ("nan", "") or tv in ("nan", ""):
            continue
        audio_id = os.path.splitext(os.path.basename(pv))[0]
        dict_transkrip[audio_id] = tv.lower()
        if has_speaker:
            dict_speaker[audio_id] = str(row[col_speaker]).strip()

    return dict_transkrip, dict_speaker


def split_by_speaker(
    audio_ids  : List[str],
    dict_speaker: Dict[str, str],
    val_ratio  : float = VAL_RATIO,
) -> Tuple[List[str], List[str]]:
    """FIX-16: salin list sebelum shuffle agar tidak memodifikasi argumen caller."""
    ids = list(audio_ids)

    if not dict_speaker:
        random.shuffle(ids)
        n_val = max(1, int(len(ids) * val_ratio))
        return ids[n_val:], ids[:n_val]

    spk2ids: Dict[str, List[str]] = defaultdict(list)
    for aid in ids:
        spk = dict_speaker.get(aid, "unknown")
        spk2ids[spk].append(aid)

    speakers  = list(spk2ids.keys())
    random.shuffle(speakers)
    n_val_spk = max(1, int(len(speakers) * val_ratio))

    val_speakers = set(speakers[:n_val_spk])
    train_ids, val_ids = [], []
    for spk, spk_ids in spk2ids.items():
        (val_ids if spk in val_speakers else train_ids).extend(spk_ids)

    return train_ids, val_ids


_SR         = 16000
_N_MFCC     = 13
_N_FFT      = 512
_HOP_LENGTH = 160
_WIN_LENGTH = 400
_N_MELS     = 26
_PRE_EMPH   = 0.97
_FMIN       = 0.0
_FMAX       = 4000.0
_DELTA_W    = 2
FEAT_DIM    = _N_MFCC * 3


def apply_cmvn(features: np.ndarray) -> np.ndarray:
    mean = features.mean(axis=0, keepdims=True)
    std  = features.std(axis=0, keepdims=True) + FEAT_EPS
    return (features - mean) / std


def _delta_numpy(feat: np.ndarray, w: int = _DELTA_W) -> np.ndarray:
    T      = feat.shape[0]
    denom  = 2.0 * sum(n * n for n in range(1, w + 1))
    padded = np.pad(feat, ((w, w), (0, 0)), mode="edge")
    delta  = np.zeros_like(feat)
    for n in range(1, w + 1):
        delta += n * (padded[w+n: w+n+T] - padded[w-n: w-n+T])
    return delta / denom


def extract_features(wav_path: str, cmvn: bool = True) -> np.ndarray:
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(f"File tidak ditemukan: {wav_path}")
    feat = _mfcc_librosa(wav_path) if _LIBROSA else (
           _mfcc_scipy(wav_path)  if _SCIPY   else
           (_ for _ in ()).throw(ImportError("Install librosa atau scipy")))
    return apply_cmvn(feat) if cmvn else feat


def _mfcc_librosa(wav_path: str) -> np.ndarray:
    y, _ = librosa.load(wav_path, sr=_SR, mono=True)
    if len(y) == 0:
        raise ValueError(f"Audio kosong: {wav_path}")
    y    = np.append(y[0], y[1:] - _PRE_EMPH * y[:-1])
    mfcc = librosa.feature.mfcc(
        y=y, sr=_SR, n_mfcc=_N_MFCC,
        n_fft=_N_FFT, hop_length=_HOP_LENGTH,
        win_length=_WIN_LENGTH, n_mels=_N_MELS,
        fmin=_FMIN, fmax=_FMAX, window="hamming",
    )
    d1 = librosa.feature.delta(mfcc, width=_DELTA_W * 2 + 1)
    d2 = librosa.feature.delta(mfcc, order=2, width=_DELTA_W * 2 + 1)
    return np.vstack([mfcc, d1, d2]).T.astype(np.float32)


def _mfcc_scipy(wav_path: str) -> np.ndarray:
    sr, audio = _wavfile.read(wav_path)
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32:
        audio = audio.astype(np.float32) / 2147483648.0
    else:
        audio = audio.astype(np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != _SR:
        n_new = int(len(audio) * _SR / sr)
        audio = np.interp(
            np.linspace(0, len(audio) - 1, n_new),
            np.arange(len(audio)), audio,
        )
    y        = np.append(audio[0], audio[1:] - _PRE_EMPH * audio[:-1])
    n_frames = max(1, 1 + (len(y) - _WIN_LENGTH) // _HOP_LENGTH)
    idx      = (np.arange(_WIN_LENGTH)[None, :]
                + _HOP_LENGTH * np.arange(n_frames)[:, None])
    idx      = np.clip(idx, 0, len(y) - 1)
    frames   = y[idx].astype(np.float32) * np.hamming(_WIN_LENGTH).astype(np.float32)
    fft_mag  = np.abs(np.fft.rfft(frames, n=_N_FFT))

    def hz2mel(f): return 1127.0 * np.log1p(f / 700.0)
    def mel2hz(m): return 700.0 * (np.exp(m / 1127.0) - 1.0)

    mel_pts = np.linspace(hz2mel(_FMIN), hz2mel(_FMAX), _N_MELS + 2)
    hz_pts  = mel2hz(mel_pts)
    bins    = np.floor((_N_FFT + 1) * hz_pts / _SR).astype(int)
    fb      = np.zeros((_N_MELS, _N_FFT // 2 + 1), dtype=np.float32)
    for m in range(1, _N_MELS + 1):
        for k in range(bins[m-1], bins[m]):
            fb[m-1, k] = (k - bins[m-1]) / max(bins[m] - bins[m-1], 1)
        for k in range(bins[m], bins[m+1]):
            fb[m-1, k] = (bins[m+1] - k) / max(bins[m+1] - bins[m], 1)

    mel_e = np.maximum(fft_mag ** 2 @ fb.T, 1e-10)
    log_e = np.log10(mel_e)
    N     = _N_MELS
    dct_m = np.cos(
        np.pi * np.arange(_N_MFCC)[:, None]
        * (np.arange(N) + 0.5)[None, :] / N
    )
    mfcc = (log_e @ dct_m.T) * np.sqrt(2.0 / N)
    d1   = _delta_numpy(mfcc)
    d2   = _delta_numpy(d1)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


def build_mfcc_cache(
    sampled_root  : str,
    dict_transkrip: Dict[str, str],
) -> Tuple[Dict[str, np.ndarray], List[dict]]:
    wav_files  = glob.glob(os.path.join(sampled_root, "**/*.wav"),  recursive=True)
    wav_files += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
    cache: Dict[str, np.ndarray] = {}
    meta:  List[dict]            = []
    for wp in wav_files:
        aid = os.path.splitext(os.path.basename(wp))[0]
        if not dict_transkrip.get(aid, ""):
            continue
        try:
            feat = extract_features(wp, cmvn=True)
        except (FileNotFoundError, ValueError, OSError):
            continue
        cache[aid] = feat
        meta.append({"Id": aid, "Frames": feat.shape[0], "Dim": feat.shape[1]})
    return cache, meta


class GaussianMixtureModel:
    def __init__(
        self,
        n_components : int   = 4,
        n_iter       : int   = 100,
        tol          : float = 1e-3,
        reg_covar    : float = COV_REG,
        random_state : Optional[int] = None,
    ):
        self.n_components = n_components
        self.n_iter       = n_iter
        self.tol          = tol
        self.reg_covar    = reg_covar
        self.random_state = random_state
        self.weights_     : Optional[np.ndarray] = None
        self.means_       : Optional[np.ndarray] = None
        self.covariances_ : Optional[np.ndarray] = None
        self.converged_   = False
        self.n_iter_done_ = 0
        self.log_likelihood_history_: List[float] = []

    def _init_params(self, X: np.ndarray) -> None:
        rng = np.random.default_rng(self.random_state)
        K   = self.n_components
        idx = rng.choice(X.shape[0], size=K, replace=False)
        self.means_       = X[idx].copy()
        self.weights_     = np.full(K, 1.0 / K)
        self.covariances_ = np.tile(np.var(X, axis=0) + self.reg_covar, (K, 1))

    def _log_gaussian_all(self, X: np.ndarray) -> np.ndarray:
        mu      = self.means_
        var     = self.covariances_
        log_det = np.sum(np.log(var), axis=1)
        diff    = X[:, None, :] - mu[None, :, :]
        maha    = np.sum(diff ** 2 / var[None, :, :], axis=2)
        log_norm = -0.5 * (X.shape[1] * np.log(2 * np.pi) + log_det)
        return log_norm[None, :] - 0.5 * maha

    def _e_step(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        log_w   = np.log(self.weights_ + LOG_EPS) + self._log_gaussian_all(X)
        lse     = np.max(log_w, axis=1, keepdims=True)
        # FIX-9: reshape(-1) menggantikan squeeze() agar aman saat T=1
        log_sum = lse.reshape(-1) + np.log(np.sum(np.exp(log_w - lse), axis=1))
        return log_w - log_sum[:, None], log_sum

    def _m_step(self, X: np.ndarray, log_resp: np.ndarray) -> None:
        resp = np.exp(log_resp)
        Nk   = resp.sum(axis=0) + LOG_EPS
        self.weights_     = Nk / X.shape[0]
        self.means_       = (resp.T @ X) / Nk[:, None]
        diff              = X[:, None, :] - self.means_[None, :, :]
        self.covariances_ = (
            np.sum(resp[:, :, None] * diff ** 2, axis=0) / Nk[:, None]
            + self.reg_covar
        )

    def fit(self, X: np.ndarray) -> "GaussianMixtureModel":
        assert X.ndim == 2
        self.n_components = min(self.n_components, X.shape[0])
        self._init_params(X)
        self.log_likelihood_history_ = []
        prev_ll = -np.inf
        for it in range(self.n_iter):
            log_resp, log_sum = self._e_step(X)
            ll = float(log_sum.mean())
            self.log_likelihood_history_.append(ll)
            if abs(ll - prev_ll) < self.tol:
                self.converged_   = True
                self.n_iter_done_ = it + 1
                break
            prev_ll = ll
            self._m_step(X, log_resp)
        else:
            self.converged_   = False
            self.n_iter_done_ = self.n_iter
        return self

    def log_prob_per_frame(self, X: np.ndarray) -> np.ndarray:
        assert X.ndim == 2
        log_w = np.log(self.weights_ + LOG_EPS) + self._log_gaussian_all(X)
        lse   = np.max(log_w, axis=1, keepdims=True)
        # FIX-9: reshape(-1) konsisten dengan _e_step
        return lse.reshape(-1) + np.log(np.sum(np.exp(log_w - lse), axis=1))


def alignment_fonem_viterbi(
    fitur        : np.ndarray,
    fonem_seq    : List[str],
    gmm_per_fonem: Dict[str, "GaussianMixtureModel"],
) -> np.ndarray:
    T = fitur.shape[0]
    n = len(fonem_seq)
    if n == 0 or T == 0:
        return np.full(T, SIL_ID, dtype=np.int32)

    skor = np.full((T, n), LOG_ZERO)
    for j, fonem in enumerate(fonem_seq):
        if fonem in gmm_per_fonem:
            skor[:, j] = gmm_per_fonem[fonem].log_prob_per_frame(fitur)
        elif fonem == "SIL":
            skor[:, j] = 0.0

    dp = np.full((T + 1, n + 1), LOG_ZERO)
    bp = np.zeros((T + 1, n + 1), dtype=np.int8)
    dp[0, 0] = 0.0

    for t in range(1, T + 1):
        for j in range(n):
            stay    = (dp[t-1, j] + skor[t-1, j]
                       if dp[t-1, j] > LOG_ZERO else LOG_ZERO)
            advance = (dp[t-1, j-1] + skor[t-1, j]
                       if j > 0 and dp[t-1, j-1] > LOG_ZERO else LOG_ZERO)
            if stay >= advance:
                dp[t, j] = stay;    bp[t, j] = 0
            else:
                dp[t, j] = advance; bp[t, j] = 1

    # FIX-10: cari kolom akhir terbaik yang valid, bukan selalu n-1
    valid_ends = [(dp[T, j], j) for j in range(n) if dp[T, j] > LOG_ZERO]
    j_end      = valid_ends[-1][1] if valid_ends else n - 1

    path = np.zeros(T, dtype=int)
    j    = j_end
    for t in range(T, 0, -1):
        path[t - 1] = j
        if bp[t, j] == 1:
            j = max(j - 1, 0)

    return np.array(
        [FONEM2ID.get(fonem_seq[k], SIL_ID) for k in path],
        dtype=np.int32,
    )


def kumpulkan_frame_fonem_naif(
    fitur    : np.ndarray,
    fonem_seq: List[str],
) -> Dict[str, np.ndarray]:
    """
    FIX-17: return Dict[str, np.ndarray] — konsisten dengan path GMM sehingga
    kumpulkan_frame_per_fonem dapat menangani kedua path dengan logika yang sama.
    """
    T, n = fitur.shape[0], len(fonem_seq)
    if n == 0:
        return {}
    n_frame    = np.ones(n, dtype=int) * (T // n)
    n_frame[:T - n_frame.sum()] += 1

    hasil: Dict[str, np.ndarray] = {}
    idx = 0
    for i, f in enumerate(fonem_seq):
        seg = fitur[idx: idx + n_frame[i]]
        if seg.shape[0] > 0:
            hasil[f] = np.vstack([hasil[f], seg]) if f in hasil else seg.copy()
        idx += n_frame[i]
    return hasil


def _bangun_fonem_seq(
    label  : str,
    lexicon: Dict[str, List[str]],
) -> List[str]:
    fonem_seq: List[str] = []
    for kata in normalisasi(label).split():
        fonem_seq.extend(lexicon.get(kata, list(kata_ke_fonem(kata))))
    return fonem_seq


def kumpulkan_frame_per_fonem(
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon       : Dict[str, List[str]],
    gmm_per_fonem : Optional[Dict[str, GaussianMixtureModel]] = None,
) -> Dict[str, np.ndarray]:
    """
    FIX-6: buffer seluruh frame per utterance ke list, vstack sekali per fonem.
    FIX-17: kedua path (naif dan GMM) menghasilkan np.ndarray, type konsisten.
    """
    use_gmm = gmm_per_fonem is not None and len(gmm_per_fonem) > 0
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)

    for audio_id, fitur in cache_mfcc.items():
        label = dict_transkrip.get(audio_id, "")
        if not label:
            continue
        fonem_seq = _bangun_fonem_seq(label, lexicon)
        if not fonem_seq:
            continue

        if use_gmm:
            labels = alignment_fonem_viterbi(fitur, fonem_seq, gmm_per_fonem)
            for fid in np.unique(labels):
                buf[FONEM_JAWA[fid]].append(fitur[labels == fid])
        else:
            for fonem, arr in kumpulkan_frame_fonem_naif(fitur, fonem_seq).items():
                buf[fonem].append(arr)

    return {fonem: np.vstack(parts) for fonem, parts in buf.items()}


def latih_gmm_per_fonem(
    frame_per_fonem: Dict[str, np.ndarray],
    n_components   : int   = N_COMPONENTS_GMM,
    n_iter         : int   = N_ITER_GMM,
    tol            : float = TOL_GMM,
    min_frame      : int   = MIN_FRAME_GMM,
) -> Dict[str, GaussianMixtureModel]:
    gmm_per_fonem: Dict[str, GaussianMixtureModel] = {}
    for fonem, X in sorted(frame_per_fonem.items()):
        if X.shape[0] < min_frame:
            continue
        gmm = GaussianMixtureModel(
            n_components=min(n_components, X.shape[0] // 2),
            n_iter=n_iter, tol=tol, reg_covar=COV_REG,
            random_state=hash(fonem) % 100_000,
        )
        gmm.fit(X)
        gmm_per_fonem[fonem] = gmm
    return gmm_per_fonem


def hitung_alignment_corpus(
    audio_ids    : List[str],
    cache_mfcc   : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon      : Dict[str, List[str]],
    gmm_per_fonem: Dict[str, GaussianMixtureModel],
) -> Dict[str, np.ndarray]:
    """
    FIX-12: hitung alignment satu kali untuk seluruh corpus dan kembalikan
    sebagai cache. Digunakan bersama oleh FrameLevelDataset dan
    kumpulkan_frame_per_fonem — tidak ada perhitungan ulang.
    """
    alignment_cache: Dict[str, np.ndarray] = {}
    for audio_id in audio_ids:
        fitur = cache_mfcc.get(audio_id)
        label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label:
            continue
        fonem_seq = _bangun_fonem_seq(label, lexicon)
        if not fonem_seq:
            continue
        alignment_cache[audio_id] = alignment_fonem_viterbi(
            fitur, fonem_seq, gmm_per_fonem,
        )
    return alignment_cache


class FrameLevelDataset(Dataset):
    """
    FIX-12: menerima alignment_cache yang sudah dihitung — tidak menghitung
    ulang alignment di dalam konstruktor.
    Seluruh frame dan label disusun ke tensor tunggal untuk efisiensi DataLoader.
    """
    def __init__(
        self,
        audio_ids      : List[str],
        cache_mfcc     : Dict[str, np.ndarray],
        alignment_cache: Dict[str, np.ndarray],
    ):
        frame_parts: List[np.ndarray] = []
        label_parts: List[np.ndarray] = []
        for audio_id in audio_ids:
            fitur  = cache_mfcc.get(audio_id)
            aligns = alignment_cache.get(audio_id)
            if fitur is None or aligns is None:
                continue
            frame_parts.append(fitur)
            label_parts.append(aligns)

        if frame_parts:
            self.frames = torch.from_numpy(np.vstack(frame_parts))
            self.labels = torch.from_numpy(
                np.concatenate(label_parts).astype(np.int64)
            )
        else:
            self.frames = torch.zeros((0, FEAT_DIM), dtype=torch.float32)
            self.labels = torch.zeros(0, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.frames)

    def __getitem__(self, i: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.frames[i], self.labels[i]


class TDNNLayer(nn.Module):
    """
    FIX-4: satu path forward tunggal yang selalu menggunakan konteks temporal
    lewat unsqueeze/squeeze. Input (B,D) di-reshape ke (B,1,D) sebelum
    padding dan splicing — konteks nyata diterapkan bahkan saat training
    frame-level, menghilangkan train-inference mismatch.
    """
    def __init__(
        self,
        input_dim : int,
        output_dim: int,
        context   : List[int],
        dropout   : float = 0.0,
    ):
        super().__init__()
        self.context = sorted(context)
        self.linear  = nn.Linear(input_dim * len(context), output_dim)
        self.bn      = nn.BatchNorm1d(output_dim)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        squeeze_out = x.dim() == 2
        if squeeze_out:
            x = x.unsqueeze(1)

        B, T, _ = x.shape
        ctx_min  = min(self.context)
        ctx_max  = max(self.context)
        x_pad    = F.pad(x.permute(0, 2, 1),
                         (max(0, -ctx_min), max(0, ctx_max))).permute(0, 2, 1)

        ctx_frames = []
        for c in self.context:
            shift = c - ctx_min
            ctx_frames.append(x_pad[:, shift: shift + T, :])
        spliced = torch.cat(ctx_frames, dim=-1)

        out = self.linear(spliced)
        out = self.bn(out.permute(0, 2, 1)).permute(0, 2, 1)
        out = F.relu(out)
        out = self.drop(out)

        return out.squeeze(1) if squeeze_out else out


class TDNNAcousticModel(nn.Module):
    def __init__(
        self,
        input_dim : int                      = FEAT_DIM,
        n_classes : int                      = N_FONEM,
        hidden_dim: int                      = TDNN_HIDDEN,
        contexts  : Optional[List[List[int]]] = None,
        dropout   : float                    = TDNN_DROPOUT,
    ):
        super().__init__()
        if contexts is None:
            contexts = TDNN_CONTEXTS
        layers, in_dim = [], input_dim
        for ctx in contexts:
            layers.append(TDNNLayer(in_dim, hidden_dim, ctx, dropout))
            in_dim = hidden_dim
        self.tdnn_layers = nn.ModuleList(layers)
        self.classifier  = nn.Linear(hidden_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.tdnn_layers:
            x = layer(x)
        return F.log_softmax(self.classifier(x), dim=-1)

    def get_posteriors_sequence(self, x: torch.Tensor) -> torch.Tensor:
        """
        FIX-15: tidak memanggil self.eval() di sini — caller yang mengatur
        mode model. x: (1, T, D) → return (T, N_FONEM).
        """
        with torch.no_grad():
            return self.forward(x)[0]


def latih_tdnn_frame_level(
    train_ids      : List[str],
    val_ids        : List[str],
    cache_mfcc     : Dict[str, np.ndarray],
    alignment_train: Dict[str, np.ndarray],
    alignment_val  : Dict[str, np.ndarray],
    dir_save       : str,
    epochs         : int                      = TDNN_EPOCHS,
    lr             : float                    = TDNN_LR,
    batch_size     : int                      = TDNN_BATCH_SIZE,
    hidden_dim     : int                      = TDNN_HIDDEN,
    contexts       : Optional[List[List[int]]] = None,
    dropout        : float                    = TDNN_DROPOUT,
) -> TDNNAcousticModel:
    """
    FIX-5: hapus early stopping — full epoch training, simpan best checkpoint.
    FIX-7: class weights dihitung dari sqrt(count) untuk mengurangi dominasi
           fonem langka, lalu dinormalisasi.
    FIX-12: terima alignment_cache — tidak dihitung ulang di sini.
    """
    if contexts is None:
        contexts = TDNN_CONTEXTS
    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = os.path.join(dir_save, "model_tdnn_fonem.pt")

    model = TDNNAcousticModel(
        input_dim=FEAT_DIM, n_classes=N_FONEM,
        hidden_dim=hidden_dim, contexts=contexts, dropout=dropout,
    ).to(device)

    if os.path.isfile(model_path):
        try:
            model.load_state_dict(torch.load(model_path, map_location=device))
            return model
        except Exception:
            os.remove(model_path)

    train_ds     = FrameLevelDataset(train_ids, cache_mfcc, alignment_train)
    val_ds       = FrameLevelDataset(val_ids,   cache_mfcc, alignment_val)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=0, pin_memory=True)

    # FIX-7: sqrt smoothing + normalisasi; tidak ada clip raksasa
    label_counts = torch.bincount(train_ds.labels, minlength=N_FONEM).float()
    label_counts = torch.clamp(label_counts, min=1.0)
    weights      = 1.0 / torch.sqrt(label_counts)
    weights      = (weights / weights.sum()).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3,
    )
    loss_fn       = nn.NLLLoss(weight=weights)
    riwayat       : List[dict] = []
    best_val_loss  = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, n_correct, n_total = 0.0, 0, 0
        for feats, lbls in train_loader:
            feats = feats.to(device)
            lbls  = lbls.to(device)
            optimizer.zero_grad()
            log_probs = model(feats)
            loss      = loss_fn(log_probs, lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * len(lbls)
            n_correct  += (log_probs.argmax(dim=1) == lbls).sum().item()
            n_total    += len(lbls)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for feats, lbls in val_loader:
                feats = feats.to(device)
                lbls  = lbls.to(device)
                lp    = model(feats)
                val_loss    += loss_fn(lp, lbls).item() * len(lbls)
                val_correct += (lp.argmax(dim=1) == lbls).sum().item()
                val_total   += len(lbls)

        avg_train = total_loss / max(n_total,    1)
        avg_val   = val_loss   / max(val_total,  1)
        acc_train = 100 * n_correct  / max(n_total,   1)
        acc_val   = 100 * val_correct / max(val_total, 1)
        scheduler.step(avg_val)

        riwayat.append({
            "Epoch"        : epoch,
            "Train Loss"   : round(avg_train, 5),
            "Val Loss"     : round(avg_val,   5),
            "Train Acc (%)": round(acc_train, 2),
            "Val Acc (%)"  : round(acc_val,   2),
        })

        # FIX-5: simpan checkpoint terbaik tanpa break
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), model_path)

    if os.path.isfile(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))

    pd.DataFrame(riwayat).to_csv(
        os.path.join(dir_save, "HasilTrainingTDNN_Fonem.csv"), index=False,
    )
    return model


class HMMViterbiDecoder:
    """
    FIX-3: HMM Viterbi sesungguhnya dengan n_hmm_states state per fonem.
    FIX-1: minimum duration MIN_FONEM_DUR sebelum fonem diterima.
    FIX-2: batasi max kata output dari estimasi durasi audio.
    FIX-14: batasi panjang lookup fonem ke kata (MAX_FONEM_PER_KATA).
    FIX-15: model.eval() diatur di __init__ dan dekode, bukan di dalam
            get_posteriors_sequence.
    """

    _LOG_SELF  = math.log(0.6)
    _LOG_NEXT  = math.log(0.3)

    def __init__(
        self,
        model        : TDNNAcousticModel,
        lexicon      : Dict[str, List[str]],
        ngram_model  : "NGramModel",
        gmm_per_fonem: Dict[str, GaussianMixtureModel],
        device       : Optional[torch.device] = None,
        n_hmm_states : int = N_HMM_STATES,
        beam_width   : int = BEAM_WIDTH,
    ):
        self.model         = model
        self.lexicon       = lexicon
        self.rev_lex: Dict[Tuple[str, ...], List[str]] = defaultdict(list)
        for kata, fonems in lexicon.items():
            self.rev_lex[tuple(fonems)].append(kata)
        self.ngram         = ngram_model
        self.gmm_per_fonem = gmm_per_fonem
        self.device        = device or torch.device("cpu")
        self.n_hmm_states  = n_hmm_states
        self.beam_width    = beam_width
        self.model.to(self.device)
        self.model.eval()

    def _get_posteriors(self, fitur: np.ndarray) -> np.ndarray:
        t = torch.from_numpy(fitur).unsqueeze(0).to(self.device)
        return self.model.get_posteriors_sequence(t).cpu().numpy()

    def _viterbi_hmm(self, log_post: np.ndarray) -> List[str]:
        """
        FIX-3: HMM DP dengan state space = N_FONEM × n_hmm_states.
        Transisi:
          - self-loop  (p=0.6): state s → state s
          - maju       (p=0.3): state s → state s+1 dalam fonem yang sama
          - antar-fonem(p=0.1/F): state terakhir fonem f → state pertama fonem g
        Emisi: log posterior TDNN dibagikan ke semua state dalam fonem yang sama.
        FIX-1: hanya terima fonem dengan durasi ≥ MIN_FONEM_DUR setelah traceback.
        """
        T, F   = log_post.shape
        S      = self.n_hmm_states
        NS     = F * S
        log_cross = math.log(0.1 / max(F, 1))

        dp = np.full((T, NS), LOG_ZERO, dtype=np.float64)
        bp = np.full((T, NS), -1,       dtype=np.int32)

        for fid in range(F):
            dp[0, fid * S] = log_post[0, fid]

        for t in range(1, T):
            for fid in range(F):
                emit = log_post[t, fid]
                for s in range(S):
                    cur  = fid * S + s
                    best_prev_score = LOG_ZERO
                    best_prev_state = -1

                    self_score = dp[t-1, cur] + self._LOG_SELF
                    if self_score > best_prev_score:
                        best_prev_score = self_score
                        best_prev_state = cur

                    if s > 0:
                        prev_s = fid * S + (s - 1)
                        adv    = dp[t-1, prev_s] + self._LOG_NEXT
                        if adv > best_prev_score:
                            best_prev_score = adv
                            best_prev_state = prev_s

                    if s == 0:
                        for prev_fid in range(F):
                            prev_last = prev_fid * S + (S - 1)
                            cross = dp[t-1, prev_last] + log_cross
                            if cross > best_prev_score:
                                best_prev_score = cross
                                best_prev_state = prev_last

                    if best_prev_score > LOG_ZERO:
                        dp[t, cur] = best_prev_score + emit
                        bp[t, cur] = best_prev_state

        best_end    = int(np.argmax(dp[T - 1]))
        path_states = [best_end]
        for t in range(T - 1, 0, -1):
            prev = bp[t, path_states[-1]]
            if prev < 0:
                break
            path_states.append(prev)
        path_states.reverse()

        path_fonem = [ps // S for ps in path_states]

        # FIX-1: collapse dengan minimum duration
        result: List[str] = []
        i = 0
        while i < len(path_fonem):
            fid = path_fonem[i]
            j   = i + 1
            while j < len(path_fonem) and path_fonem[j] == fid:
                j += 1
            if (j - i) >= MIN_FONEM_DUR:
                result.append(FONEM_JAWA[fid])
            i = j

        return [f for f in result if f != "SIL"]

    def _fonem_seq_ke_kata(
        self,
        fonem_seq: List[str],
        T        : int,
    ) -> List[List[str]]:
        """
        FIX-2: max_words dari durasi audio.
        FIX-14: batas panjang lookup MAX_FONEM_PER_KATA.
        """
        n         = len(fonem_seq)
        max_words = max(1, T // AVG_FRAMES_PER_WORD)
        if n == 0:
            return [[]]

        dp: List[List[Tuple[float, List[str]]]] = [[] for _ in range(n + 1)]
        dp[0] = [(0.0, [])]

        for i in range(n):
            if not dp[i]:
                continue
            # FIX-14: batasi panjang substring fonem
            j_max = min(n + 1, i + MAX_FONEM_PER_KATA + 1)
            for j in range(i + 1, j_max):
                sub = tuple(fonem_seq[i:j])
                if sub not in self.rev_lex:
                    continue
                for kata in self.rev_lex[sub]:
                    for score, words in dp[i]:
                        # FIX-2: lewati jika sudah mencapai estimasi max kata
                        if len(words) >= max_words:
                            continue
                        new_words = words + [kata]
                        new_score = score + self.ngram.log_prob(
                            kata, tuple(words[-(self.ngram.n - 1):])
                        )
                        dp[j].append((new_score, new_words))

                if len(dp[j]) > self.beam_width * 3:
                    dp[j].sort(key=lambda x: x[0], reverse=True)
                    dp[j] = dp[j][: self.beam_width]

        if dp[n]:
            dp[n].sort(key=lambda x: x[0], reverse=True)
            return [w for _, w in dp[n][: self.beam_width]]
        return [fonem_seq]

    def dekode(self, fitur: np.ndarray) -> List[str]:
        if fitur.shape[0] == 0:
            return []
        self.model.eval()
        log_post      = self._get_posteriors(fitur)
        fonem_seq     = self._viterbi_hmm(log_post)
        if not fonem_seq:
            return []
        kata_kandidat = self._fonem_seq_ke_kata(fonem_seq, T=fitur.shape[0])
        if not kata_kandidat:
            return fonem_seq
        return self.ngram.rescore(kata_kandidat)


class NGramModel:
    def __init__(self, n: int = 3, discount: float = 0.75):
        self.n              = n
        self.discount       = discount
        self.ngram_count   : Dict[Tuple, int] = defaultdict(int)
        self.context_count : Dict[Tuple, int] = defaultdict(int)
        self.unigram_count : Dict[str,   int] = defaultdict(int)
        self.vocab         : set              = set()
        self.N_total        = 0
        self._ctx_div      : Dict[Tuple, int] = {}

    def _tokenize(self, kalimat: str) -> List[str]:
        tokens = normalisasi(kalimat).split()
        return ["<s>"] * (self.n - 1) + tokens + ["</s>"]

    def fit(self, korpus: List[str]) -> "NGramModel":
        ctx_words: Dict[Tuple, set] = defaultdict(set)
        for kalimat in korpus:
            tokens = self._tokenize(kalimat)
            self.vocab.update(tokens)
            self.N_total += len(tokens)
            for tok in tokens:
                self.unigram_count[tok] += 1
            for i in range(len(tokens) - self.n + 1):
                ngram = tuple(tokens[i: i + self.n])
                ctx   = ngram[:-1]
                self.ngram_count[ngram]  += 1
                self.context_count[ctx]  += 1
                ctx_words[ctx].add(ngram[-1])
        self._ctx_div = {c: len(w) for c, w in ctx_words.items()}
        return self

    def log_prob(self, kata: str, context: Tuple) -> float:
        """
        FIX-13: backoff unigram menggunakan self.unigram_count (dict terpisah)
        bukan self.ngram_count dengan tuple panjang 1, yang selalu 0 untuk n>2.
        """
        ngram = context + (kata,)
        C_ng  = self.ngram_count.get(ngram, 0)
        C_ctx = self.context_count.get(context, 0)
        C_uni = self.unigram_count.get(kata, 0)
        p_uni = (C_uni + 1) / (self.N_total + len(self.vocab))

        if C_ctx == 0:
            return math.log(p_uni + LOG_EPS)

        lam  = self.discount * self._ctx_div.get(context, 0) / C_ctx
        prob = max(C_ng - self.discount, 0) / C_ctx + lam * p_uni
        return math.log(prob + LOG_EPS)

    def score_sequence(self, kata_list: List[str]) -> float:
        tokens = ["<s>"] * (self.n - 1) + kata_list + ["</s>"]
        log_p  = 0.0
        for i in range(self.n - 1, len(tokens)):
            log_p += self.log_prob(
                tokens[i], tuple(tokens[i - self.n + 1: i])
            )
        return log_p

    def rescore(self, kandidat: List[List[str]]) -> List[str]:
        if not kandidat:
            return []
        return max(kandidat, key=self.score_sequence)


def compute_wer(ref: List[str], hyp: List[str]) -> Dict:
    R, H = len(ref), len(hyp)
    dp   = [[0] * (H + 1) for _ in range(R + 1)]
    ops  = [[""] * (H + 1) for _ in range(R + 1)]
    for i in range(R + 1): dp[i][0] = i;  ops[i][0] = "D" * i
    for j in range(H + 1): dp[0][j] = j;  ops[0][j] = "I" * j
    for i in range(1, R + 1):
        for j in range(1, H + 1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]; ops[i][j] = "M"
            else:
                sub  = dp[i-1][j-1] + 1
                dlt  = dp[i-1][j]   + 1
                ins  = dp[i][j-1]   + 1
                best = min(sub, dlt, ins)
                dp[i][j]  = best
                ops[i][j] = "S" if best == sub else ("D" if best == dlt else "I")
    S = D = I = 0
    i, j = R, H
    while i > 0 or j > 0:
        op = ops[i][j]
        if   op == "M": i -= 1; j -= 1
        elif op == "S": S += 1; i -= 1; j -= 1
        elif op == "D": D += 1; i -= 1
        elif op == "I": I += 1; j -= 1
        else:
            if i > 0: D += 1; i -= 1
            else:     I += 1; j -= 1
    wer = (S + D + I) / R if R > 0 else 0.0
    return {"wer_persen": round(wer * 100, 2), "S": S, "D": D, "I": I, "N": R}


def dekode_semua_audio(
    cache_mfcc    : Dict[str, np.ndarray],
    audio_ids     : List[str],
    dict_transkrip: Dict[str, str],
    decoder       : HMMViterbiDecoder,
) -> List[dict]:
    hasil_wer: List[dict] = []
    for audio_id in audio_ids:
        fitur = cache_mfcc.get(audio_id)
        label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label:
            continue
        ref_kata = normalisasi(label).split()
        if not ref_kata:
            continue
        try:
            hyp_kata = decoder.dekode(fitur)
        except (RuntimeError, MemoryError):
            continue
        wer_res = compute_wer(ref_kata, hyp_kata)
        hasil_wer.append({
            "Id"       : audio_id,
            "Referensi": " ".join(ref_kata),
            "Hipotesis": " ".join(hyp_kata),
            "WER (%)"  : wer_res["wer_persen"],
            "S"        : wer_res["S"],
            "D"        : wer_res["D"],
            "I"        : wer_res["I"],
            "N"        : wer_res["N"],
        })
    return hasil_wer


def ringkasan_wer(df_wer: pd.DataFrame) -> Dict:
    if df_wer.empty:
        return {}
    S = int(df_wer["S"].sum())
    D = int(df_wer["D"].sum())
    I = int(df_wer["I"].sum())
    N = int(df_wer["N"].sum())
    wer_c = (S + D + I) / N * 100 if N > 0 else 100.0
    return {
        "wer_corpus" : round(wer_c, 2),
        "wer_mean"   : round(df_wer["WER (%)"].mean(),   2),
        "wer_median" : round(df_wer["WER (%)"].median(), 2),
        "wer_std"    : round(df_wer["WER (%)"].std(),    2),
        "total_S"    : S,
        "total_D"    : D,
        "total_I"    : I,
        "total_N"    : N,
    }

In [4]:
def main() -> None:
    """
    Pipeline utama TDNN-HMM untuk pengenalan ucapan bahasa Jawa.

    Tahapan:
      1.  Baca transkrip & speaker dari CSV
      2.  Bangun cache MFCC dari seluruh audio
      3.  Split train / val (per speaker jika tersedia)
      4.  Bangun leksikon G2P
      5.  Latih GMM per fonem (bootstrap naif)
      6.  Iterasi GMM: re-alignment dengan Viterbi → latih ulang GMM
      7.  Hitung alignment corpus (train + val) sekali pakai
      8.  Latih TDNN frame-level
      9.  Bangun NGram dari transkrip train
      10. Dekode seluruh audio val dengan HMMViterbiDecoder
      11. Hitung & simpan WER
    """
    import os
    import time
    import random

    import numpy  as np
    import pandas as pd

    # ── reproducibility ────────────────────────────────────────────────────────
    random.seed(42)
    np.random.seed(42)

    # ── 0. siapkan direktori keluaran ──────────────────────────────────────────
    os.makedirs(DIR_SAVE, exist_ok=True)

    t_total = time.time()
    print("=" * 60)
    print("  Pipeline TDNN-HMM – Pengenalan Ucapan Bahasa Jawa")
    print("=" * 60)

    # ── 1. baca CSV ────────────────────────────────────────────────────────────
    print("\n[1/11] Membaca transkrip dari CSV …")
    dict_transkrip, dict_speaker = baca_transkrip_csv(CSV_PATH, COL_SPEAKER)
    print(f"       {len(dict_transkrip):,} utterance ditemukan, "
          f"{len(set(dict_speaker.values())):,} speaker unik.")

    # ── 2. cache MFCC ──────────────────────────────────────────────────────────
    print("\n[2/11] Ekstraksi fitur MFCC …")
    t0 = time.time()
    cache_mfcc, mfcc_meta = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    print(f"       {len(cache_mfcc):,} file berhasil diekstrak "
          f"({time.time() - t0:.1f} s).")

    if not cache_mfcc:
        raise RuntimeError(
            "Tidak ada audio yang berhasil diekstrak. "
            "Periksa SAMPLED_ROOT dan CSV_PATH."
        )

    # simpan meta fitur
    pd.DataFrame(mfcc_meta).to_csv(
        os.path.join(DIR_SAVE, "meta_mfcc.csv"), index=False
    )

    # ── 3. train / val split ──────────────────────────────────────────────────
    print("\n[3/11] Split train / val …")
    audio_ids        = list(cache_mfcc.keys())
    train_ids, val_ids = split_by_speaker(audio_ids, dict_speaker, VAL_RATIO)
    print(f"       train={len(train_ids):,}  val={len(val_ids):,}")

    # ── 4. leksikon G2P ───────────────────────────────────────────────────────
    print("\n[4/11] Membangun leksikon G2P …")
    all_transkrip = [dict_transkrip[aid]
                     for aid in train_ids if aid in dict_transkrip]
    lexicon       = buat_lexicon(all_transkrip)
    print(f"       {len(lexicon):,} entri kata dalam leksikon.")

    # ── 5. GMM bootstrap (alignment naif) ────────────────────────────────────
    print("\n[5/11] Bootstrap GMM per fonem (alignment naif) …")
    t0 = time.time()

    cache_train = {aid: cache_mfcc[aid]
                   for aid in train_ids if aid in cache_mfcc}

    frame_per_fonem_naif = kumpulkan_frame_per_fonem(
        cache_mfcc     = cache_train,
        dict_transkrip = dict_transkrip,
        lexicon        = lexicon,
        gmm_per_fonem  = None,          # tanpa GMM → alignment naif
    )
    gmm_per_fonem = latih_gmm_per_fonem(
        frame_per_fonem_naif,
        n_components = N_COMPONENTS_GMM,
        n_iter       = N_ITER_GMM,
        tol          = TOL_GMM,
        min_frame    = MIN_FRAME_GMM,
    )
    print(f"       {len(gmm_per_fonem):,} fonem terlatih "
          f"({time.time() - t0:.1f} s).")

    # ── 6. iterasi GMM (Viterbi re-alignment) ─────────────────────────────────
    N_GMM_ITER = 2
    print(f"\n[6/11] Iterasi GMM ({N_GMM_ITER}×) dengan Viterbi re-alignment …")
    for it in range(1, N_GMM_ITER + 1):
        t0 = time.time()
        frame_per_fonem_gmm = kumpulkan_frame_per_fonem(
            cache_mfcc     = cache_train,
            dict_transkrip = dict_transkrip,
            lexicon        = lexicon,
            gmm_per_fonem  = gmm_per_fonem,
        )
        gmm_per_fonem = latih_gmm_per_fonem(
            frame_per_fonem_gmm,
            n_components = N_COMPONENTS_GMM,
            n_iter       = N_ITER_GMM,
            tol          = TOL_GMM,
            min_frame    = MIN_FRAME_GMM,
        )
        print(f"       iter {it}/{N_GMM_ITER}: "
              f"{len(gmm_per_fonem):,} fonem ({time.time() - t0:.1f} s).")

    # ── 7. hitung alignment corpus (train + val) ──────────────────────────────
    print("\n[7/11] Menghitung alignment Viterbi untuk seluruh corpus …")
    t0 = time.time()
    alignment_train = hitung_alignment_corpus(
        train_ids, cache_mfcc, dict_transkrip, lexicon, gmm_per_fonem
    )
    alignment_val = hitung_alignment_corpus(
        val_ids, cache_mfcc, dict_transkrip, lexicon, gmm_per_fonem
    )
    print(f"       train={len(alignment_train):,}  val={len(alignment_val):,} "
          f"({time.time() - t0:.1f} s).")

    # ── 8. latih TDNN ─────────────────────────────────────────────────────────
    print("\n[8/11] Melatih TDNN frame-level …")
    t0 = time.time()
    model_tdnn = latih_tdnn_frame_level(
        train_ids       = train_ids,
        val_ids         = val_ids,
        cache_mfcc      = cache_mfcc,
        alignment_train = alignment_train,
        alignment_val   = alignment_val,
        dir_save        = DIR_SAVE,
        epochs          = TDNN_EPOCHS,
        lr              = TDNN_LR,
        batch_size      = TDNN_BATCH_SIZE,
        hidden_dim      = TDNN_HIDDEN,
        contexts        = TDNN_CONTEXTS,
        dropout         = TDNN_DROPOUT,
    )
    print(f"       Selesai ({time.time() - t0:.1f} s).")

    # ── 9. bangun NGram dari transkrip train ──────────────────────────────────
    print("\n[9/11] Melatih model N-Gram …")
    ngram_model = NGramModel(n=N_GRAM)
    ngram_model.fit(all_transkrip)
    print(f"       Vocab: {len(ngram_model.vocab):,} token, "
          f"N={N_GRAM}.")

    # ── 10. dekode val ────────────────────────────────────────────────────────
    print("\n[10/11] Dekode audio validasi …")
    import torch

    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    decoder = HMMViterbiDecoder(
        model         = model_tdnn,
        lexicon       = lexicon,
        ngram_model   = ngram_model,
        gmm_per_fonem = gmm_per_fonem,
        device        = device,
        n_hmm_states  = N_HMM_STATES,
        beam_width    = BEAM_WIDTH,
    )
    t0 = time.time()
    hasil_wer = dekode_semua_audio(
        cache_mfcc     = cache_mfcc,
        audio_ids      = val_ids,
        dict_transkrip = dict_transkrip,
        decoder        = decoder,
    )
    print(f"       {len(hasil_wer):,} utterance didekode ({time.time() - t0:.1f} s).")

    # ── 11. hitung & simpan WER ───────────────────────────────────────────────
    print("\n[11/11] Menghitung & menyimpan WER …")
    if hasil_wer:
        df_wer = pd.DataFrame(hasil_wer)
        df_wer.to_csv(os.path.join(DIR_SAVE, "HasilWER.csv"), index=False)

        ringkasan = ringkasan_wer(df_wer)
        df_ringkasan = pd.DataFrame([ringkasan])
        df_ringkasan.to_csv(
            os.path.join(DIR_SAVE, "RingkasanWER.csv"), index=False
        )

        print("\n  ── Ringkasan WER ──────────────────────────────")
        for k, v in ringkasan.items():
            print(f"     {k:<20}: {v}")
    else:
        print("  [PERINGATAN] Tidak ada hasil dekode yang diperoleh.")

    print(f"\n{'=' * 60}")
    print(f"  Pipeline selesai dalam {time.time() - t_total:.1f} detik.")
    print(f"  Output tersimpan di: {DIR_SAVE}")
    print("=" * 60)


if __name__ == "__main__":
    main()

  Pipeline TDNN-HMM – Pengenalan Ucapan Bahasa Jawa

[1/11] Membaca transkrip dari CSV …
       1,749 utterance ditemukan, 0 speaker unik.

[2/11] Ekstraksi fitur MFCC …
       1,749 file berhasil diekstrak (15.5 s).

[3/11] Split train / val …
       train=1,400  val=349

[4/11] Membangun leksikon G2P …
       4,272 entri kata dalam leksikon.

[5/11] Bootstrap GMM per fonem (alignment naif) …
       27 fonem terlatih (979.7 s).

[6/11] Iterasi GMM (2×) dengan Viterbi re-alignment …
       iter 1/2: 27 fonem (1063.6 s).
       iter 2/2: 27 fonem (1082.8 s).

[7/11] Menghitung alignment Viterbi untuk seluruh corpus …
       train=1,400  val=349 (386.1 s).

[8/11] Melatih TDNN frame-level …
       Selesai (0.1 s).

[9/11] Melatih model N-Gram …
       Vocab: 4,274 token, N=3.

[10/11] Dekode audio validasi …
       349 utterance didekode (106.0 s).

[11/11] Menghitung & menyimpan WER …

  ── Ringkasan WER ──────────────────────────────
     wer_corpus          : 270.14
     wer_mean     

In [5]:
# =============================================================================
#  HYBRID TDNN-HMM ASR — Bahasa Jawa — v2 (IMPROVED)
# =============================================================================
#
#  DAFTAR PERUBAHAN DARI v1:
#  [FIX-A] N_HMM_STATES: 3 → 4        (paper: 4-state optimal)
#  [FIX-B] TDNN_CONTEXTS: tambah input layer ±2, total 7 layer
#  [FIX-C] GMM iteratif: 1 → 3 iterasi EM bootstrapping
#  [FIX-D] Triphone GMM: alignment pakai triphone, fallback ke monophone
#  [FIX-E] Speaker CMVN: normalisasi per-speaker sebelum per-utterance
#  [FIX-F] HMM cross-phone: 0.1/F → 0.4/F (lebih permissive)
#  [FIX-G] HMM non-final state: normalisasi 0.65/0.35 agar sum=1
#  [FIX-H] N_COMPONENTS_GMM: 16 → 8, adaptif berdasarkan jumlah frame
#  [FIX-I] LR warmup + cosine scheduler menggantikan ReduceLROnPlateau
#  [FIX-J] main() pipeline yang benar dengan urutan iterasi yang tepat
# =============================================================================

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\hybrid\src\TDNN-HMM\out"
)

COL_SPEAKER = "speaker_id"

# ── [FIX-H] GMM: kurangi komponen, lebih stabil untuk fonem langka ──────────
N_COMPONENTS_GMM  = 8      # was 16 — terlalu besar untuk fonem langka
N_ITER_GMM        = 100
TOL_GMM           = 1e-4
MIN_FRAME_GMM     = 20     # was 10 — pastikan cukup data per GMM

# ── [FIX-C] Iterasi EM untuk bootstrap GMM ──────────────────────────────────
GMM_EM_ITERS      = 3      # was 1 (hanya naive alignment)

# ── [FIX-D] Triphone ─────────────────────────────────────────────────────────
USE_TRIPHONE      = True
TRIPHONE_MIN_FRAME = 30    # minimum frame untuk triphone punya GMM sendiri

# ── [FIX-B] TDNN contexts: tambah input layer ±2, 7 layer total ─────────────
TDNN_HIDDEN       = 512
TDNN_DROPOUT      = 0.1
TDNN_EPOCHS       = 50
TDNN_LR           = 5e-4   # was 1e-3; lebih kecil, lebih stabil
TDNN_BATCH_SIZE   = 256
TDNN_WARMUP_EPOCH = 5      # [FIX-I] warmup epoch baru
TDNN_CONTEXTS     = [      # [FIX-B] was 5 layers tanpa input splice
    [-2, -1, 0, 1, 2],    # Input layer: splice ±2 (baru)
    [-1,  0, 1],           # Hidden 1
    [-1,  0, 1],           # Hidden 2
    [-3,  0, 3],           # Hidden 3
    [-3,  0, 3],           # Hidden 4
    [-6,  0, 6],           # Hidden 5
    [-6,  0, 6],           # Hidden 6
]

# ── [FIX-A] HMM states ───────────────────────────────────────────────────────
N_HMM_STATES      = 4      # was 3 — paper: 4-state konsisten optimal

# ── [FIX-F] HMM transitions (semua harus dikalibrasi bersama) ───────────────
HMM_P_SELF        = 0.60   # probabilitas self-loop
HMM_P_ADVANCE     = 0.35   # maju ke state berikutnya dalam fonem
HMM_P_CROSS_TOTAL = 0.40   # [FIX-F] was 0.10 — total prob keluar ke fonem baru

N_GRAM            = 3
BEAM_WIDTH        = 10
VAL_RATIO         = 0.2
SPEAKER_SPLIT     = True
MAX_FONEM_PER_KATA = 8
MIN_FONEM_DUR     = 2
AVG_FRAMES_PER_WORD = 30

LOG_ZERO = -1e15
LOG_EPS  = 1e-300
COV_REG  = 1e-6
FEAT_EPS = 1e-8

import os, re, glob, math, time, random, warnings
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    _TORCH = True
except ImportError:
    raise ImportError("PyTorch diperlukan: pip install torch")

try:
    import librosa
    _LIBROSA = True
except ImportError:
    _LIBROSA = False

try:
    from scipy.io import wavfile as _wavfile
    _SCIPY = True
except ImportError:
    _SCIPY = False


# =============================================================================
#  FONEM JAWA
# =============================================================================

FONEM_JAWA: List[str] = [
    "a", "i", "u", "e", "o", "eu", "ae",
    "p", "b", "t", "d", "k", "g", "dh", "th",
    "m", "n", "ng", "ny",
    "s", "h", "f",
    "j", "c",
    "l", "r",
    "w", "y",
    "SIL",
]
FONEM2ID: Dict[str, int] = {f: i for i, f in enumerate(FONEM_JAWA)}
N_FONEM: int = len(FONEM_JAWA)
SIL_ID: int  = FONEM2ID["SIL"]

_G2P_DIGRAF: Dict[str, str] = {
    "dh": "dh", "th": "th", "ng": "ng", "ny": "ny",
}

_kata_fonem_cache:    Dict[str, Tuple[str, ...]] = {}
_normalisasi_cache:   Dict[str, str]             = {}


def kata_ke_fonem(kata: str) -> Tuple[str, ...]:
    if kata in _kata_fonem_cache:
        return _kata_fonem_cache[kata]
    fonem: List[str] = []
    i = 0
    while i < len(kata):
        if i + 1 < len(kata) and kata[i:i+2] in _G2P_DIGRAF:
            fonem.append(_G2P_DIGRAF[kata[i:i+2]])
            i += 2
        elif kata[i] in FONEM2ID:
            fonem.append(kata[i])
            i += 1
        else:
            i += 1
    result = tuple(fonem) if fonem else ("SIL",)
    _kata_fonem_cache[kata] = result
    return result


def normalisasi(teks: str) -> str:
    if teks in _normalisasi_cache:
        return _normalisasi_cache[teks]
    t = teks.lower().strip()
    t = re.sub(r"[^a-z\s]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    _normalisasi_cache[teks] = t
    return t


def buat_lexicon(transkrip_list: List[str]) -> Dict[str, List[str]]:
    lex: Dict[str, List[str]] = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lex:
                lex[kata] = list(kata_ke_fonem(kata))
    return lex


# =============================================================================
#  [FIX-D] TRIPHONE SUPPORT
# =============================================================================

def buat_triphone_seq(fonem_seq: List[str]) -> List[str]:
    """
    Ubah sequence monophone ke triphone dengan konteks kiri-kanan.
    ['b','a','t','o'] → ['SIL-b+a','b-a+t','a-t+o','t-o+SIL']
    """
    if not fonem_seq:
        return []
    padded = ["SIL"] + list(fonem_seq) + ["SIL"]
    return [
        f"{padded[i-1]}-{padded[i]}+{padded[i+1]}"
        for i in range(1, len(padded) - 1)
    ]


def triphone_center(triphone: str) -> str:
    """Ekstrak fonem pusat: 'ng-a+k' → 'a'"""
    try:
        return triphone.split("-")[1].split("+")[0]
    except IndexError:
        return triphone


# =============================================================================
#  CSV & SPLIT
# =============================================================================

def baca_transkrip_csv(
    csv_path   : str,
    col_speaker: Optional[str] = COL_SPEAKER,
) -> Tuple[Dict[str, str], Dict[str, str]]:
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"CSV tidak ditemukan: {csv_path}")

    df = None
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(csv_path, encoding=enc)
            break
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue
    if df is None:
        raise ValueError(f"Gagal membaca CSV: {csv_path}")

    df.columns = [c.strip() for c in df.columns]
    col_path_cands = ["Path_Data", "path_data", "path", "audio_path",
                      "filename", "file", "wav_path"]
    col_text_cands = ["sentences", "sentence", "text", "transcription",
                      "label", "transcript"]

    col_path = next((c for c in col_path_cands if c in df.columns), None)
    col_text = next((c for c in col_text_cands if c in df.columns), None)
    if col_path is None:
        raise ValueError(f"Kolom path tidak ditemukan. Kolom: {list(df.columns)}")
    if col_text is None:
        raise ValueError(f"Kolom transkrip tidak ditemukan. Kolom: {list(df.columns)}")

    dict_transkrip: Dict[str, str] = {}
    dict_speaker:   Dict[str, str] = {}
    has_speaker = bool(col_speaker and col_speaker in df.columns)

    for _, row in df.iterrows():
        pv = str(row[col_path]).strip()
        tv = str(row[col_text]).strip()
        if pv in ("nan", "") or tv in ("nan", ""):
            continue
        audio_id = os.path.splitext(os.path.basename(pv))[0]
        dict_transkrip[audio_id] = tv.lower()
        if has_speaker:
            dict_speaker[audio_id] = str(row[col_speaker]).strip()

    return dict_transkrip, dict_speaker


def split_by_speaker(
    audio_ids   : List[str],
    dict_speaker: Dict[str, str],
    val_ratio   : float = VAL_RATIO,
) -> Tuple[List[str], List[str]]:
    ids = list(audio_ids)
    if not dict_speaker:
        random.shuffle(ids)
        n_val = max(1, int(len(ids) * val_ratio))
        return ids[n_val:], ids[:n_val]

    spk2ids: Dict[str, List[str]] = defaultdict(list)
    for aid in ids:
        spk = dict_speaker.get(aid, "unknown")
        spk2ids[spk].append(aid)

    speakers  = list(spk2ids.keys())
    random.shuffle(speakers)
    n_val_spk = max(1, int(len(speakers) * val_ratio))
    val_spk   = set(speakers[:n_val_spk])

    train_ids, val_ids = [], []
    for spk, spk_ids in spk2ids.items():
        (val_ids if spk in val_spk else train_ids).extend(spk_ids)
    return train_ids, val_ids


# =============================================================================
#  FEATURE EXTRACTION
# =============================================================================

_SR         = 16000
_N_MFCC     = 13
_N_FFT      = 512
_HOP_LENGTH = 160
_WIN_LENGTH = 400
_N_MELS     = 26
_PRE_EMPH   = 0.97
_FMIN       = 0.0
_FMAX       = 4000.0
_DELTA_W    = 2
FEAT_DIM    = _N_MFCC * 3   # 39


def apply_cmvn(features: np.ndarray) -> np.ndarray:
    """CMVN per utterance."""
    mean = features.mean(axis=0, keepdims=True)
    std  = features.std(axis=0,  keepdims=True) + FEAT_EPS
    return (features - mean) / std


# ── [FIX-E] Speaker-level CMVN ───────────────────────────────────────────────

def apply_cmvn_speaker(
    cache_mfcc  : Dict[str, np.ndarray],
    dict_speaker: Dict[str, str],
) -> Dict[str, np.ndarray]:
    """
    [FIX-E] Normalisasi per-speaker dulu, kemudian per-utterance.
    Ini menghilangkan offset level energi dan spektral antar speaker
    sebelum CMVN per-utterance diaplikasikan.
    """
    # Kumpulkan semua frame per speaker
    spk_frames: Dict[str, List[np.ndarray]] = defaultdict(list)
    for aid, feat in cache_mfcc.items():
        spk = dict_speaker.get(aid, "unknown")
        spk_frames[spk].append(feat)

    # Hitung statistik per speaker
    spk_mean: Dict[str, np.ndarray] = {}
    spk_std:  Dict[str, np.ndarray] = {}
    for spk, frames in spk_frames.items():
        all_f         = np.vstack(frames)
        spk_mean[spk] = all_f.mean(axis=0)
        spk_std[spk]  = all_f.std(axis=0) + FEAT_EPS

    # Terapkan normalisasi per-speaker → per-utterance
    result: Dict[str, np.ndarray] = {}
    for aid, feat in cache_mfcc.items():
        spk        = dict_speaker.get(aid, "unknown")
        normalized = (feat - spk_mean[spk]) / spk_std[spk]
        result[aid] = apply_cmvn(normalized)   # lanjut per-utterance

    return result


def _delta_numpy(feat: np.ndarray, w: int = _DELTA_W) -> np.ndarray:
    T      = feat.shape[0]
    denom  = 2.0 * sum(n * n for n in range(1, w + 1))
    padded = np.pad(feat, ((w, w), (0, 0)), mode="edge")
    delta  = np.zeros_like(feat)
    for n in range(1, w + 1):
        delta += n * (padded[w+n: w+n+T] - padded[w-n: w-n+T])
    return delta / denom


def extract_features(wav_path: str, cmvn: bool = True) -> np.ndarray:
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(f"File tidak ditemukan: {wav_path}")
    feat = _mfcc_librosa(wav_path) if _LIBROSA else (
           _mfcc_scipy(wav_path)  if _SCIPY   else
           (_ for _ in ()).throw(ImportError("Install librosa atau scipy")))
    # Catatan: cmvn per-utterance dilakukan di sini sebagai dasar;
    # speaker-level CMVN dilakukan kemudian via apply_cmvn_speaker()
    return apply_cmvn(feat) if cmvn else feat


def _mfcc_librosa(wav_path: str) -> np.ndarray:
    y, _ = librosa.load(wav_path, sr=_SR, mono=True)
    if len(y) == 0:
        raise ValueError(f"Audio kosong: {wav_path}")
    y    = np.append(y[0], y[1:] - _PRE_EMPH * y[:-1])
    mfcc = librosa.feature.mfcc(
        y=y, sr=_SR, n_mfcc=_N_MFCC,
        n_fft=_N_FFT, hop_length=_HOP_LENGTH,
        win_length=_WIN_LENGTH, n_mels=_N_MELS,
        fmin=_FMIN, fmax=_FMAX, window="hamming",
    )
    d1 = librosa.feature.delta(mfcc, width=_DELTA_W * 2 + 1)
    d2 = librosa.feature.delta(mfcc, order=2, width=_DELTA_W * 2 + 1)
    return np.vstack([mfcc, d1, d2]).T.astype(np.float32)


def _mfcc_scipy(wav_path: str) -> np.ndarray:
    sr, audio = _wavfile.read(wav_path)
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32:
        audio = audio.astype(np.float32) / 2147483648.0
    else:
        audio = audio.astype(np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != _SR:
        n_new = int(len(audio) * _SR / sr)
        audio = np.interp(
            np.linspace(0, len(audio) - 1, n_new),
            np.arange(len(audio)), audio,
        )
    y        = np.append(audio[0], audio[1:] - _PRE_EMPH * audio[:-1])
    n_frames = max(1, 1 + (len(y) - _WIN_LENGTH) // _HOP_LENGTH)
    idx      = (np.arange(_WIN_LENGTH)[None, :]
                + _HOP_LENGTH * np.arange(n_frames)[:, None])
    idx      = np.clip(idx, 0, len(y) - 1)
    frames   = y[idx].astype(np.float32) * np.hamming(_WIN_LENGTH).astype(np.float32)
    fft_mag  = np.abs(np.fft.rfft(frames, n=_N_FFT))

    def hz2mel(f): return 1127.0 * np.log1p(f / 700.0)
    def mel2hz(m): return 700.0 * (np.exp(m / 1127.0) - 1.0)

    mel_pts = np.linspace(hz2mel(_FMIN), hz2mel(_FMAX), _N_MELS + 2)
    hz_pts  = mel2hz(mel_pts)
    bins    = np.floor((_N_FFT + 1) * hz_pts / _SR).astype(int)
    fb      = np.zeros((_N_MELS, _N_FFT // 2 + 1), dtype=np.float32)
    for m in range(1, _N_MELS + 1):
        for k in range(bins[m-1], bins[m]):
            fb[m-1, k] = (k - bins[m-1]) / max(bins[m] - bins[m-1], 1)
        for k in range(bins[m], bins[m+1]):
            fb[m-1, k] = (bins[m+1] - k) / max(bins[m+1] - bins[m], 1)

    mel_e = np.maximum(fft_mag ** 2 @ fb.T, 1e-10)
    log_e = np.log10(mel_e)
    N     = _N_MELS
    dct_m = np.cos(
        np.pi * np.arange(_N_MFCC)[:, None]
        * (np.arange(N) + 0.5)[None, :] / N
    )
    mfcc = (log_e @ dct_m.T) * np.sqrt(2.0 / N)
    d1   = _delta_numpy(mfcc)
    d2   = _delta_numpy(d1)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


def build_mfcc_cache(
    sampled_root  : str,
    dict_transkrip: Dict[str, str],
) -> Tuple[Dict[str, np.ndarray], List[dict]]:
    wav_files  = glob.glob(os.path.join(sampled_root, "**/*.wav"),  recursive=True)
    wav_files += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
    cache: Dict[str, np.ndarray] = {}
    meta:  List[dict]            = []
    for wp in wav_files:
        aid = os.path.splitext(os.path.basename(wp))[0]
        if not dict_transkrip.get(aid, ""):
            continue
        try:
            # CMVN per-utterance dasar; speaker CMVN diterapkan kemudian
            feat = extract_features(wp, cmvn=True)
        except (FileNotFoundError, ValueError, OSError):
            continue
        cache[aid] = feat
        meta.append({"Id": aid, "Frames": feat.shape[0], "Dim": feat.shape[1]})
    return cache, meta


# =============================================================================
#  GMM
# =============================================================================

class GaussianMixtureModel:
    def __init__(
        self,
        n_components : int   = 4,
        n_iter       : int   = 100,
        tol          : float = 1e-3,
        reg_covar    : float = COV_REG,
        random_state : Optional[int] = None,
    ):
        self.n_components = n_components
        self.n_iter       = n_iter
        self.tol          = tol
        self.reg_covar    = reg_covar
        self.random_state = random_state
        self.weights_     : Optional[np.ndarray] = None
        self.means_       : Optional[np.ndarray] = None
        self.covariances_ : Optional[np.ndarray] = None
        self.converged_   = False
        self.n_iter_done_ = 0
        self.log_likelihood_history_: List[float] = []

    def _init_params(self, X: np.ndarray) -> None:
        rng = np.random.default_rng(self.random_state)
        K   = self.n_components
        idx = rng.choice(X.shape[0], size=K, replace=False)
        self.means_       = X[idx].copy()
        self.weights_     = np.full(K, 1.0 / K)
        self.covariances_ = np.tile(np.var(X, axis=0) + self.reg_covar, (K, 1))

    def _log_gaussian_all(self, X: np.ndarray) -> np.ndarray:
        mu      = self.means_
        var     = self.covariances_
        log_det = np.sum(np.log(var), axis=1)
        diff    = X[:, None, :] - mu[None, :, :]
        maha    = np.sum(diff ** 2 / var[None, :, :], axis=2)
        log_norm = -0.5 * (X.shape[1] * np.log(2 * np.pi) + log_det)
        return log_norm[None, :] - 0.5 * maha

    def _e_step(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        log_w   = np.log(self.weights_ + LOG_EPS) + self._log_gaussian_all(X)
        lse     = np.max(log_w, axis=1, keepdims=True)
        log_sum = lse.reshape(-1) + np.log(np.sum(np.exp(log_w - lse), axis=1))
        return log_w - log_sum[:, None], log_sum

    def _m_step(self, X: np.ndarray, log_resp: np.ndarray) -> None:
        resp = np.exp(log_resp)
        Nk   = resp.sum(axis=0) + LOG_EPS
        self.weights_     = Nk / X.shape[0]
        self.means_       = (resp.T @ X) / Nk[:, None]
        diff              = X[:, None, :] - self.means_[None, :, :]
        self.covariances_ = (
            np.sum(resp[:, :, None] * diff ** 2, axis=0) / Nk[:, None]
            + self.reg_covar
        )

    def fit(self, X: np.ndarray) -> "GaussianMixtureModel":
        assert X.ndim == 2
        # [FIX-H] komponen adaptif: tidak lebih dari N/4 agar GMM tidak overfitting
        self.n_components = min(self.n_components, max(1, X.shape[0] // 4))
        self._init_params(X)
        self.log_likelihood_history_ = []
        prev_ll = -np.inf
        for it in range(self.n_iter):
            log_resp, log_sum = self._e_step(X)
            ll = float(log_sum.mean())
            self.log_likelihood_history_.append(ll)
            if abs(ll - prev_ll) < self.tol:
                self.converged_   = True
                self.n_iter_done_ = it + 1
                break
            prev_ll = ll
            self._m_step(X, log_resp)
        else:
            self.converged_   = False
            self.n_iter_done_ = self.n_iter
        return self

    def log_prob_per_frame(self, X: np.ndarray) -> np.ndarray:
        assert X.ndim == 2
        log_w = np.log(self.weights_ + LOG_EPS) + self._log_gaussian_all(X)
        lse   = np.max(log_w, axis=1, keepdims=True)
        return lse.reshape(-1) + np.log(np.sum(np.exp(log_w - lse), axis=1))


# =============================================================================
#  ALIGNMENT
# =============================================================================

def alignment_fonem_viterbi(
    fitur        : np.ndarray,
    fonem_seq    : List[str],
    gmm_per_fonem: Dict[str, "GaussianMixtureModel"],
) -> np.ndarray:
    """Alignment Viterbi dengan monophone GMM. Return frame-level FONEM_ID."""
    T = fitur.shape[0]
    n = len(fonem_seq)
    if n == 0 or T == 0:
        return np.full(T, SIL_ID, dtype=np.int32)

    skor = np.full((T, n), LOG_ZERO)
    for j, fonem in enumerate(fonem_seq):
        if fonem in gmm_per_fonem:
            skor[:, j] = gmm_per_fonem[fonem].log_prob_per_frame(fitur)
        elif fonem == "SIL":
            skor[:, j] = 0.0

    dp = np.full((T + 1, n + 1), LOG_ZERO)
    bp = np.zeros((T + 1, n + 1), dtype=np.int8)
    dp[0, 0] = 0.0

    for t in range(1, T + 1):
        for j in range(n):
            stay    = dp[t-1, j] + skor[t-1, j] if dp[t-1, j] > LOG_ZERO else LOG_ZERO
            advance = (dp[t-1, j-1] + skor[t-1, j]
                       if j > 0 and dp[t-1, j-1] > LOG_ZERO else LOG_ZERO)
            if stay >= advance:
                dp[t, j] = stay;    bp[t, j] = 0
            else:
                dp[t, j] = advance; bp[t, j] = 1

    valid_ends = [(dp[T, j], j) for j in range(n) if dp[T, j] > LOG_ZERO]
    j_end      = valid_ends[-1][1] if valid_ends else n - 1
    path       = np.zeros(T, dtype=int)
    j          = j_end
    for t in range(T, 0, -1):
        path[t - 1] = j
        if bp[t, j] == 1:
            j = max(j - 1, 0)

    return np.array(
        [FONEM2ID.get(fonem_seq[k], SIL_ID) for k in path],
        dtype=np.int32,
    )


def alignment_triphone_viterbi(
    fitur    : np.ndarray,
    fonem_seq: List[str],
    gmm_tri  : Dict[str, "GaussianMixtureModel"],
    gmm_mono : Dict[str, "GaussianMixtureModel"],
) -> np.ndarray:
    """
    [FIX-D] Alignment menggunakan triphone GMM dengan fallback ke monophone.
    Return frame-level FONEM_ID (center phoneme) untuk training TDNN.
    """
    triphone_seq = buat_triphone_seq(fonem_seq)
    T = fitur.shape[0]
    n = len(triphone_seq)
    if n == 0 or T == 0:
        return np.full(T, SIL_ID, dtype=np.int32)

    skor = np.full((T, n), LOG_ZERO)
    for j, tri in enumerate(triphone_seq):
        if tri in gmm_tri:
            skor[:, j] = gmm_tri[tri].log_prob_per_frame(fitur)
        else:
            # Fallback ke center monophone
            center = triphone_center(tri)
            if center in gmm_mono:
                skor[:, j] = gmm_mono[center].log_prob_per_frame(fitur)
            elif center == "SIL":
                skor[:, j] = 0.0

    dp = np.full((T + 1, n + 1), LOG_ZERO)
    bp = np.zeros((T + 1, n + 1), dtype=np.int8)
    dp[0, 0] = 0.0

    for t in range(1, T + 1):
        for j in range(n):
            stay    = dp[t-1, j] + skor[t-1, j] if dp[t-1, j] > LOG_ZERO else LOG_ZERO
            advance = (dp[t-1, j-1] + skor[t-1, j]
                       if j > 0 and dp[t-1, j-1] > LOG_ZERO else LOG_ZERO)
            if stay >= advance:
                dp[t, j] = stay;    bp[t, j] = 0
            else:
                dp[t, j] = advance; bp[t, j] = 1

    valid_ends = [(dp[T, j], j) for j in range(n) if dp[T, j] > LOG_ZERO]
    j_end      = valid_ends[-1][1] if valid_ends else n - 1
    path       = np.zeros(T, dtype=int)
    j          = j_end
    for t in range(T, 0, -1):
        path[t - 1] = j
        if bp[t, j] == 1:
            j = max(j - 1, 0)

    # Konversi triphone path ke center monophone ID
    centers = [triphone_center(triphone_seq[k]) for k in path]
    return np.array(
        [FONEM2ID.get(c, SIL_ID) for c in centers],
        dtype=np.int32,
    )


def _bangun_fonem_seq(label: str, lexicon: Dict[str, List[str]]) -> List[str]:
    fonem_seq: List[str] = []
    for kata in normalisasi(label).split():
        fonem_seq.extend(lexicon.get(kata, list(kata_ke_fonem(kata))))
    return fonem_seq


def kumpulkan_frame_fonem_naif(
    fitur    : np.ndarray,
    fonem_seq: List[str],
) -> Dict[str, np.ndarray]:
    T, n = fitur.shape[0], len(fonem_seq)
    if n == 0:
        return {}
    n_frame    = np.ones(n, dtype=int) * (T // n)
    n_frame[:T - n_frame.sum()] += 1
    hasil: Dict[str, np.ndarray] = {}
    idx = 0
    for i, f in enumerate(fonem_seq):
        seg = fitur[idx: idx + n_frame[i]]
        if seg.shape[0] > 0:
            hasil[f] = np.vstack([hasil[f], seg]) if f in hasil else seg.copy()
        idx += n_frame[i]
    return hasil


def kumpulkan_frame_per_fonem(
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon       : Dict[str, List[str]],
    gmm_mono      : Optional[Dict[str, GaussianMixtureModel]] = None,
    gmm_tri       : Optional[Dict[str, GaussianMixtureModel]] = None,
) -> Dict[str, np.ndarray]:
    """
    Kumpulkan frame per fonem (monophone) menggunakan alignment terbaik yang tersedia.
    Priority: triphone GMM > monophone GMM > naive alignment
    """
    use_tri  = USE_TRIPHONE and gmm_tri is not None and len(gmm_tri) > 0
    use_mono = gmm_mono is not None and len(gmm_mono) > 0
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)

    for audio_id, fitur in cache_mfcc.items():
        label = dict_transkrip.get(audio_id, "")
        if not label:
            continue
        fonem_seq = _bangun_fonem_seq(label, lexicon)
        if not fonem_seq:
            continue

        if use_tri:
            labels = alignment_triphone_viterbi(fitur, fonem_seq, gmm_tri, gmm_mono or {})
            for fid in np.unique(labels):
                buf[FONEM_JAWA[fid]].append(fitur[labels == fid])
        elif use_mono:
            labels = alignment_fonem_viterbi(fitur, fonem_seq, gmm_mono)
            for fid in np.unique(labels):
                buf[FONEM_JAWA[fid]].append(fitur[labels == fid])
        else:
            for fonem, arr in kumpulkan_frame_fonem_naif(fitur, fonem_seq).items():
                buf[fonem].append(arr)

    return {fonem: np.vstack(parts) for fonem, parts in buf.items()}


def kumpulkan_frame_per_triphone(
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon       : Dict[str, List[str]],
    gmm_mono      : Dict[str, GaussianMixtureModel],
) -> Dict[str, np.ndarray]:
    """
    [FIX-D] Kumpulkan frame per TRIPHONE menggunakan monophone alignment.
    Triphone jarang (< TRIPHONE_MIN_FRAME frame) ditandai untuk state tying.
    """
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)

    for audio_id, fitur in cache_mfcc.items():
        label = dict_transkrip.get(audio_id, "")
        if not label:
            continue
        fonem_seq = _bangun_fonem_seq(label, lexicon)
        if not fonem_seq:
            continue

        # Align dengan monophone GMM
        labels      = alignment_fonem_viterbi(fitur, fonem_seq, gmm_mono)
        triphones   = buat_triphone_seq(fonem_seq)

        # Terapkan alignment ke triphone frame
        n_tri  = len(triphones)
        T      = fitur.shape[0]
        seg_len = max(1, T // n_tri) if n_tri > 0 else T

        idx = 0
        for k, tri in enumerate(triphones):
            end  = idx + seg_len if k < n_tri - 1 else T
            seg  = fitur[idx:end]
            if seg.shape[0] > 0:
                buf[tri].append(seg)
            idx = end

    return {tri: np.vstack(parts) for tri, parts in buf.items()
            if np.vstack(parts).shape[0] >= TRIPHONE_MIN_FRAME}


def latih_gmm_per_fonem(
    frame_per_fonem: Dict[str, np.ndarray],
    n_components   : int   = N_COMPONENTS_GMM,
    n_iter         : int   = N_ITER_GMM,
    tol            : float = TOL_GMM,
    min_frame      : int   = MIN_FRAME_GMM,
) -> Dict[str, GaussianMixtureModel]:
    gmm_per_fonem: Dict[str, GaussianMixtureModel] = {}
    for fonem, X in sorted(frame_per_fonem.items()):
        if X.shape[0] < min_frame:
            continue
        gmm = GaussianMixtureModel(
            n_components=n_components,
            n_iter=n_iter, tol=tol, reg_covar=COV_REG,
            random_state=hash(fonem) % 100_000,
        )
        gmm.fit(X)
        gmm_per_fonem[fonem] = gmm
    return gmm_per_fonem


# ── [FIX-C] Iterative GMM training ───────────────────────────────────────────

def latih_gmm_iteratif(
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon       : Dict[str, List[str]],
    n_iters       : int = GMM_EM_ITERS,
) -> Tuple[Dict[str, GaussianMixtureModel], Dict[str, GaussianMixtureModel]]:
    """
    [FIX-C] Latih GMM monophone secara iteratif:
      Iter 1: naive alignment → GMM_1
      Iter 2: GMM_1 alignment → GMM_2
      Iter 3: GMM_2 alignment → GMM_3  (final)
    Kemudian latih triphone GMM di atas GMM_3.

    Return: (gmm_mono_final, gmm_tri_final)
    """
    gmm_mono: Dict[str, GaussianMixtureModel] = {}

    print("=" * 60)
    print("FASE 1: Iterative monophone GMM training")
    print("=" * 60)

    for it in range(n_iters):
        print(f"\n  GMM EM iter {it+1}/{n_iters} ...")
        frames_mono = kumpulkan_frame_per_fonem(
            cache_mfcc, dict_transkrip, lexicon,
            gmm_mono=gmm_mono if it > 0 else None,
            gmm_tri=None,
        )
        gmm_mono = latih_gmm_per_fonem(frames_mono)
        print(f"  → {len(gmm_mono)} model monophone dilatih")

    # Triphone GMM
    gmm_tri: Dict[str, GaussianMixtureModel] = {}
    if USE_TRIPHONE:
        print("\n" + "=" * 60)
        print("FASE 2: Triphone GMM training (dengan state tying)")
        print("=" * 60)
        frames_tri = kumpulkan_frame_per_triphone(
            cache_mfcc, dict_transkrip, lexicon, gmm_mono
        )
        gmm_tri = latih_gmm_per_fonem(
            frames_tri,
            n_components=max(2, N_COMPONENTS_GMM // 2),  # lebih kecil untuk triphone
            min_frame=TRIPHONE_MIN_FRAME,
        )
        print(f"  → {len(gmm_tri)} model triphone dilatih")
        print(f"  → Triphone jarang akan fallback ke monophone saat alignment")

    return gmm_mono, gmm_tri


def hitung_alignment_corpus(
    audio_ids     : List[str],
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon       : Dict[str, List[str]],
    gmm_mono      : Dict[str, GaussianMixtureModel],
    gmm_tri       : Optional[Dict[str, GaussianMixtureModel]] = None,
) -> Dict[str, np.ndarray]:
    """
    Hitung alignment untuk seluruh corpus menggunakan GMM terbaik yang tersedia.
    """
    alignment_cache: Dict[str, np.ndarray] = {}
    use_tri = USE_TRIPHONE and gmm_tri is not None and len(gmm_tri) > 0

    for audio_id in audio_ids:
        fitur = cache_mfcc.get(audio_id)
        label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label:
            continue
        fonem_seq = _bangun_fonem_seq(label, lexicon)
        if not fonem_seq:
            continue

        if use_tri:
            alignment_cache[audio_id] = alignment_triphone_viterbi(
                fitur, fonem_seq, gmm_tri, gmm_mono
            )
        else:
            alignment_cache[audio_id] = alignment_fonem_viterbi(
                fitur, fonem_seq, gmm_mono
            )

    return alignment_cache


# =============================================================================
#  DATASET
# =============================================================================

class FrameLevelDataset(Dataset):
    def __init__(
        self,
        audio_ids      : List[str],
        cache_mfcc     : Dict[str, np.ndarray],
        alignment_cache: Dict[str, np.ndarray],
    ):
        frame_parts: List[np.ndarray] = []
        label_parts: List[np.ndarray] = []
        for audio_id in audio_ids:
            fitur  = cache_mfcc.get(audio_id)
            aligns = alignment_cache.get(audio_id)
            if fitur is None or aligns is None:
                continue
            frame_parts.append(fitur)
            label_parts.append(aligns)

        if frame_parts:
            self.frames = torch.from_numpy(np.vstack(frame_parts))
            self.labels = torch.from_numpy(
                np.concatenate(label_parts).astype(np.int64)
            )
        else:
            self.frames = torch.zeros((0, FEAT_DIM), dtype=torch.float32)
            self.labels = torch.zeros(0, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.frames)

    def __getitem__(self, i: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.frames[i], self.labels[i]


# =============================================================================
#  TDNN
# =============================================================================

class TDNNLayer(nn.Module):
    """
    [FIX-B] Forward path tunggal dengan context splicing.
    Input bisa (B,D) atau (B,T,D) — keduanya ditangani konsisten.
    """
    def __init__(
        self,
        input_dim : int,
        output_dim: int,
        context   : List[int],
        dropout   : float = 0.0,
    ):
        super().__init__()
        self.context = sorted(context)
        self.linear  = nn.Linear(input_dim * len(context), output_dim)
        self.bn      = nn.BatchNorm1d(output_dim)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        squeeze_out = x.dim() == 2
        if squeeze_out:
            x = x.unsqueeze(1)

        B, T, _ = x.shape
        ctx_min  = min(self.context)
        ctx_max  = max(self.context)
        x_pad    = F.pad(x.permute(0, 2, 1),
                         (max(0, -ctx_min), max(0, ctx_max))).permute(0, 2, 1)

        ctx_frames = []
        for c in self.context:
            shift = c - ctx_min
            ctx_frames.append(x_pad[:, shift: shift + T, :])
        spliced = torch.cat(ctx_frames, dim=-1)

        out = self.linear(spliced)
        out = self.bn(out.permute(0, 2, 1)).permute(0, 2, 1)
        out = F.relu(out)
        out = self.drop(out)

        return out.squeeze(1) if squeeze_out else out


class TDNNAcousticModel(nn.Module):
    def __init__(
        self,
        input_dim : int                       = FEAT_DIM,
        n_classes : int                       = N_FONEM,
        hidden_dim: int                       = TDNN_HIDDEN,
        contexts  : Optional[List[List[int]]] = None,
        dropout   : float                     = TDNN_DROPOUT,
    ):
        super().__init__()
        if contexts is None:
            contexts = TDNN_CONTEXTS  # [FIX-B] 7 layers dengan input ±2
        layers, in_dim = [], input_dim
        for ctx in contexts:
            layers.append(TDNNLayer(in_dim, hidden_dim, ctx, dropout))
            in_dim = hidden_dim
        self.tdnn_layers = nn.ModuleList(layers)
        self.classifier  = nn.Linear(hidden_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.tdnn_layers:
            x = layer(x)
        return F.log_softmax(self.classifier(x), dim=-1)

    def get_posteriors_sequence(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            return self.forward(x)[0]


def latih_tdnn_frame_level(
    train_ids      : List[str],
    val_ids        : List[str],
    cache_mfcc     : Dict[str, np.ndarray],
    alignment_train: Dict[str, np.ndarray],
    alignment_val  : Dict[str, np.ndarray],
    dir_save       : str,
    epochs         : int                       = TDNN_EPOCHS,
    lr             : float                     = TDNN_LR,
    batch_size     : int                       = TDNN_BATCH_SIZE,
    hidden_dim     : int                       = TDNN_HIDDEN,
    contexts       : Optional[List[List[int]]] = None,
    dropout        : float                     = TDNN_DROPOUT,
    warmup_epochs  : int                       = TDNN_WARMUP_EPOCH,
) -> TDNNAcousticModel:
    """
    [FIX-B] TDNN dengan konteks yang benar.
    [FIX-I] LR warmup + cosine decay untuk training yang lebih stabil.
    """
    if contexts is None:
        contexts = TDNN_CONTEXTS
    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = os.path.join(dir_save, "model_tdnn_fonem.pt")
    os.makedirs(dir_save, exist_ok=True)

    model = TDNNAcousticModel(
        input_dim=FEAT_DIM, n_classes=N_FONEM,
        hidden_dim=hidden_dim, contexts=contexts, dropout=dropout,
    ).to(device)

    if os.path.isfile(model_path):
        try:
            model.load_state_dict(torch.load(model_path, map_location=device))
            print(f"  Model dimuat dari checkpoint: {model_path}")
            return model
        except Exception:
            os.remove(model_path)

    train_ds     = FrameLevelDataset(train_ids, cache_mfcc, alignment_train)
    val_ds       = FrameLevelDataset(val_ids,   cache_mfcc, alignment_val)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=0, pin_memory=True)

    if len(train_ds) == 0:
        warnings.warn("Training dataset kosong!")
        return model

    # Class weights: sqrt smoothing
    label_counts = torch.bincount(train_ds.labels, minlength=N_FONEM).float()
    label_counts = torch.clamp(label_counts, min=1.0)
    weights      = 1.0 / torch.sqrt(label_counts)
    weights      = (weights / weights.sum()).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # [FIX-I] LR schedule: linear warmup → cosine decay
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(warmup_epochs)
        progress = (epoch - warmup_epochs) / max(epochs - warmup_epochs, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    loss_fn   = nn.NLLLoss(weight=weights)

    riwayat        : List[dict] = []
    best_val_loss   = float("inf")

    print(f"\n  Device: {device} | Train frames: {len(train_ds):,} | Val: {len(val_ds):,}")

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, n_correct, n_total = 0.0, 0, 0
        for feats, lbls in train_loader:
            feats = feats.to(device)
            lbls  = lbls.to(device)
            optimizer.zero_grad()
            log_probs = model(feats)
            loss      = loss_fn(log_probs, lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * len(lbls)
            n_correct  += (log_probs.argmax(dim=1) == lbls).sum().item()
            n_total    += len(lbls)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for feats, lbls in val_loader:
                feats = feats.to(device)
                lbls  = lbls.to(device)
                lp    = model(feats)
                val_loss    += loss_fn(lp, lbls).item() * len(lbls)
                val_correct += (lp.argmax(dim=1) == lbls).sum().item()
                val_total   += len(lbls)

        scheduler.step()

        avg_train = total_loss / max(n_total,   1)
        avg_val   = val_loss   / max(val_total, 1)
        acc_train = 100 * n_correct   / max(n_total,   1)
        acc_val   = 100 * val_correct / max(val_total, 1)
        cur_lr    = optimizer.param_groups[0]['lr']

        riwayat.append({
            "Epoch"        : epoch,
            "LR"           : round(cur_lr, 7),
            "Train Loss"   : round(avg_train, 5),
            "Val Loss"     : round(avg_val,   5),
            "Train Acc (%)": round(acc_train, 2),
            "Val Acc (%)"  : round(acc_val,   2),
        })

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs} | "
                  f"Loss {avg_train:.4f}/{avg_val:.4f} | "
                  f"Acc {acc_train:.1f}/{acc_val:.1f}% | "
                  f"LR {cur_lr:.2e}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), model_path)

    if os.path.isfile(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))

    pd.DataFrame(riwayat).to_csv(
        os.path.join(dir_save, "HasilTrainingTDNN_Fonem.csv"), index=False,
    )
    return model


# =============================================================================
#  HMM VITERBI DECODER
# =============================================================================

class HMMViterbiDecoder:
    """
    [FIX-A] N_HMM_STATES = 4
    [FIX-F] Cross-phone transition: 0.1/F → 0.4/F
    [FIX-G] Non-final states: normalize 0.65+0.35=1.0
    """

    # [FIX-G] Probabilitas untuk non-final state (hanya self + advance)
    _LOG_SELF_NF   = math.log(0.65)   # normalized: 0.65/(0.65+0.35)
    _LOG_ADV_NF    = math.log(0.35)   # normalized: 0.35/(0.65+0.35)

    # Probabilitas untuk last state (self + cross-phoneme)
    _LOG_SELF_LAST = math.log(HMM_P_SELF)
    _LOG_ADV_LAST  = math.log(HMM_P_ADVANCE)
    # Cross dihitung dinamis: HMM_P_CROSS_TOTAL / F

    def __init__(
        self,
        model        : TDNNAcousticModel,
        lexicon      : Dict[str, List[str]],
        ngram_model  : "NGramModel",
        gmm_per_fonem: Dict[str, GaussianMixtureModel],
        device       : Optional[torch.device] = None,
        n_hmm_states : int = N_HMM_STATES,
        beam_width   : int = BEAM_WIDTH,
    ):
        self.model         = model
        self.lexicon       = lexicon
        self.rev_lex: Dict[Tuple[str, ...], List[str]] = defaultdict(list)
        for kata, fonems in lexicon.items():
            self.rev_lex[tuple(fonems)].append(kata)
        self.ngram         = ngram_model
        self.gmm_per_fonem = gmm_per_fonem
        self.device        = device or torch.device("cpu")
        self.n_hmm_states  = n_hmm_states
        self.beam_width    = beam_width
        self.model.to(self.device)
        self.model.eval()

    def _get_posteriors(self, fitur: np.ndarray) -> np.ndarray:
        t = torch.from_numpy(fitur).unsqueeze(0).to(self.device)
        return self.model.get_posteriors_sequence(t).cpu().numpy()

    def _viterbi_hmm(self, log_post: np.ndarray) -> List[str]:
        """
        [FIX-A] n_hmm_states = 4
        [FIX-F] log_cross = log(HMM_P_CROSS_TOTAL / F) — lebih permissive
        [FIX-G] Non-final states: LOG_SELF_NF + LOG_ADV_NF = log(1.0)
        """
        T, F   = log_post.shape
        S      = self.n_hmm_states
        NS     = F * S

        # [FIX-F] Cross-phone lebih permissive (0.4/F bukan 0.1/F)
        log_cross = math.log(max(HMM_P_CROSS_TOTAL / max(F, 1), 1e-15))

        dp = np.full((T, NS), LOG_ZERO, dtype=np.float64)
        bp = np.full((T, NS), -1,       dtype=np.int32)

        # Inisialisasi: semua fonem bisa menjadi fonem pertama
        for fid in range(F):
            dp[0, fid * S] = log_post[0, fid] + math.log(1.0 / F)

        for t in range(1, T):
            # Precompute: nilai terbaik dari last state semua fonem (untuk cross-phone)
            # Optimasi: O(F) bukan O(F²)
            best_cross_val = LOG_ZERO
            for prev_fid in range(F):
                v = dp[t-1, prev_fid * S + (S - 1)]
                if v > best_cross_val:
                    best_cross_val = v

            for fid in range(F):
                emit = log_post[t, fid]

                for s in range(S):
                    cur = fid * S + s

                    # Kandidat 1: self-loop
                    candidates = [(dp[t-1, cur] + (self._LOG_SELF_NF if s < S-1 else self._LOG_SELF_LAST),
                                   cur)]

                    # Kandidat 2: advance dari state sebelumnya dalam fonem
                    if s > 0:
                        prev_in = fid * S + (s - 1)
                        adv_score = dp[t-1, prev_in] + (self._LOG_ADV_NF if s < S-1 else self._LOG_ADV_LAST)
                        candidates.append((adv_score, prev_in))

                    # Kandidat 3: cross dari last state fonem lain (hanya ke state 0)
                    if s == 0:
                        # Iterasi semua fonem untuk cross transition
                        for prev_fid in range(F):
                            prev_last = prev_fid * S + (S - 1)
                            cross_score = dp[t-1, prev_last] + log_cross
                            candidates.append((cross_score, prev_last))

                    best_score, best_prev = max(candidates, key=lambda x: x[0])

                    if best_score > LOG_ZERO:
                        dp[t, cur] = best_score + emit
                        bp[t, cur] = best_prev

        # Backtrack
        best_end    = int(np.argmax(dp[T - 1]))
        path_states = [best_end]
        for t in range(T - 1, 0, -1):
            prev = bp[t, path_states[-1]]
            if prev < 0:
                break
            path_states.append(prev)
        path_states.reverse()

        path_fonem = [ps // S for ps in path_states]

        # Collapse dengan minimum duration [FIX-1 dari v1]
        result: List[str] = []
        i = 0
        while i < len(path_fonem):
            fid = path_fonem[i]
            j   = i + 1
            while j < len(path_fonem) and path_fonem[j] == fid:
                j += 1
            if (j - i) >= MIN_FONEM_DUR:
                result.append(FONEM_JAWA[fid])
            i = j

        return [f for f in result if f != "SIL"]

    def _fonem_seq_ke_kata(
        self,
        fonem_seq: List[str],
        T        : int,
    ) -> List[List[str]]:
        n         = len(fonem_seq)
        max_words = max(1, T // AVG_FRAMES_PER_WORD)
        if n == 0:
            return [[]]

        dp: List[List[Tuple[float, List[str]]]] = [[] for _ in range(n + 1)]
        dp[0] = [(0.0, [])]

        for i in range(n):
            if not dp[i]:
                continue
            j_max = min(n + 1, i + MAX_FONEM_PER_KATA + 1)
            for j in range(i + 1, j_max):
                sub = tuple(fonem_seq[i:j])
                if sub not in self.rev_lex:
                    continue
                for kata in self.rev_lex[sub]:
                    for score, words in dp[i]:
                        if len(words) >= max_words:
                            continue
                        new_words = words + [kata]
                        new_score = score + self.ngram.log_prob(
                            kata, tuple(words[-(self.ngram.n - 1):])
                        )
                        dp[j].append((new_score, new_words))

                if len(dp[j]) > self.beam_width * 3:
                    dp[j].sort(key=lambda x: x[0], reverse=True)
                    dp[j] = dp[j][: self.beam_width]

        if dp[n]:
            dp[n].sort(key=lambda x: x[0], reverse=True)
            return [w for _, w in dp[n][: self.beam_width]]
        return [fonem_seq]

    def dekode(self, fitur: np.ndarray) -> List[str]:
        if fitur.shape[0] == 0:
            return []
        self.model.eval()
        log_post      = self._get_posteriors(fitur)
        fonem_seq     = self._viterbi_hmm(log_post)
        if not fonem_seq:
            return []
        kata_kandidat = self._fonem_seq_ke_kata(fonem_seq, T=fitur.shape[0])
        if not kata_kandidat:
            return fonem_seq
        return self.ngram.rescore(kata_kandidat)


# =============================================================================
#  N-GRAM LM
# =============================================================================

class NGramModel:
    def __init__(self, n: int = 3, discount: float = 0.75):
        self.n              = n
        self.discount       = discount
        self.ngram_count   : Dict[Tuple, int] = defaultdict(int)
        self.context_count : Dict[Tuple, int] = defaultdict(int)
        self.unigram_count : Dict[str,   int] = defaultdict(int)
        self.vocab         : set              = set()
        self.N_total        = 0
        self._ctx_div      : Dict[Tuple, int] = {}

    def _tokenize(self, kalimat: str) -> List[str]:
        tokens = normalisasi(kalimat).split()
        return ["<s>"] * (self.n - 1) + tokens + ["</s>"]

    def fit(self, korpus: List[str]) -> "NGramModel":
        ctx_words: Dict[Tuple, set] = defaultdict(set)
        for kalimat in korpus:
            tokens = self._tokenize(kalimat)
            self.vocab.update(tokens)
            self.N_total += len(tokens)
            for tok in tokens:
                self.unigram_count[tok] += 1
            for i in range(len(tokens) - self.n + 1):
                ngram = tuple(tokens[i: i + self.n])
                ctx   = ngram[:-1]
                self.ngram_count[ngram]  += 1
                self.context_count[ctx]  += 1
                ctx_words[ctx].add(ngram[-1])
        self._ctx_div = {c: len(w) for c, w in ctx_words.items()}
        return self

    def log_prob(self, kata: str, context: Tuple) -> float:
        ngram = context + (kata,)
        C_ng  = self.ngram_count.get(ngram, 0)
        C_ctx = self.context_count.get(context, 0)
        C_uni = self.unigram_count.get(kata, 0)
        p_uni = (C_uni + 1) / (self.N_total + len(self.vocab))

        if C_ctx == 0:
            return math.log(p_uni + LOG_EPS)

        lam  = self.discount * self._ctx_div.get(context, 0) / C_ctx
        prob = max(C_ng - self.discount, 0) / C_ctx + lam * p_uni
        return math.log(prob + LOG_EPS)

    def score_sequence(self, kata_list: List[str]) -> float:
        tokens = ["<s>"] * (self.n - 1) + kata_list + ["</s>"]
        log_p  = 0.0
        for i in range(self.n - 1, len(tokens)):
            log_p += self.log_prob(
                tokens[i], tuple(tokens[i - self.n + 1: i])
            )
        return log_p

    def rescore(self, kandidat: List[List[str]]) -> List[str]:
        if not kandidat:
            return []
        return max(kandidat, key=self.score_sequence)


# =============================================================================
#  EVALUASI
# =============================================================================

def compute_wer(ref: List[str], hyp: List[str]) -> Dict:
    R, H = len(ref), len(hyp)
    dp   = [[0] * (H + 1) for _ in range(R + 1)]
    ops  = [[""] * (H + 1) for _ in range(R + 1)]
    for i in range(R + 1): dp[i][0] = i;  ops[i][0] = "D" * i
    for j in range(H + 1): dp[0][j] = j;  ops[0][j] = "I" * j
    for i in range(1, R + 1):
        for j in range(1, H + 1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]; ops[i][j] = "M"
            else:
                sub  = dp[i-1][j-1] + 1
                dlt  = dp[i-1][j]   + 1
                ins  = dp[i][j-1]   + 1
                best = min(sub, dlt, ins)
                dp[i][j]  = best
                ops[i][j] = "S" if best == sub else ("D" if best == dlt else "I")
    S = D = I = 0
    i, j = R, H
    while i > 0 or j > 0:
        op = ops[i][j]
        if   op == "M": i -= 1; j -= 1
        elif op == "S": S += 1; i -= 1; j -= 1
        elif op == "D": D += 1; i -= 1
        elif op == "I": I += 1; j -= 1
        else:
            if i > 0: D += 1; i -= 1
            else:     I += 1; j -= 1
    wer = (S + D + I) / R if R > 0 else 0.0
    return {"wer_persen": round(wer * 100, 2), "S": S, "D": D, "I": I, "N": R}


def dekode_semua_audio(
    cache_mfcc    : Dict[str, np.ndarray],
    audio_ids     : List[str],
    dict_transkrip: Dict[str, str],
    decoder       : HMMViterbiDecoder,
) -> List[dict]:
    hasil_wer: List[dict] = []
    for audio_id in audio_ids:
        fitur = cache_mfcc.get(audio_id)
        label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label:
            continue
        ref_kata = normalisasi(label).split()
        if not ref_kata:
            continue
        try:
            hyp_kata = decoder.dekode(fitur)
        except (RuntimeError, MemoryError):
            continue
        wer_res = compute_wer(ref_kata, hyp_kata)
        hasil_wer.append({
            "Id"       : audio_id,
            "Referensi": " ".join(ref_kata),
            "Hipotesis": " ".join(hyp_kata),
            "WER (%)"  : wer_res["wer_persen"],
            "S"        : wer_res["S"],
            "D"        : wer_res["D"],
            "I"        : wer_res["I"],
            "N"        : wer_res["N"],
        })
    return hasil_wer


def ringkasan_wer(df_wer: pd.DataFrame) -> Dict:
    if df_wer.empty:
        return {}
    S = int(df_wer["S"].sum())
    D = int(df_wer["D"].sum())
    I = int(df_wer["I"].sum())
    N = int(df_wer["N"].sum())
    wer_c = (S + D + I) / N * 100 if N > 0 else 100.0
    return {
        "wer_corpus" : round(wer_c, 2),
        "wer_mean"   : round(df_wer["WER (%)"].mean(),   2),
        "wer_median" : round(df_wer["WER (%)"].median(), 2),
        "wer_std"    : round(df_wer["WER (%)"].std(),    2),
        "total_S"    : S, "total_D": D, "total_I": I, "total_N": N,
    }


# =============================================================================
#  [FIX-J] MAIN PIPELINE — urutan yang benar
# =============================================================================

def main():
    """
    Pipeline lengkap dengan semua perbaikan.

    Urutan yang benar:
    1. Load CSV + cache MFCC
    2. Speaker CMVN  ← [FIX-E] normalisasi level speaker
    3. Split train/val per speaker
    4. Iterative GMM training (3x EM)  ← [FIX-C]
       4a. Naive → GMM_1 → GMM_2 → GMM_3 (monophone)
       4b. GMM_3 → Triphone GMM  ← [FIX-D]
    5. Hitung alignment dengan GMM terbaik
    6. Latih TDNN (7 layer, konteks benar)  ← [FIX-B]
    7. Latih N-gram LM
    8. Decode + evaluasi
    """
    os.makedirs(DIR_SAVE, exist_ok=True)
    random.seed(42)
    np.random.seed(42)

    # ── Step 1: Load data ─────────────────────────────────────────────────────
    print("\n[1/8] Membaca CSV dan cache MFCC ...")
    dict_transkrip, dict_speaker = baca_transkrip_csv(CSV_PATH)
    cache_mfcc_raw, meta         = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    print(f"  {len(cache_mfcc_raw)} audio dimuat | "
          f"{len(set(dict_speaker.values()))} speaker")

    # ── Step 2: Speaker CMVN ─────────────────────────────────────────────────
    print("\n[2/8] Menerapkan speaker-level CMVN ...")
    cache_mfcc = apply_cmvn_speaker(cache_mfcc_raw, dict_speaker)

    # ── Step 3: Split ─────────────────────────────────────────────────────────
    print("\n[3/8] Split train/val per speaker ...")
    all_ids             = list(cache_mfcc.keys())
    train_ids, val_ids  = split_by_speaker(all_ids, dict_speaker, VAL_RATIO)
    print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")

    # ── Step 4: Lexicon + N-gram LM ───────────────────────────────────────────
    print("\n[4/8] Membangun lexicon dan N-gram LM ...")
    semua_transkrip = list(dict_transkrip.values())
    lexicon         = buat_lexicon(semua_transkrip)
    lm              = NGramModel(n=N_GRAM)
    lm.fit([dict_transkrip[aid] for aid in train_ids
            if aid in dict_transkrip])
    print(f"  Lexicon: {len(lexicon)} kata | Vocab LM: {len(lm.vocab)}")

    # ── Step 5: Iterative GMM [FIX-C, FIX-D] ─────────────────────────────────
    print("\n[5/8] Iterative GMM training ...")
    # Gunakan hanya train_ids untuk GMM training
    cache_train = {k: cache_mfcc[k] for k in train_ids if k in cache_mfcc}
    gmm_mono, gmm_tri = latih_gmm_iteratif(
        cache_train, dict_transkrip, lexicon, n_iters=GMM_EM_ITERS
    )

    # ── Step 6: Alignment ─────────────────────────────────────────────────────
    print("\n[6/8] Menghitung alignment corpus ...")
    align_train = hitung_alignment_corpus(
        train_ids, cache_mfcc, dict_transkrip, lexicon, gmm_mono, gmm_tri
    )
    align_val   = hitung_alignment_corpus(
        val_ids,   cache_mfcc, dict_transkrip, lexicon, gmm_mono, gmm_tri
    )
    print(f"  Train aligned: {len(align_train)} | Val: {len(align_val)}")

    # ── Step 7: Latih TDNN [FIX-B, FIX-I] ───────────────────────────────────
    print("\n[7/8] Melatih TDNN ...")
    model = latih_tdnn_frame_level(
        train_ids, val_ids,
        cache_mfcc, align_train, align_val,
        DIR_SAVE,
    )

    # ── Step 8: Decode + Evaluasi ─────────────────────────────────────────────
    print("\n[8/8] Dekode dan evaluasi ...")
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    decoder = HMMViterbiDecoder(
        model=model,
        lexicon=lexicon,
        ngram_model=lm,
        gmm_per_fonem=gmm_mono,
        device=device,
        n_hmm_states=N_HMM_STATES,   # [FIX-A] = 4
        beam_width=BEAM_WIDTH,
    )

    print("  Dekode train set ...")
    hasil_train = dekode_semua_audio(cache_mfcc, train_ids, dict_transkrip, decoder)
    print("  Dekode val set ...")
    hasil_val   = dekode_semua_audio(cache_mfcc, val_ids,   dict_transkrip, decoder)

    df_train = pd.DataFrame(hasil_train)
    df_val   = pd.DataFrame(hasil_val)

    ring_train = ringkasan_wer(df_train)
    ring_val   = ringkasan_wer(df_val)

    print("\n" + "=" * 60)
    print("HASIL AKHIR")
    print("=" * 60)
    print(f"  Train WER corpus : {ring_train.get('wer_corpus', 'N/A')}%")
    print(f"  Val   WER corpus : {ring_val.get('wer_corpus',   'N/A')}%")
    print(f"  Val   WER mean   : {ring_val.get('wer_mean',     'N/A')}%")
    print(f"  Val   S/D/I/N    : "
          f"{ring_val.get('total_S')}/{ring_val.get('total_D')}/"
          f"{ring_val.get('total_I')}/{ring_val.get('total_N')}")
    print("=" * 60)

    # Simpan hasil
    df_train.to_csv(os.path.join(DIR_SAVE, "HasilWER_Train.csv"), index=False)
    df_val.to_csv(  os.path.join(DIR_SAVE, "HasilWER_Val.csv"),   index=False)
    pd.DataFrame([{"set": "train", **ring_train},
                  {"set": "val",   **ring_val}]).to_csv(
        os.path.join(DIR_SAVE, "RingkasanWER.csv"), index=False
    )
    print(f"\n  Hasil disimpan di: {DIR_SAVE}")
    return model, decoder, ring_val


if __name__ == "__main__":
    main()


[1/8] Membaca CSV dan cache MFCC ...
  1749 audio dimuat | 0 speaker

[2/8] Menerapkan speaker-level CMVN ...

[3/8] Split train/val per speaker ...
  Train: 1400 | Val: 349

[4/8] Membangun lexicon dan N-gram LM ...
  Lexicon: 5006 kata | Vocab LM: 4274

[5/8] Iterative GMM training ...
FASE 1: Iterative monophone GMM training

  GMM EM iter 1/3 ...
  → 26 model monophone dilatih

  GMM EM iter 2/3 ...
  → 27 model monophone dilatih

  GMM EM iter 3/3 ...
  → 27 model monophone dilatih

FASE 2: Triphone GMM training (dengan state tying)
  → 3554 model triphone dilatih
  → Triphone jarang akan fallback ke monophone saat alignment

[6/8] Menghitung alignment corpus ...
  Train aligned: 1400 | Val: 349

[7/8] Melatih TDNN ...

  Device: cpu | Train frames: 798,123 | Val: 199,955
  Epoch   1/50 | Loss 3.4249/3.0722 | Acc 14.2/18.6% | LR 2.00e-04
  Epoch   5/50 | Loss 3.0897/3.0254 | Acc 15.5/17.2% | LR 5.00e-04
  Epoch  10/50 | Loss 2.9713/3.0554 | Acc 14.7/15.0% | LR 4.85e-04
  Epoch  1

In [6]:
# =============================================================================
#  HYBRID TDNN-HMM ASR — Bahasa Jawa — v3 (PERBAIKAN DARI v2)
#
#  PERBAIKAN KRITIS:
#
#  [FIX-1] TDNN overfit parah (val acc 18.6%→10.3%):
#          Naikkan dropout 0.1→0.3, tambah weight decay 1e-3,
#          tambah label smoothing pada loss, kurangi hidden 512→256
#
#  [FIX-2] Speaker CMVN gagal (0 speaker terdeteksi):
#          Cek ketersediaan speaker sebelum apply CMVN,
#          jika tidak ada speaker → skip speaker CMVN, hanya per-utterance
#
#  [FIX-3] LR cosine turun ke 0 di epoch terakhir:
#          Ganti LambdaLR cosine → CosineAnnealingLR dengan eta_min=1e-6
#          sehingga LR tidak pernah mencapai nol
#
#  [FIX-4] Alignment Viterbi O(T×n) Python loop lambat:
#          Vektorisasi dengan NumPy broadcasting, eliminasi inner Python loop
#
#  [FIX-5] HMM decoder O(F²×S×T) terlalu berat:
#          Sederhanakan: gunakan frame-level argmax + CTC collapse + fonem→kata
#          Tambah batasan max_words dari durasi audio (AVG_FRAMES_PER_WORD)
#
#  [FIX-6] FrameLevelDataset buat ulang alignment setiap kali:
#          Terima pre-computed alignment_cache langsung, tidak alignment ulang
# =============================================================================

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\hybrid\src\TDNN-HMM\out_v3"
)

COL_SPEAKER = "speaker_id"

# ── GMM ───────────────────────────────────────────────────────────────────────
N_COMPONENTS_GMM   = 8
N_ITER_GMM         = 100
TOL_GMM            = 1e-4
MIN_FRAME_GMM      = 20
GMM_EM_ITERS       = 3

# ── TDNN ──────────────────────────────────────────────────────────────────────
TDNN_HIDDEN        = 256     # [FIX-1] 512→256, kurangi kapasitas untuk cegah overfit
TDNN_DROPOUT       = 0.3     # [FIX-1] 0.1→0.3, regularisasi lebih kuat
TDNN_WEIGHT_DECAY  = 1e-3    # [FIX-1] tambah L2 regularisasi
TDNN_LABEL_SMOOTH  = 0.1     # [FIX-1] label smoothing
TDNN_EPOCHS        = 60
TDNN_LR            = 5e-4
TDNN_LR_MIN        = 1e-6    # [FIX-3] eta_min agar LR tidak pernah nol
TDNN_BATCH_SIZE    = 512
TDNN_WARMUP_EPOCH  = 5
TDNN_CONTEXTS      = [
    [-2, -1, 0, 1, 2],
    [-1,  0, 1],
    [-1,  0, 1],
    [-3,  0, 3],
    [-3,  0, 3],
    [-6,  0, 6],
]

# ── HMM / Decode ─────────────────────────────────────────────────────────────
N_HMM_STATES       = 4
BEAM_WIDTH         = 8
AVG_FRAMES_PER_WORD = 30    # estimasi panjang kata dalam frame
MIN_FONEM_DUR      = 2      # CTC: min frame per fonem setelah collapse
MAX_FONEM_PER_KATA = 8

# ── LM ───────────────────────────────────────────────────────────────────────
N_GRAM             = 3
VAL_RATIO          = 0.2
SPEAKER_SPLIT      = True

# ── Triphone ─────────────────────────────────────────────────────────────────
USE_TRIPHONE       = True
TRIPHONE_MIN_FRAME = 30

LOG_ZERO = -1e15
LOG_EPS  = 1e-300
COV_REG  = 1e-6
FEAT_EPS = 1e-8

# =============================================================================
#  IMPORT
# =============================================================================
import os, re, glob, math, time, random, warnings
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    _TORCH = True
except ImportError:
    raise ImportError("PyTorch diperlukan: pip install torch")

try:
    import librosa; _LIBROSA = True
except ImportError:
    _LIBROSA = False

try:
    from scipy.io import wavfile as _wavfile; _SCIPY = True
except ImportError:
    _SCIPY = False

warnings.filterwarnings("ignore")

# =============================================================================
#  FONEM
# =============================================================================
FONEM_JAWA: List[str] = [
    "a","i","u","e","o","eu","ae",
    "p","b","t","d","k","g","dh","th",
    "m","n","ng","ny",
    "s","h","f",
    "j","c",
    "l","r",
    "w","y",
    "SIL",
]
FONEM2ID: Dict[str, int] = {f: i for i, f in enumerate(FONEM_JAWA)}
N_FONEM: int = len(FONEM_JAWA)
SIL_ID: int  = FONEM2ID["SIL"]

_G2P_DIGRAF = {"dh":"dh","th":"th","ng":"ng","ny":"ny"}
_kf_cache: Dict[str, Tuple[str,...]] = {}
_norm_cache: Dict[str, str] = {}

def kata_ke_fonem(kata: str) -> Tuple[str,...]:
    if kata in _kf_cache: return _kf_cache[kata]
    fonem, i = [], 0
    while i < len(kata):
        if i+1 < len(kata) and kata[i:i+2] in _G2P_DIGRAF:
            fonem.append(_G2P_DIGRAF[kata[i:i+2]]); i += 2
        elif kata[i] in FONEM2ID:
            fonem.append(kata[i]); i += 1
        else:
            i += 1
    result = tuple(fonem) if fonem else ("SIL",)
    _kf_cache[kata] = result
    return result

def normalisasi(teks: str) -> str:
    if teks in _norm_cache: return _norm_cache[teks]
    t = re.sub(r"\s+"," ", re.sub(r"[^a-z\s]","", teks.lower().strip())).strip()
    _norm_cache[teks] = t
    return t

def buat_lexicon(transkrip_list):
    lex = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lex:
                lex[kata] = list(kata_ke_fonem(kata))
    return lex

def _bangun_fonem_seq(label, lexicon):
    fs = []
    for kata in normalisasi(label).split():
        fs.extend(lexicon.get(kata, list(kata_ke_fonem(kata))))
    return fs

# =============================================================================
#  TRIPHONE
# =============================================================================
def buat_triphone_seq(fonem_seq):
    if not fonem_seq: return []
    p = ["SIL"] + list(fonem_seq) + ["SIL"]
    return [f"{p[i-1]}-{p[i]}+{p[i+1]}" for i in range(1, len(p)-1)]

def triphone_center(tri):
    try: return tri.split("-")[1].split("+")[0]
    except: return tri

# =============================================================================
#  CSV & SPLIT
# =============================================================================
def baca_transkrip_csv(csv_path, col_speaker=COL_SPEAKER):
    if not os.path.isfile(csv_path): raise FileNotFoundError(csv_path)
    df = None
    for enc in ("utf-8","utf-8-sig","latin-1","cp1252"):
        try: df = pd.read_csv(csv_path, encoding=enc); break
        except: continue
    if df is None: raise ValueError(f"Gagal baca CSV: {csv_path}")
    df.columns = [c.strip() for c in df.columns]
    col_path = next((c for c in ["Path_Data","path_data","path","audio_path","filename"] if c in df.columns), None)
    col_text = next((c for c in ["sentences","sentence","text","transcription","label"] if c in df.columns), None)
    if not col_path: raise ValueError(f"Kolom path tidak ada. Kolom: {list(df.columns)}")
    if not col_text: raise ValueError(f"Kolom teks tidak ada. Kolom: {list(df.columns)}")
    dt, ds = {}, {}
    has_spk = bool(col_speaker and col_speaker in df.columns)
    for _, row in df.iterrows():
        pv = str(row[col_path]).strip(); tv = str(row[col_text]).strip()
        if pv in ("nan","") or tv in ("nan",""): continue
        aid = os.path.splitext(os.path.basename(pv))[0]
        dt[aid] = tv.lower()
        if has_spk: ds[aid] = str(row[col_speaker]).strip()
    n_spk = len(set(ds.values())) if ds else 0
    print(f"  CSV: {len(dt)} audio | {n_spk} speaker unik")
    return dt, ds

def split_by_speaker(audio_ids, dict_speaker, val_ratio=VAL_RATIO):
    ids = list(audio_ids)
    if not dict_speaker:
        random.shuffle(ids); n_val = max(1, int(len(ids)*val_ratio))
        return ids[n_val:], ids[:n_val]
    spk2ids = defaultdict(list)
    for aid in ids: spk2ids[dict_speaker.get(aid,"unknown")].append(aid)
    speakers = list(spk2ids.keys()); random.shuffle(speakers)
    n_val_spk = max(1, int(len(speakers)*val_ratio))
    val_spk = set(speakers[:n_val_spk])
    train_ids, val_ids = [], []
    for spk, sids in spk2ids.items():
        (val_ids if spk in val_spk else train_ids).extend(sids)
    print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")
    return train_ids, val_ids

# =============================================================================
#  MFCC
# =============================================================================
_SR=16000; _N_MFCC=13; _N_FFT=512; _HOP_LENGTH=160
_WIN_LENGTH=400; _N_MELS=26; _PRE_EMPH=0.97
_FMIN=0.0; _FMAX=4000.0; _DELTA_W=2
FEAT_DIM = _N_MFCC * 3

def apply_cmvn(f):
    return (f - f.mean(0, keepdims=True)) / (f.std(0, keepdims=True) + FEAT_EPS)

def apply_cmvn_speaker(cache_mfcc, dict_speaker):
    """
    [FIX-2] Hanya apply jika ada speaker yang terdefinisi.
    Jika 0 speaker → kembalikan cache_mfcc tidak berubah.
    """
    if not dict_speaker:
        print("  [FIX-2] Tidak ada kolom speaker → skip speaker CMVN")
        return cache_mfcc

    spk_frames = defaultdict(list)
    for aid, feat in cache_mfcc.items():
        spk_frames[dict_speaker.get(aid,"unknown")].append(feat)

    # Hanya lakukan jika ada lebih dari 1 speaker unik
    unique_spks = set(dict_speaker.values())
    if len(unique_spks) <= 1:
        print(f"  [FIX-2] Hanya {len(unique_spks)} speaker → skip speaker CMVN")
        return cache_mfcc

    spk_mean, spk_std = {}, {}
    for spk, frames in spk_frames.items():
        all_f = np.vstack(frames)
        spk_mean[spk] = all_f.mean(0); spk_std[spk] = all_f.std(0) + FEAT_EPS

    result = {}
    for aid, feat in cache_mfcc.items():
        spk = dict_speaker.get(aid,"unknown")
        result[aid] = apply_cmvn((feat - spk_mean[spk]) / spk_std[spk])
    print(f"  Speaker CMVN diterapkan: {len(unique_spks)} speaker")
    return result

def _delta_numpy(feat, w=_DELTA_W):
    T = feat.shape[0]; den = 2.0*sum(n*n for n in range(1,w+1))
    pad = np.pad(feat, ((w,w),(0,0)), mode="edge"); d = np.zeros_like(feat)
    for n in range(1, w+1):
        d += n*(pad[w+n:w+n+T] - pad[w-n:w-n+T])
    return d/den

def extract_features(wav_path, cmvn=True):
    if not os.path.isfile(wav_path): raise FileNotFoundError(wav_path)
    feat = _mfcc_librosa(wav_path) if _LIBROSA else _mfcc_scipy(wav_path)
    return apply_cmvn(feat) if cmvn else feat

def _mfcc_librosa(wav_path):
    y,_ = librosa.load(wav_path, sr=_SR, mono=True)
    if len(y)==0: raise ValueError(f"Audio kosong: {wav_path}")
    y  = np.append(y[0], y[1:]-_PRE_EMPH*y[:-1])
    m  = librosa.feature.mfcc(y=y, sr=_SR, n_mfcc=_N_MFCC, n_fft=_N_FFT,
                                hop_length=_HOP_LENGTH, win_length=_WIN_LENGTH,
                                n_mels=_N_MELS, fmin=_FMIN, fmax=_FMAX, window="hamming")
    d1 = librosa.feature.delta(m, width=_DELTA_W*2+1)
    d2 = librosa.feature.delta(m, order=2, width=_DELTA_W*2+1)
    return np.vstack([m,d1,d2]).T.astype(np.float32)

def _mfcc_scipy(wav_path):
    sr,audio = _wavfile.read(wav_path)
    if audio.dtype==np.int16:   audio=audio.astype(np.float32)/32768.0
    elif audio.dtype==np.int32: audio=audio.astype(np.float32)/2147483648.0
    else:                       audio=audio.astype(np.float32)
    if audio.ndim>1: audio=audio.mean(1)
    if sr!=_SR:
        n_new=int(len(audio)*_SR/sr)
        audio=np.interp(np.linspace(0,len(audio)-1,n_new),np.arange(len(audio)),audio)
    y=np.append(audio[0], audio[1:]-_PRE_EMPH*audio[:-1])
    nf=max(1,1+(len(y)-_WIN_LENGTH)//_HOP_LENGTH)
    idx=np.arange(_WIN_LENGTH)[None,:]+_HOP_LENGTH*np.arange(nf)[:,None]
    idx=np.clip(idx,0,len(y)-1)
    fr=y[idx].astype(np.float32)*np.hamming(_WIN_LENGTH).astype(np.float32)
    mag=np.abs(np.fft.rfft(fr,n=_N_FFT))
    h2m=lambda f: 1127.0*np.log1p(f/700.0); m2h=lambda m: 700.0*(np.exp(m/1127.0)-1.0)
    mp=np.linspace(h2m(_FMIN),h2m(_FMAX),_N_MELS+2); hp=m2h(mp)
    bins=np.floor((_N_FFT+1)*hp/_SR).astype(int)
    fb=np.zeros((_N_MELS,_N_FFT//2+1),dtype=np.float32)
    for m in range(1,_N_MELS+1):
        for k in range(bins[m-1],bins[m]): fb[m-1,k]=(k-bins[m-1])/max(bins[m]-bins[m-1],1)
        for k in range(bins[m],bins[m+1]): fb[m-1,k]=(bins[m+1]-k)/max(bins[m+1]-bins[m],1)
    loge=np.log10(np.maximum(mag**2@fb.T,1e-10))
    N=_N_MELS; dct=np.cos(np.pi*np.arange(_N_MFCC)[:,None]*(np.arange(N)+0.5)[None,:]/N)
    mfcc=(loge@dct.T)*np.sqrt(2.0/N)
    d1=_delta_numpy(mfcc); d2=_delta_numpy(d1)
    return np.concatenate([mfcc,d1,d2],axis=1).astype(np.float32)

def build_mfcc_cache(sampled_root, dict_transkrip):
    wf  = glob.glob(os.path.join(sampled_root,"**/*.wav"),recursive=True)
    wf += glob.glob(os.path.join(sampled_root,"**/*.flac"),recursive=True)
    cache, meta = {}, []
    print(f"  Ekstrak MFCC dari {len(wf)} file ...")
    t0=time.time()
    for idx,wp in enumerate(wf,1):
        aid=os.path.splitext(os.path.basename(wp))[0]
        if not dict_transkrip.get(aid,""): continue
        try:
            feat=extract_features(wp, cmvn=True)
            cache[aid]=feat; meta.append({"Id":aid,"Frames":feat.shape[0],"Dim":feat.shape[1]})
        except Exception as e:
            print(f"    [SKIP] {os.path.basename(wp)}: {e}")
        if idx%200==0: print(f"    [{idx}/{len(wf)}] {time.time()-t0:.0f}s")
    print(f"  Selesai: {len(cache)} audio")
    return cache, meta

# =============================================================================
#  GMM
# =============================================================================
class GaussianMixtureModel:
    def __init__(self, n_components=4, n_iter=100, tol=1e-3,
                 reg_covar=COV_REG, random_state=None):
        self.n_components=n_components; self.n_iter=n_iter
        self.tol=tol; self.reg_covar=reg_covar; self.random_state=random_state
        self.weights_=self.means_=self.covariances_=None
        self.converged_=False; self.n_iter_done_=0
        self.log_likelihood_history_: List[float]=[]

    def _init_params(self, X):
        rng=np.random.default_rng(self.random_state); K=self.n_components
        idx=rng.choice(X.shape[0],size=K,replace=False)
        self.means_=X[idx].copy(); self.weights_=np.full(K,1.0/K)
        self.covariances_=np.tile(np.var(X,axis=0)+self.reg_covar,(K,1))

    def _log_gaussian_all(self, X):
        mu=self.means_; var=self.covariances_
        log_det=np.sum(np.log(var),axis=1)
        diff=X[:,None,:]-mu[None,:,:]
        maha=np.sum(diff**2/var[None,:,:],axis=2)
        return (-0.5*(X.shape[1]*np.log(2*np.pi)+log_det))[None,:]-0.5*maha

    def _e_step(self, X):
        lw=np.log(self.weights_+LOG_EPS)+self._log_gaussian_all(X)
        lse=np.max(lw,axis=1,keepdims=True)
        ls=lse.reshape(-1)+np.log(np.sum(np.exp(lw-lse),axis=1))
        return lw-ls[:,None], ls

    def _m_step(self, X, lr):
        r=np.exp(lr); Nk=r.sum(0)+LOG_EPS
        self.weights_=Nk/X.shape[0]; self.means_=(r.T@X)/Nk[:,None]
        diff=X[:,None,:]-self.means_[None,:,:]
        self.covariances_=(np.sum(r[:,:,None]*diff**2,axis=0)/Nk[:,None]+self.reg_covar)

    def fit(self, X):
        assert X.ndim==2
        self.n_components=min(self.n_components, max(1,X.shape[0]//4))
        self._init_params(X); self.log_likelihood_history_=[]; prev=-np.inf
        for it in range(self.n_iter):
            lr,ls=self._e_step(X); ll=float(ls.mean()); self.log_likelihood_history_.append(ll)
            if abs(ll-prev)<self.tol: self.converged_=True; self.n_iter_done_=it+1; break
            prev=ll; self._m_step(X,lr)
        else: self.converged_=False; self.n_iter_done_=self.n_iter
        return self

    def log_prob_per_frame(self, X):
        assert X.ndim==2
        lw=np.log(self.weights_+LOG_EPS)+self._log_gaussian_all(X)
        lse=np.max(lw,axis=1,keepdims=True)
        return lse.reshape(-1)+np.log(np.sum(np.exp(lw-lse),axis=1))

# =============================================================================
#  [FIX-4] ALIGNMENT VITERBI — VEKTORISASI NUMPY
# =============================================================================
def alignment_fonem_viterbi(fitur, fonem_seq, gmm_per_fonem):
    """
    [FIX-4] Vektorisasi: eliminasi inner Python loop for j in n.
    Menggunakan np.maximum.accumulate untuk propagasi DP.
    """
    T = fitur.shape[0]; n = len(fonem_seq)
    if n==0 or T==0: return np.full(T, SIL_ID, dtype=np.int32)

    # Hitung emission scores: (T, n)
    skor = np.full((T,n), LOG_ZERO, dtype=np.float64)
    for j,fonem in enumerate(fonem_seq):
        if fonem in gmm_per_fonem:
            skor[:,j] = gmm_per_fonem[fonem].log_prob_per_frame(fitur)
        elif fonem=="SIL":
            skor[:,j] = 0.0

    # DP monotonic Viterbi (vektorisasi per t)
    dp = np.full((T+1,n+1), LOG_ZERO, dtype=np.float64)
    bp = np.zeros((T+1,n+1), dtype=np.int8)
    dp[0,0] = 0.0

    for t in range(1, T+1):
        e = skor[t-1]                        # (n,)
        stay    = dp[t-1,:n] + e             # self-loop: tetap di posisi j
        advance = np.full(n, LOG_ZERO)
        advance[1:] = dp[t-1,:n-1] + e[1:]  # maju dari j-1

        use_adv = advance > stay
        dp[t,:n] = np.where(use_adv, advance, stay)
        bp[t,:n] = np.where(use_adv, 1, 0)
        # Mask posisi yang tidak reachable
        dp[t,:n] = np.where(dp[t,:n]>LOG_ZERO, dp[t,:n], LOG_ZERO)

    # Traceback
    valid = [(dp[T,j],j) for j in range(n) if dp[T,j]>LOG_ZERO]
    j_end = valid[-1][1] if valid else n-1
    path  = np.zeros(T, dtype=int); j=j_end
    for t in range(T, 0, -1):
        path[t-1]=j
        if bp[t,j]==1: j=max(j-1,0)

    return np.array([FONEM2ID.get(fonem_seq[k],SIL_ID) for k in path], dtype=np.int32)


def alignment_triphone_viterbi(fitur, fonem_seq, gmm_tri, gmm_mono):
    triphone_seq = buat_triphone_seq(fonem_seq)
    T=fitur.shape[0]; n=len(triphone_seq)
    if n==0 or T==0: return np.full(T,SIL_ID,dtype=np.int32)
    skor=np.full((T,n),LOG_ZERO,dtype=np.float64)
    for j,tri in enumerate(triphone_seq):
        if tri in gmm_tri:
            skor[:,j]=gmm_tri[tri].log_prob_per_frame(fitur)
        else:
            center=triphone_center(tri)
            if center in gmm_mono: skor[:,j]=gmm_mono[center].log_prob_per_frame(fitur)
            elif center=="SIL":    skor[:,j]=0.0
    # Sama dengan monophone tapi pakai triphone scores
    dp=np.full((T+1,n+1),LOG_ZERO,dtype=np.float64); bp=np.zeros((T+1,n+1),dtype=np.int8)
    dp[0,0]=0.0
    for t in range(1,T+1):
        e=skor[t-1]; stay=dp[t-1,:n]+e; advance=np.full(n,LOG_ZERO)
        advance[1:]=dp[t-1,:n-1]+e[1:]
        use_adv=advance>stay; dp[t,:n]=np.where(use_adv,advance,stay); bp[t,:n]=np.where(use_adv,1,0)
        dp[t,:n]=np.where(dp[t,:n]>LOG_ZERO,dp[t,:n],LOG_ZERO)
    valid=[(dp[T,j],j) for j in range(n) if dp[T,j]>LOG_ZERO]
    j_end=valid[-1][1] if valid else n-1
    path=np.zeros(T,dtype=int); j=j_end
    for t in range(T,0,-1):
        path[t-1]=j
        if bp[t,j]==1: j=max(j-1,0)
    centers=[triphone_center(triphone_seq[k]) for k in path]
    return np.array([FONEM2ID.get(c,SIL_ID) for c in centers],dtype=np.int32)


def kumpulkan_frame_fonem_naif(fitur, fonem_seq):
    T,n=fitur.shape[0],len(fonem_seq)
    if n==0: return {}
    nf=np.ones(n,dtype=int)*(T//n); nf[:T-nf.sum()]+=1
    hasil={}; idx=0
    for i,f in enumerate(fonem_seq):
        seg=fitur[idx:idx+nf[i]]
        if seg.shape[0]>0:
            hasil[f]=np.vstack([hasil[f],seg]) if f in hasil else seg.copy()
        idx+=nf[i]
    return hasil


def kumpulkan_frame_per_fonem(cache_mfcc, dict_transkrip, lexicon,
                               gmm_mono=None, gmm_tri=None, audio_ids=None):
    use_tri  = USE_TRIPHONE and gmm_tri  is not None and len(gmm_tri)>0
    use_mono = gmm_mono is not None and len(gmm_mono)>0
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)
    ids = audio_ids if audio_ids else list(cache_mfcc.keys())

    for audio_id in ids:
        fitur=cache_mfcc.get(audio_id); label=dict_transkrip.get(audio_id,"")
        if fitur is None or not label: continue
        fs=_bangun_fonem_seq(label,lexicon)
        if not fs: continue

        if use_tri:
            lbls=alignment_triphone_viterbi(fitur,fs,gmm_tri,gmm_mono or {})
            for fid in np.unique(lbls):
                buf[FONEM_JAWA[fid]].append(fitur[lbls==fid])
        elif use_mono:
            lbls=alignment_fonem_viterbi(fitur,fs,gmm_mono)
            for fid in np.unique(lbls):
                buf[FONEM_JAWA[fid]].append(fitur[lbls==fid])
        else:
            for fonem,arr in kumpulkan_frame_fonem_naif(fitur,fs).items():
                buf[fonem].append(arr)

    return {f:np.vstack(parts) for f,parts in buf.items()}


def kumpulkan_frame_per_triphone(cache_mfcc, dict_transkrip, lexicon, gmm_mono, audio_ids=None):
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)
    ids = audio_ids if audio_ids else list(cache_mfcc.keys())
    for audio_id in ids:
        fitur=cache_mfcc.get(audio_id); label=dict_transkrip.get(audio_id,"")
        if fitur is None or not label: continue
        fs=_bangun_fonem_seq(label,lexicon)
        if not fs: continue
        tris=buat_triphone_seq(fs); T=fitur.shape[0]
        n_tri=len(tris); seg_len=max(1,T//n_tri) if n_tri>0 else T
        idx=0
        for k,tri in enumerate(tris):
            end=idx+seg_len if k<n_tri-1 else T
            seg=fitur[idx:end]
            if seg.shape[0]>0: buf[tri].append(seg)
            idx=end
    return {tri:np.vstack(parts) for tri,parts in buf.items()
            if np.vstack(parts).shape[0]>=TRIPHONE_MIN_FRAME}


def latih_gmm_per_fonem(frame_per_fonem, n_components=N_COMPONENTS_GMM,
                         n_iter=N_ITER_GMM, tol=TOL_GMM, min_frame=MIN_FRAME_GMM,
                         verbose=True):
    gmm={}; t0=time.time()
    for i,(fonem,X) in enumerate(sorted(frame_per_fonem.items())):
        if X.shape[0]<min_frame:
            if verbose: print(f"    SKIP {fonem}: {X.shape[0]} frame")
            continue
        g=GaussianMixtureModel(n_components=n_components,n_iter=n_iter,tol=tol,
                                reg_covar=COV_REG,random_state=hash(fonem)%100_000)
        g.fit(X); gmm[fonem]=g
        if verbose:
            ll=round(g.log_likelihood_history_[-1],3)
            print(f"    [{i+1:2d}] {fonem:4s}: {X.shape[0]:6d} frame, "
                  f"K={g.n_components}, iter={g.n_iter_done_}, conv={g.converged_}, LL={ll}")
    print(f"  GMM: {len(gmm)} model dalam {time.time()-t0:.1f}s")
    return gmm


def latih_gmm_iteratif(cache_train, dict_transkrip, lexicon, n_iters=GMM_EM_ITERS):
    gmm_mono={}
    print("="*56); print("FASE 1: Iterative monophone GMM")
    for it in range(n_iters):
        print(f"\n  Iter {it+1}/{n_iters} ...")
        fpf=kumpulkan_frame_per_fonem(cache_train,dict_transkrip,lexicon,
                                       gmm_mono=gmm_mono if it>0 else None)
        gmm_mono=latih_gmm_per_fonem(fpf, verbose=(it==n_iters-1))
        print(f"  → {len(gmm_mono)} model monophone")

    gmm_tri={}
    if USE_TRIPHONE:
        print("\n"+"="*56); print("FASE 2: Triphone GMM")
        fpt=kumpulkan_frame_per_triphone(cache_train,dict_transkrip,lexicon,gmm_mono)
        gmm_tri=latih_gmm_per_fonem(fpt,n_components=max(2,N_COMPONENTS_GMM//2),
                                     min_frame=TRIPHONE_MIN_FRAME,verbose=False)
        print(f"  → {len(gmm_tri)} model triphone")
    return gmm_mono, gmm_tri


def hitung_alignment_corpus(audio_ids, cache_mfcc, dict_transkrip, lexicon,
                              gmm_mono, gmm_tri=None):
    use_tri = USE_TRIPHONE and gmm_tri is not None and len(gmm_tri)>0
    ac={}
    for aid in audio_ids:
        fitur=cache_mfcc.get(aid); label=dict_transkrip.get(aid,"")
        if fitur is None or not label: continue
        fs=_bangun_fonem_seq(label,lexicon)
        if not fs: continue
        if use_tri:
            ac[aid]=alignment_triphone_viterbi(fitur,fs,gmm_tri,gmm_mono)
        else:
            ac[aid]=alignment_fonem_viterbi(fitur,fs,gmm_mono)
    return ac

# =============================================================================
#  DATASET — [FIX-6] Terima pre-computed alignment
# =============================================================================
class FrameLevelDataset(Dataset):
    """[FIX-6] Gunakan alignment_cache yang sudah dihitung, tidak ulangi alignment."""
    def __init__(self, audio_ids, cache_mfcc, alignment_cache):
        fp, lp = [], []
        for aid in audio_ids:
            fitur=cache_mfcc.get(aid); aligns=alignment_cache.get(aid)
            if fitur is None or aligns is None: continue
            fp.append(fitur); lp.append(aligns)
        if fp:
            self.frames = torch.from_numpy(np.vstack(fp))
            self.labels = torch.from_numpy(np.concatenate(lp).astype(np.int64))
        else:
            self.frames = torch.zeros((0,FEAT_DIM),dtype=torch.float32)
            self.labels = torch.zeros(0,dtype=torch.long)
        print(f"  Dataset: {len(self.frames):,} frame")

    def __len__(self): return len(self.frames)
    def __getitem__(self, i): return self.frames[i], self.labels[i]

# =============================================================================
#  TDNN
# =============================================================================
class TDNNLayer(nn.Module):
    def __init__(self, input_dim, output_dim, context, dropout=0.0):
        super().__init__()
        self.context=sorted(context)
        self.linear=nn.Linear(input_dim*len(context), output_dim)
        self.bn=nn.BatchNorm1d(output_dim)
        self.drop=nn.Dropout(dropout)

    def forward(self, x):
        sq = x.dim()==2
        if sq: x=x.unsqueeze(1)
        B,T,_=x.shape
        cmin=min(self.context); cmax=max(self.context)
        xp=F.pad(x.permute(0,2,1),(max(0,-cmin),max(0,cmax))).permute(0,2,1)
        ctx=[xp[:,c-cmin:c-cmin+T,:] for c in self.context]
        sp=torch.cat(ctx,dim=-1)
        out=self.linear(sp); out=self.bn(out.permute(0,2,1)).permute(0,2,1)
        out=F.relu(out); out=self.drop(out)
        return out.squeeze(1) if sq else out


class TDNNAcousticModel(nn.Module):
    def __init__(self, input_dim=FEAT_DIM, n_classes=N_FONEM,
                 hidden_dim=TDNN_HIDDEN, contexts=None, dropout=TDNN_DROPOUT):
        super().__init__()
        if contexts is None: contexts=TDNN_CONTEXTS
        layers=[]; in_dim=input_dim
        for ctx in contexts:
            layers.append(TDNNLayer(in_dim,hidden_dim,ctx,dropout)); in_dim=hidden_dim
        self.tdnn_layers=nn.ModuleList(layers)
        self.classifier=nn.Linear(hidden_dim,n_classes)

    def forward(self, x):
        for layer in self.tdnn_layers: x=layer(x)
        return F.log_softmax(self.classifier(x),dim=-1)

    def get_posteriors_sequence(self, x):
        with torch.no_grad(): return self.forward(x)[0]

# =============================================================================
#  TRAINING — [FIX-1, FIX-3]
# =============================================================================
class LabelSmoothingNLLLoss(nn.Module):
    """[FIX-1] Label smoothing untuk mencegah model terlalu yakin."""
    def __init__(self, n_classes, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing=smoothing; self.n_classes=n_classes; self.weight=weight

    def forward(self, log_probs, targets):
        # Smooth targets: (1-eps)*one_hot + eps/K
        with torch.no_grad():
            smooth_val = self.smoothing / self.n_classes
            smooth_dist = torch.full_like(log_probs, smooth_val)
            smooth_dist.scatter_(1, targets.unsqueeze(1), 1.0-self.smoothing+smooth_val)
        loss = -(smooth_dist * log_probs)
        if self.weight is not None:
            loss = loss * self.weight.unsqueeze(0)
        return loss.sum(dim=1).mean()


def latih_tdnn_frame_level(
    train_ids, val_ids, cache_mfcc, alignment_train, alignment_val,
    dir_save, epochs=TDNN_EPOCHS, lr=TDNN_LR, batch_size=TDNN_BATCH_SIZE,
    hidden_dim=TDNN_HIDDEN, contexts=None, dropout=TDNN_DROPOUT,
    warmup_epochs=TDNN_WARMUP_EPOCH,
):
    if contexts is None: contexts=TDNN_CONTEXTS
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path=os.path.join(dir_save,"model_tdnn_fonem.pt")
    os.makedirs(dir_save, exist_ok=True)

    model=TDNNAcousticModel(input_dim=FEAT_DIM, n_classes=N_FONEM,
                             hidden_dim=hidden_dim, contexts=contexts, dropout=dropout).to(device)

    if os.path.isfile(model_path):
        try:
            model.load_state_dict(torch.load(model_path,map_location=device))
            print(f"  Checkpoint dimuat: {model_path}"); return model
        except Exception:
            os.remove(model_path)

    print("  Membangun dataset ...")
    train_ds=FrameLevelDataset(train_ids, cache_mfcc, alignment_train)
    val_ds  =FrameLevelDataset(val_ids,   cache_mfcc, alignment_val)
    if len(train_ds)==0:
        print("  [ERROR] Train dataset kosong!"); return model

    train_loader=DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=False)
    val_loader  =DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)

    # [FIX-1] sqrt smoothing untuk class weights (lebih konservatif dari 1/count)
    label_counts=torch.bincount(train_ds.labels, minlength=N_FONEM).float()
    label_counts=torch.clamp(label_counts, min=1.0)
    weights=(1.0/torch.sqrt(label_counts)); weights=(weights/weights.sum()).to(device)

    # [FIX-3] AdamW dengan weight decay
    optimizer=torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=TDNN_WEIGHT_DECAY)

    # [FIX-3] CosineAnnealingLR dengan eta_min > 0 agar tidak pernah nol
    # Linear warmup dulu baru cosine decay
    def lr_lambda(ep):
        if ep < warmup_epochs:
            return float(ep+1)/float(warmup_epochs)
        return 1.0  # setelah warmup, CosineAnnealingLR yang atur
    warmup_sched  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    cosine_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs-warmup_epochs,1), eta_min=TDNN_LR_MIN)

    # [FIX-1] Label smoothing loss
    loss_fn=LabelSmoothingNLLLoss(N_FONEM, smoothing=TDNN_LABEL_SMOOTH, weight=weights)

    riwayat=[]; best_val=float("inf"); t0=time.time()
    print(f"\n  Device: {device} | Train: {len(train_ds):,} | Val: {len(val_ds):,}")
    print(f"  [FIX-1] dropout={dropout}, weight_decay={TDNN_WEIGHT_DECAY}, label_smooth={TDNN_LABEL_SMOOTH}")
    print(f"  [FIX-3] LR: warmup {warmup_epochs} ep → cosine decay ke {TDNN_LR_MIN}")

    for epoch in range(1, epochs+1):
        # Training
        model.train()
        tl=nc=nt=0
        for feats,lbls in train_loader:
            feats=feats.to(device); lbls=lbls.to(device)
            optimizer.zero_grad()
            lp=model(feats); loss=loss_fn(lp,lbls)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5.0)
            optimizer.step()
            tl+=loss.item()*len(lbls); nc+=(lp.argmax(1)==lbls).sum().item(); nt+=len(lbls)

        # Validasi
        model.eval(); vl=vc=vt=0
        with torch.no_grad():
            for feats,lbls in val_loader:
                feats=feats.to(device); lbls=lbls.to(device)
                lp=model(feats); vl+=loss_fn(lp,lbls).item()*len(lbls)
                vc+=(lp.argmax(1)==lbls).sum().item(); vt+=len(lbls)

        # LR schedule
        if epoch <= warmup_epochs: warmup_sched.step()
        else: cosine_sched.step()

        atr=tl/max(nt,1); avl=vl/max(vt,1)
        actr=100*nc/max(nt,1); acvl=100*vc/max(vt,1)
        cur_lr=optimizer.param_groups[0]['lr']
        riwayat.append({"Epoch":epoch,"LR":round(cur_lr,7),"Train Loss":round(atr,5),
                         "Val Loss":round(avl,5),"Train Acc (%)":round(actr,2),
                         "Val Acc (%)":round(acvl,2)})
        if epoch%5==0 or epoch==1:
            print(f"  Epoch {epoch:3d}/{epochs} | Loss {atr:.4f}/{avl:.4f} | "
                  f"Acc {actr:.1f}/{acvl:.1f}% | LR {cur_lr:.2e}")

        if avl < best_val:
            best_val=avl; torch.save(model.state_dict(),model_path)

    if os.path.isfile(model_path):
        model.load_state_dict(torch.load(model_path,map_location=device))
        print(f"  Best model dimuat (val_loss={best_val:.5f})")

    pd.DataFrame(riwayat).to_csv(os.path.join(dir_save,"HasilTrainingTDNN.csv"),index=False)
    return model

# =============================================================================
#  [FIX-5] DECODER — DISEDERHANAKAN (bukan O(F²×S×T))
# =============================================================================
class TDNNHMMDecoder:
    """
    [FIX-5] Decoder yang jauh lebih efisien:
    1. TDNN posterior → argmax per frame (O(F×T))
    2. CTC collapse dengan min-duration (hapus repetisi pendek)
    3. Fonem sequence → kata via lexicon DP (bounded)
    4. N-gram rescore
    Tidak ada loop F² seperti di v2 — efisien untuk 29 fonem.
    """
    def __init__(self, model, lexicon, ngram_model, gmm_per_fonem,
                 device=None, n_hmm_states=N_HMM_STATES, beam_width=BEAM_WIDTH):
        self.model=model; self.lexicon=lexicon; self.ngram=ngram_model
        self.gmm_per_fonem=gmm_per_fonem
        self.device=device or torch.device("cpu")
        self.n_hmm_states=n_hmm_states; self.beam_width=beam_width
        self.rev_lex: Dict[Tuple,List[str]] = defaultdict(list)
        for kata,fonems in lexicon.items():
            self.rev_lex[tuple(fonems)].append(kata)
        lengths=[len(v) for v in lexicon.values()]
        self.avg_fpw=float(np.mean(lengths)) if lengths else 3.0
        self.model.to(self.device); self.model.eval()

    def _get_posteriors(self, fitur):
        t=torch.from_numpy(fitur).unsqueeze(0).to(self.device)
        return self.model.get_posteriors_sequence(t).cpu().numpy()

    def _ctc_collapse(self, fonem_ids):
        """
        CTC collapse: hapus repetisi berurutan, filter durasi pendek,
        filter SIL. n_hmm_states digunakan sebagai min-duration.
        """
        result = []
        i = 0
        while i < len(fonem_ids):
            fid = fonem_ids[i]; j=i+1
            while j < len(fonem_ids) and fonem_ids[j]==fid: j+=1
            dur = j-i
            # [FIX-5] Gunakan n_hmm_states sebagai min-duration per fonem
            if dur >= self.n_hmm_states and fid != SIL_ID:
                result.append(FONEM_JAWA[fid])
            i = j
        return result

    def _fonem_ke_kata(self, fonem_seq, T):
        n=len(fonem_seq)
        if n==0: return [[]]
        max_words=max(1, T//AVG_FRAMES_PER_WORD)
        dp=[[] for _ in range(n+1)]; dp[0]=[(0.0,[])]
        for i in range(n):
            if not dp[i]: continue
            for j in range(i+1, min(n+1,i+MAX_FONEM_PER_KATA+1)):
                sub=tuple(fonem_seq[i:j])
                if sub not in self.rev_lex: continue
                for kata in self.rev_lex[sub]:
                    for score,words in dp[i]:
                        if len(words)>=max_words: continue
                        new_words=words+[kata]
                        new_score=score+self.ngram.log_prob(kata,tuple(words[-(self.ngram.n-1):]))
                        dp[j].append((new_score,new_words))
                if len(dp[j])>self.beam_width*3:
                    dp[j].sort(key=lambda x:x[0],reverse=True); dp[j]=dp[j][:self.beam_width]
        if dp[n]:
            dp[n].sort(key=lambda x:x[0],reverse=True)
            return [w for _,w in dp[n][:self.beam_width]]
        return [fonem_seq]

    def dekode(self, fitur):
        if fitur.shape[0]==0: return []
        log_post=self._get_posteriors(fitur)         # (T, N_FONEM)
        fonem_ids=np.argmax(log_post, axis=1)        # (T,) argmax
        fonem_seq=self._ctc_collapse(fonem_ids)      # CTC collapse
        if not fonem_seq: return []
        kata_cand=self._fonem_ke_kata(fonem_seq, fitur.shape[0])
        if not kata_cand: return fonem_seq
        return self.ngram.rescore(kata_cand)

# =============================================================================
#  N-GRAM LM
# =============================================================================
class NGramModel:
    def __init__(self, n=3, discount=0.75):
        self.n=n; self.discount=discount
        self.ngram_count=defaultdict(int); self.context_count=defaultdict(int)
        self.unigram_count=defaultdict(int); self.vocab=set(); self.N_total=0; self._ctx_div={}

    def _tok(self, s):
        t=normalisasi(s).split(); return ["<s>"]*(self.n-1)+t+["</s>"]

    def fit(self, korpus):
        cw=defaultdict(set)
        for s in korpus:
            t=self._tok(s); self.vocab.update(t); self.N_total+=len(t)
            for tok in t: self.unigram_count[tok]+=1
            for i in range(len(t)-self.n+1):
                ng=tuple(t[i:i+self.n]); c=ng[:-1]
                self.ngram_count[ng]+=1; self.context_count[c]+=1; cw[c].add(ng[-1])
        self._ctx_div={c:len(w) for c,w in cw.items()}; return self

    def log_prob(self, kata, ctx):
        ng=ctx+(kata,); Cng=self.ngram_count.get(ng,0); Cc=self.context_count.get(ctx,0)
        Cu=self.unigram_count.get(kata,0); puni=(Cu+1)/(self.N_total+len(self.vocab))
        if Cc==0: return math.log(puni+LOG_EPS)
        lam=self.discount*self._ctx_div.get(ctx,0)/Cc
        return math.log(max(Cng-self.discount,0)/Cc+lam*puni+LOG_EPS)

    def score_sequence(self, words):
        t=["<s>"]*(self.n-1)+words+["</s>"]; lp=0.0
        for i in range(self.n-1,len(t)):
            lp+=self.log_prob(t[i],tuple(t[i-self.n+1:i]))
        return lp

    def rescore(self, kandidat):
        return max(kandidat,key=self.score_sequence) if kandidat else []

# =============================================================================
#  WER & EVALUASI
# =============================================================================
def compute_wer(ref, hyp):
    R,H=len(ref),len(hyp); dp=[[0]*(H+1) for _ in range(R+1)]
    ops=[[""]*(H+1) for _ in range(R+1)]
    for i in range(R+1): dp[i][0]=i; ops[i][0]="D"*i
    for j in range(H+1): dp[0][j]=j; ops[0][j]="I"*j
    for i in range(1,R+1):
        for j in range(1,H+1):
            if ref[i-1]==hyp[j-1]: dp[i][j]=dp[i-1][j-1]; ops[i][j]="M"
            else:
                sub=dp[i-1][j-1]+1; dlt=dp[i-1][j]+1; ins=dp[i][j-1]+1
                best=min(sub,dlt,ins); dp[i][j]=best
                ops[i][j]="S" if best==sub else ("D" if best==dlt else "I")
    S=D=I=0; i,j=R,H
    while i>0 or j>0:
        op=ops[i][j]
        if   op=="M": i-=1;j-=1
        elif op=="S": S+=1;i-=1;j-=1
        elif op=="D": D+=1;i-=1
        elif op=="I": I+=1;j-=1
        else:
            if i>0: D+=1;i-=1
            else:   I+=1;j-=1
    return {"wer_persen":round((S+D+I)/R*100,2) if R>0 else 0.0,"S":S,"D":D,"I":I,"N":R}


def dekode_semua_audio(cache_mfcc, audio_ids, dict_transkrip, decoder, verbose=True):
    hasil=[]; t0=time.time()
    for idx,aid in enumerate(audio_ids,1):
        fitur=cache_mfcc.get(aid); label=dict_transkrip.get(aid,"")
        if fitur is None or not label: continue
        ref=normalisasi(label).split()
        if not ref: continue
        try: hyp=decoder.dekode(fitur)
        except Exception as e:
            if verbose: print(f"    [ERROR] {aid}: {e}"); continue
        wr=compute_wer(ref,hyp)
        hasil.append({"Id":aid,"Referensi":" ".join(ref),"Hipotesis":" ".join(hyp),
                       "WER (%)":wr["wer_persen"],"S":wr["S"],"D":wr["D"],"I":wr["I"],"N":wr["N"]})
        if verbose and idx%100==0: print(f"    [{idx}/{len(audio_ids)}] {time.time()-t0:.1f}s")
    return hasil


def ringkasan_wer(df, label=""):
    if df.empty: return {}
    S=int(df.S.sum()); D=int(df.D.sum()); I=int(df.I.sum()); N=int(df.N.sum())
    wc=(S+D+I)/N*100 if N>0 else 100.0
    print(f"\n  === WER: {label} ===")
    print(f"  Audio: {len(df)} | WER corpus: {wc:.2f}% | Mean: {df['WER (%)'].mean():.2f}%")
    print(f"  S={S} D={D} I={I} N={N}")
    return {"wer_corpus":round(wc,2),"wer_mean":round(df['WER (%)'].mean(),2),
            "wer_median":round(df['WER (%)'].median(),2),"wer_std":round(df['WER (%)'].std(),2),
            "total_S":S,"total_D":D,"total_I":I,"total_N":N}

# =============================================================================
#  MAIN
# =============================================================================
def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    random.seed(42); np.random.seed(42)
    t_start=time.time()

    # [1] Load data
    print("\n[1/8] Membaca CSV dan cache MFCC ...")
    dict_transkrip, dict_speaker = baca_transkrip_csv(CSV_PATH)
    cache_mfcc_raw, meta = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    pd.DataFrame(meta).to_csv(f"{DIR_SAVE}/HasilMFCC.csv",index=False)

    # [2] Speaker CMVN — [FIX-2] skip jika tidak ada speaker
    print("\n[2/8] Speaker CMVN ...")
    cache_mfcc = apply_cmvn_speaker(cache_mfcc_raw, dict_speaker)

    # [3] Split
    print("\n[3/8] Split train/val ...")
    all_ids=list(cache_mfcc.keys())
    if SPEAKER_SPLIT and dict_speaker:
        train_ids, val_ids = split_by_speaker(all_ids, dict_speaker, VAL_RATIO)
    else:
        random.shuffle(all_ids); n_val=max(1,int(len(all_ids)*VAL_RATIO))
        train_ids, val_ids = all_ids[n_val:], all_ids[:n_val]
        print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")

    # [4] Lexicon + LM
    print("\n[4/8] Lexicon dan N-gram LM ...")
    semua_transkrip=list(dict_transkrip.values())
    lexicon=buat_lexicon(semua_transkrip)
    pd.DataFrame([{"Kata":k,"Fonem":" ".join(v)} for k,v in lexicon.items()]
                 ).to_csv(f"{DIR_SAVE}/HasilLexicon.csv",index=False)
    lm=NGramModel(n=N_GRAM); lm.fit([dict_transkrip[a] for a in train_ids if a in dict_transkrip])
    print(f"  Lexicon: {len(lexicon)} kata | LM vocab: {len(lm.vocab)}")

    # [5] GMM iteratif
    print("\n[5/8] Iterative GMM ...")
    cache_train={k:cache_mfcc[k] for k in train_ids if k in cache_mfcc}
    gmm_mono, gmm_tri = latih_gmm_iteratif(cache_train, dict_transkrip, lexicon)

    gmm_stat=[{"Fonem":f,"Frames":frame_per_fonem.get(f,np.empty((0,39))).shape[0],
                "K":g.n_components,"Iter":g.n_iter_done_,"Conv":g.converged_,
                "LL":round(g.log_likelihood_history_[-1],4) if g.log_likelihood_history_ else None}
               for f,g in gmm_mono.items()]
    pd.DataFrame(gmm_stat).to_csv(f"{DIR_SAVE}/HasilGMM_Fonem.csv",index=False)

    # [6] Alignment corpus
    print("\n[6/8] Alignment corpus ...")
    align_train=hitung_alignment_corpus(train_ids,cache_mfcc,dict_transkrip,lexicon,gmm_mono,gmm_tri)
    align_val  =hitung_alignment_corpus(val_ids,  cache_mfcc,dict_transkrip,lexicon,gmm_mono,gmm_tri)
    print(f"  Train: {len(align_train)} | Val: {len(align_val)}")

    # [7] TDNN — [FIX-1,3,6]
    print("\n[7/8] Training TDNN ...")
    model=latih_tdnn_frame_level(train_ids,val_ids,cache_mfcc,align_train,align_val,DIR_SAVE)

    # [8] Decode + evaluasi
    print("\n[8/8] Dekode dan evaluasi ...")
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    decoder=TDNNHMMDecoder(model=model,lexicon=lexicon,ngram_model=lm,
                            gmm_per_fonem=gmm_mono,device=device,
                            n_hmm_states=N_HMM_STATES,beam_width=BEAM_WIDTH)

    print("  Dekode train ...")
    hasil_train=dekode_semua_audio(cache_mfcc,train_ids,dict_transkrip,decoder)
    print("  Dekode val ...")
    hasil_val  =dekode_semua_audio(cache_mfcc,val_ids,  dict_transkrip,decoder)

    df_tr=pd.DataFrame(hasil_train); df_vl=pd.DataFrame(hasil_val)
    df_tr.to_csv(f"{DIR_SAVE}/HasilWER_Train.csv",index=False)
    df_vl.to_csv(f"{DIR_SAVE}/HasilWER_Val.csv",  index=False)

    r_tr=ringkasan_wer(df_tr,"Train"); r_vl=ringkasan_wer(df_vl,"Val")
    pd.DataFrame([{"set":"train",**r_tr},{"set":"val",**r_vl}]
                 ).to_csv(f"{DIR_SAVE}/RingkasanWER.csv",index=False)

    print("\n"+"="*56)
    print(f"  HASIL AKHIR")
    print(f"  Train WER: {r_tr.get('wer_corpus','N/A')}%")
    print(f"  Val   WER: {r_vl.get('wer_corpus','N/A')}%  (mean: {r_vl.get('wer_mean','N/A')}%)")
    print(f"  Total: {(time.time()-t_start)/60:.1f} menit")
    print("="*56)


if __name__=="__main__":
    # Simpan frame_per_fonem untuk statistik GMM (gunakan global sementara)
    frame_per_fonem = {}
    main()


[1/8] Membaca CSV dan cache MFCC ...
  CSV: 1749 audio | 0 speaker unik
  Ekstrak MFCC dari 1749 file ...
    [200/1749] 2s
    [400/1749] 3s
    [600/1749] 5s
    [800/1749] 6s
    [1000/1749] 7s
    [1200/1749] 9s
    [1400/1749] 10s
    [1600/1749] 11s
  Selesai: 1749 audio

[2/8] Speaker CMVN ...
  [FIX-2] Tidak ada kolom speaker → skip speaker CMVN

[3/8] Split train/val ...
  Train: 1400 | Val: 349

[4/8] Lexicon dan N-gram LM ...
  Lexicon: 5006 kata | LM vocab: 4274

[5/8] Iterative GMM ...
FASE 1: Iterative monophone GMM

  Iter 1/3 ...
  GMM: 26 model dalam 403.2s
  → 26 model monophone

  Iter 2/3 ...
  GMM: 27 model dalam 299.7s
  → 27 model monophone

  Iter 3/3 ...
    [ 1] SIL :    435 frame, K=8, iter=42, conv=True, LL=-47.622
    [ 2] a   :  89165 frame, K=8, iter=54, conv=True, LL=-51.212
    [ 3] b   :  18746 frame, K=8, iter=66, conv=True, LL=-46.506
    [ 4] c   :   9890 frame, K=8, iter=52, conv=True, LL=-45.956
    [ 5] d   :  26894 frame, K=8, iter=33, conv=Tru

KeyboardInterrupt: 

In [7]:
# =============================================================================
#  HYBRID TDNN-HMM ASR — Bahasa Jawa — v4 (VTLN + SAT/fMLLR)
#
#  PERBAIKAN DARI v3:
#
#  [FIX-v3-1] TDNN overfit → dropout 0.3, weight decay 1e-3, label smoothing
#  [FIX-v3-2] Speaker CMVN gagal → skip jika 0 speaker terdeteksi
#  [FIX-v3-3] LR cosine ke nol → CosineAnnealingLR + eta_min=1e-6
#  [FIX-v3-4] Alignment Viterbi lambat → vektorisasi NumPy
#  [FIX-v3-5] HMM decoder O(F²×S×T) → argmax + CTC collapse
#  [FIX-v3-6] FrameLevelDataset buat ulang alignment → pre-computed cache
#
#  PENAMBAHAN BARU v4:
#
#  [NEW-1] VTLN (Vocal Tract Length Normalization):
#          Warp mel filterbank per speaker menggunakan faktor α ∈ [0.80,1.20].
#          Grid search α optimal via log-likelihood GMM per speaker.
#          Alpha disimpan di speaker_vtln_cache → skip re-search saat decode.
#          Implementasi: warp frekuensi mel sebelum ekstraksi MFCC,
#          sesuai persamaan Wegmann & Petrone (1996).
#
#  [NEW-2] SAT (Speaker Adaptive Training) via fMLLR / CMLLR:
#          Estimasi transform affine A, b per speaker dari statistik GMM.
#          fMLLR: x̂ = A·x + b, di mana A ∈ ℝ^(D×D), b ∈ ℝ^D.
#          Optimasi auxiliary function Q(A,b) closed-form (Gales 1998).
#          TDNN dilatih pada fitur fMLLR-transformed → speaker-invariant.
#          Test-time: estimasi fMLLR dari utterance baru sebelum decode.
#
#  [NEW-3] fMLLR iteratif dengan GMM:
#          Iterasi: GMM (raw) → align → fMLLR → GMM (fMLLR) → align → TDNN
#          Setiap iterasi memperbarui transform dan memperbaiki alignment.
#
#  [BUG-FIX-1] frame_per_fonem global → latih_gmm_iteratif() sekarang
#              me-return (gmm_mono, gmm_tri, frame_per_fonem) untuk menghindari
#              KeyError di blok statistik GMM pada main().
#
#  [BUG-FIX-2] Warping VTLN pada scipy fallback: warp diterapkan pada
#              filterbank bins, bukan pada frekuensi post-FFT, konsisten
#              dengan implementasi librosa.
# =============================================================================

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\hybrid\src\TDNN-HMM\out_v4"
)

COL_SPEAKER = "speaker_id"

# ── GMM ───────────────────────────────────────────────────────────────────────
N_COMPONENTS_GMM   = 8
N_ITER_GMM         = 100
TOL_GMM            = 1e-4
MIN_FRAME_GMM      = 20
GMM_EM_ITERS       = 3

# ── TDNN ──────────────────────────────────────────────────────────────────────
TDNN_HIDDEN        = 256
TDNN_DROPOUT       = 0.3
TDNN_WEIGHT_DECAY  = 1e-3
TDNN_LABEL_SMOOTH  = 0.1
TDNN_EPOCHS        = 60
TDNN_LR            = 5e-4
TDNN_LR_MIN        = 1e-6
TDNN_BATCH_SIZE    = 512
TDNN_WARMUP_EPOCH  = 5
TDNN_CONTEXTS      = [
    [-2, -1, 0, 1, 2],
    [-1,  0, 1],
    [-1,  0, 1],
    [-3,  0, 3],
    [-3,  0, 3],
    [-6,  0, 6],
]

# ── HMM / Decode ─────────────────────────────────────────────────────────────
N_HMM_STATES        = 4
BEAM_WIDTH          = 8
AVG_FRAMES_PER_WORD = 30
MIN_FONEM_DUR       = 2
MAX_FONEM_PER_KATA  = 8

# ── LM ───────────────────────────────────────────────────────────────────────
N_GRAM         = 3
VAL_RATIO      = 0.2
SPEAKER_SPLIT  = True

# ── Triphone ─────────────────────────────────────────────────────────────────
USE_TRIPHONE       = True
TRIPHONE_MIN_FRAME = 30

# ── VTLN ─────────────────────────────────────────────────────────────────────
# [NEW-1] Vocal Tract Length Normalization
USE_VTLN           = True
VTLN_ALPHA_RANGE   = [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20]
# Batas frekuensi warp (Hz). Di bawah VTLN_FBREAK, warp linear.
# Di atas batas atas, tidak di-warp (capping agar tidak keluar Nyquist).
VTLN_FBREAK        = 4000.0   # Hz — titik patah warp piecewise linear
VTLN_FHIGH         = 7200.0   # Hz — batas atas sebelum capping (sesuaikan dengan SR/2)

# ── SAT / fMLLR ──────────────────────────────────────────────────────────────
# [NEW-2] Speaker Adaptive Training via feature-space MLLR
USE_SAT            = True
FMLLR_ITERS        = 3        # iterasi estimasi A,b per speaker
FMLLR_MIN_FRAME    = 50       # minimal frame untuk estimasi transform
SAT_ITERS          = 2        # siklus GMM-align-fMLLR sebelum latih TDNN

LOG_ZERO = -1e15
LOG_EPS  = 1e-300
COV_REG  = 1e-6
FEAT_EPS = 1e-8

# =============================================================================
#  IMPORT
# =============================================================================
import os, re, glob, math, time, random, warnings
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    _TORCH = True
except ImportError:
    raise ImportError("PyTorch diperlukan: pip install torch")

try:
    import librosa; _LIBROSA = True
except ImportError:
    _LIBROSA = False

try:
    from scipy.io import wavfile as _wavfile; _SCIPY = True
except ImportError:
    _SCIPY = False

warnings.filterwarnings("ignore")

# =============================================================================
#  FONEM
# =============================================================================
FONEM_JAWA: List[str] = [
    "a","i","u","e","o","eu","ae",
    "p","b","t","d","k","g","dh","th",
    "m","n","ng","ny",
    "s","h","f",
    "j","c",
    "l","r",
    "w","y",
    "SIL",
]
FONEM2ID: Dict[str, int] = {f: i for i, f in enumerate(FONEM_JAWA)}
N_FONEM: int = len(FONEM_JAWA)
SIL_ID: int  = FONEM2ID["SIL"]

_G2P_DIGRAF = {"dh":"dh","th":"th","ng":"ng","ny":"ny"}
_kf_cache: Dict[str, Tuple[str,...]] = {}
_norm_cache: Dict[str, str] = {}

def kata_ke_fonem(kata: str) -> Tuple[str,...]:
    if kata in _kf_cache: return _kf_cache[kata]
    fonem, i = [], 0
    while i < len(kata):
        if i+1 < len(kata) and kata[i:i+2] in _G2P_DIGRAF:
            fonem.append(_G2P_DIGRAF[kata[i:i+2]]); i += 2
        elif kata[i] in FONEM2ID:
            fonem.append(kata[i]); i += 1
        else:
            i += 1
    result = tuple(fonem) if fonem else ("SIL",)
    _kf_cache[kata] = result
    return result

def normalisasi(teks: str) -> str:
    if teks in _norm_cache: return _norm_cache[teks]
    t = re.sub(r"\s+"," ", re.sub(r"[^a-z\s]","", teks.lower().strip())).strip()
    _norm_cache[teks] = t
    return t

def buat_lexicon(transkrip_list):
    lex = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lex:
                lex[kata] = list(kata_ke_fonem(kata))
    return lex

def _bangun_fonem_seq(label, lexicon):
    fs = []
    for kata in normalisasi(label).split():
        fs.extend(lexicon.get(kata, list(kata_ke_fonem(kata))))
    return fs

# =============================================================================
#  TRIPHONE
# =============================================================================
def buat_triphone_seq(fonem_seq):
    if not fonem_seq: return []
    p = ["SIL"] + list(fonem_seq) + ["SIL"]
    return [f"{p[i-1]}-{p[i]}+{p[i+1]}" for i in range(1, len(p)-1)]

def triphone_center(tri):
    try: return tri.split("-")[1].split("+")[0]
    except: return tri

# =============================================================================
#  CSV & SPLIT
# =============================================================================
def baca_transkrip_csv(csv_path, col_speaker=COL_SPEAKER):
    if not os.path.isfile(csv_path): raise FileNotFoundError(csv_path)
    df = None
    for enc in ("utf-8","utf-8-sig","latin-1","cp1252"):
        try: df = pd.read_csv(csv_path, encoding=enc); break
        except: continue
    if df is None: raise ValueError(f"Gagal baca CSV: {csv_path}")
    df.columns = [c.strip() for c in df.columns]
    col_path = next((c for c in ["Path_Data","path_data","path","audio_path","filename"] if c in df.columns), None)
    col_text = next((c for c in ["sentences","sentence","text","transcription","label"] if c in df.columns), None)
    if not col_path: raise ValueError(f"Kolom path tidak ada. Kolom: {list(df.columns)}")
    if not col_text: raise ValueError(f"Kolom teks tidak ada. Kolom: {list(df.columns)}")
    dt, ds = {}, {}
    has_spk = bool(col_speaker and col_speaker in df.columns)
    for _, row in df.iterrows():
        pv = str(row[col_path]).strip(); tv = str(row[col_text]).strip()
        if pv in ("nan","") or tv in ("nan",""): continue
        aid = os.path.splitext(os.path.basename(pv))[0]
        dt[aid] = tv.lower()
        if has_spk: ds[aid] = str(row[col_speaker]).strip()
    n_spk = len(set(ds.values())) if ds else 0
    print(f"  CSV: {len(dt)} audio | {n_spk} speaker unik")
    return dt, ds

def split_by_speaker(audio_ids, dict_speaker, val_ratio=VAL_RATIO):
    ids = list(audio_ids)
    if not dict_speaker:
        random.shuffle(ids); n_val = max(1, int(len(ids)*val_ratio))
        return ids[n_val:], ids[:n_val]
    spk2ids = defaultdict(list)
    for aid in ids: spk2ids[dict_speaker.get(aid,"unknown")].append(aid)
    speakers = list(spk2ids.keys()); random.shuffle(speakers)
    n_val_spk = max(1, int(len(speakers)*val_ratio))
    val_spk = set(speakers[:n_val_spk])
    train_ids, val_ids = [], []
    for spk, sids in spk2ids.items():
        (val_ids if spk in val_spk else train_ids).extend(sids)
    print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")
    return train_ids, val_ids

# =============================================================================
#  MFCC & VTLN
# =============================================================================
_SR = 16000; _N_MFCC = 13; _N_FFT = 512; _HOP_LENGTH = 160
_WIN_LENGTH = 400; _N_MELS = 26; _PRE_EMPH = 0.97
_FMIN = 0.0; _FMAX = 4000.0; _DELTA_W = 2
FEAT_DIM = _N_MFCC * 3


def vtln_warp_freq(freq_hz: float, alpha: float,
                   f_low: float = _FMIN,
                   f_high: float = VTLN_FHIGH,
                   f_break: float = VTLN_FBREAK) -> float:
    """
    [NEW-1] VTLN piecewise-linear warping function (Wegmann & Petrone 1996).

    Untuk α > 1: spektrum di-compress → speaker dengan vocal tract pendek.
    Untuk α < 1: spektrum di-expand  → speaker dengan vocal tract panjang.

    Warp:
        f* = α·f                          jika f <= f_break / α
        f* = f_break + (f - f_break/α) ·
             (f_high - f_break) / (f_high - f_break/α)  otherwise

    Di luar [f_low, f_high]: linear passthrough.
    """
    if alpha == 1.0:
        return freq_hz
    f_break_warped = f_break / alpha
    if freq_hz <= f_break_warped:
        return alpha * freq_hz
    else:
        scale = (f_high - f_break) / max(f_high - f_break_warped, 1e-8)
        return f_break + (freq_hz - f_break_warped) * scale


def build_vtln_filterbank(n_mels: int, n_fft: int, sr: int, alpha: float,
                           f_low: float = _FMIN, f_high: float = _FMAX) -> np.ndarray:
    """
    [NEW-1] Bangun mel filterbank dengan VTLN warping.
    Frekuensi mel center di-warp oleh vtln_warp_freq sebelum
    dikonversi kembali ke Hz untuk menentukan bins FFT.

    Returns:
        fb: (n_mels, n_fft//2+1) filterbank matrix
    """
    hz2mel = lambda f: 1127.0 * np.log1p(f / 700.0)
    mel2hz = lambda m: 700.0 * (np.exp(m / 1127.0) - 1.0)

    # Titik mel dari f_low ke f_high (tanpa warp dulu, sebagai referensi)
    mel_pts = np.linspace(hz2mel(f_low), hz2mel(f_high), n_mels + 2)
    hz_pts  = mel2hz(mel_pts)

    # [NEW-1] Warp frekuensi center Hz menggunakan VTLN
    hz_warped = np.array([
        vtln_warp_freq(f, alpha, f_low, VTLN_FHIGH, VTLN_FBREAK)
        for f in hz_pts
    ])
    # Clamp agar tidak keluar Nyquist
    hz_warped = np.clip(hz_warped, f_low, sr / 2)

    # Konversi ke bins FFT
    bins = np.floor((n_fft + 1) * hz_warped / sr).astype(int)
    bins = np.clip(bins, 0, n_fft // 2)

    fb = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)
    for m in range(1, n_mels + 1):
        lo, mid, hi = bins[m-1], bins[m], bins[m+1]
        for k in range(lo, mid):
            fb[m-1, k] = (k - lo) / max(mid - lo, 1)
        for k in range(mid, hi):
            fb[m-1, k] = (hi - k) / max(hi - mid, 1)
    return fb


def apply_cmvn(f: np.ndarray) -> np.ndarray:
    return (f - f.mean(0, keepdims=True)) / (f.std(0, keepdims=True) + FEAT_EPS)


def apply_cmvn_speaker(cache_mfcc: dict, dict_speaker: dict) -> dict:
    """
    [FIX-v3-2] Skip speaker CMVN jika tidak ada speaker atau hanya 1 speaker.
    """
    if not dict_speaker:
        print("  [FIX-v3-2] Tidak ada kolom speaker → skip speaker CMVN")
        return cache_mfcc

    unique_spks = set(dict_speaker.values())
    if len(unique_spks) <= 1:
        print(f"  [FIX-v3-2] Hanya {len(unique_spks)} speaker → skip speaker CMVN")
        return cache_mfcc

    spk_frames = defaultdict(list)
    for aid, feat in cache_mfcc.items():
        spk_frames[dict_speaker.get(aid, "unknown")].append(feat)

    spk_mean, spk_std = {}, {}
    for spk, frames in spk_frames.items():
        all_f = np.vstack(frames)
        spk_mean[spk] = all_f.mean(0)
        spk_std[spk]  = all_f.std(0) + FEAT_EPS

    result = {}
    for aid, feat in cache_mfcc.items():
        spk = dict_speaker.get(aid, "unknown")
        result[aid] = apply_cmvn((feat - spk_mean[spk]) / spk_std[spk])
    print(f"  Speaker CMVN diterapkan: {len(unique_spks)} speaker")
    return result


def _delta_numpy(feat: np.ndarray, w: int = _DELTA_W) -> np.ndarray:
    T = feat.shape[0]; den = 2.0 * sum(n*n for n in range(1, w+1))
    pad = np.pad(feat, ((w,w),(0,0)), mode="edge"); d = np.zeros_like(feat)
    for n in range(1, w+1):
        d += n * (pad[w+n:w+n+T] - pad[w-n:w-n+T])
    return d / den


def _mfcc_librosa_alpha(wav_path: str, alpha: float = 1.0) -> np.ndarray:
    """
    [NEW-1] Ekstraksi MFCC dengan VTLN alpha via librosa.
    Untuk alpha=1.0 identik dengan v3. Untuk alpha≠1.0, filterbank di-warp.
    """
    y, _ = librosa.load(wav_path, sr=_SR, mono=True)
    if len(y) == 0: raise ValueError(f"Audio kosong: {wav_path}")
    y = np.append(y[0], y[1:] - _PRE_EMPH * y[:-1])

    # Spectrum
    stft = np.abs(librosa.stft(y, n_fft=_N_FFT, hop_length=_HOP_LENGTH,
                                 win_length=_WIN_LENGTH, window="hamming")) ** 2

    # [NEW-1] Gunakan filterbank yang di-warp
    fb = build_vtln_filterbank(_N_MELS, _N_FFT, _SR, alpha, _FMIN, _FMAX)
    mel_spec = np.maximum(fb @ stft, 1e-10)
    log_mel  = np.log(mel_spec)

    # DCT untuk MFCC
    N = _N_MELS
    dct = np.cos(np.pi * np.arange(_N_MFCC)[:, None] *
                 (np.arange(N) + 0.5)[None, :] / N)
    mfcc = (log_mel.T @ dct.T) * np.sqrt(2.0 / N)  # (T, n_mfcc)

    d1 = librosa.feature.delta(mfcc.T, width=_DELTA_W*2+1).T
    d2 = librosa.feature.delta(mfcc.T, order=2, width=_DELTA_W*2+1).T
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


def _mfcc_scipy_alpha(wav_path: str, alpha: float = 1.0) -> np.ndarray:
    """
    [NEW-1] Ekstraksi MFCC + VTLN fallback (scipy).
    [BUG-FIX-2] Warp diterapkan pada filterbank bins, konsisten dengan librosa.
    """
    sr, audio = _wavfile.read(wav_path)
    if audio.dtype == np.int16:   audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32: audio = audio.astype(np.float32) / 2147483648.0
    else:                          audio = audio.astype(np.float32)
    if audio.ndim > 1: audio = audio.mean(1)
    if sr != _SR:
        n_new = int(len(audio) * _SR / sr)
        audio = np.interp(np.linspace(0, len(audio)-1, n_new),
                          np.arange(len(audio)), audio)

    y = np.append(audio[0], audio[1:] - _PRE_EMPH * audio[:-1])
    nf = max(1, 1 + (len(y) - _WIN_LENGTH) // _HOP_LENGTH)
    idx = np.arange(_WIN_LENGTH)[None,:] + _HOP_LENGTH * np.arange(nf)[:,None]
    idx = np.clip(idx, 0, len(y)-1)
    fr  = y[idx].astype(np.float32) * np.hamming(_WIN_LENGTH).astype(np.float32)
    mag = np.abs(np.fft.rfft(fr, n=_N_FFT)) ** 2  # power spectrum (T, nfft//2+1)

    # [NEW-1, BUG-FIX-2] Filterbank dengan VTLN warp
    fb = build_vtln_filterbank(_N_MELS, _N_FFT, _SR, alpha, _FMIN, _FMAX)
    log_mel = np.log10(np.maximum(mag @ fb.T, 1e-10))  # (T, n_mels)

    N = _N_MELS
    dct = np.cos(np.pi * np.arange(_N_MFCC)[:, None] *
                 (np.arange(N) + 0.5)[None, :] / N)
    mfcc = (log_mel @ dct.T) * np.sqrt(2.0 / N)
    d1 = _delta_numpy(mfcc); d2 = _delta_numpy(d1)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


def extract_features_alpha(wav_path: str, alpha: float = 1.0,
                            cmvn: bool = True) -> np.ndarray:
    """Ekstraksi fitur dengan VTLN alpha (alpha=1.0 = tanpa warp)."""
    if not os.path.isfile(wav_path): raise FileNotFoundError(wav_path)
    feat = (_mfcc_librosa_alpha(wav_path, alpha) if _LIBROSA
            else _mfcc_scipy_alpha(wav_path, alpha))
    return apply_cmvn(feat) if cmvn else feat


def build_mfcc_cache(sampled_root: str, dict_transkrip: dict,
                     alpha_map: Optional[Dict[str, float]] = None,
                     dict_speaker: Optional[Dict[str, str]] = None) -> Tuple[dict, list]:
    """
    [NEW-1] Bangun cache MFCC. Jika alpha_map tersedia (hasil VTLN search),
    gunakan alpha per speaker untuk ekstraksi. Default alpha=1.0.

    alpha_map: {speaker_id: alpha} atau None
    dict_speaker: {audio_id: speaker_id}
    """
    wf  = glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True)
    wf += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
    cache, meta = {}, []
    print(f"  Ekstrak MFCC dari {len(wf)} file ...")
    t0 = time.time()
    for idx, wp in enumerate(wf, 1):
        aid = os.path.splitext(os.path.basename(wp))[0]
        if not dict_transkrip.get(aid, ""): continue
        try:
            # [NEW-1] Tentukan alpha per speaker
            alpha = 1.0
            if alpha_map and dict_speaker:
                spk = dict_speaker.get(aid, "unknown")
                alpha = alpha_map.get(spk, 1.0)
            feat = extract_features_alpha(wp, alpha=alpha, cmvn=True)
            cache[aid] = feat
            meta.append({"Id": aid, "Frames": feat.shape[0],
                          "Dim": feat.shape[1], "VTLN_alpha": alpha})
        except Exception as e:
            print(f"    [SKIP] {os.path.basename(wp)}: {e}")
        if idx % 200 == 0:
            print(f"    [{idx}/{len(wf)}] {time.time()-t0:.0f}s")
    print(f"  Selesai: {len(cache)} audio")
    return cache, meta


# =============================================================================
#  [NEW-1] VTLN ALPHA SEARCH
# =============================================================================
def cari_vtln_alpha_per_speaker(
    wav_paths_per_speaker: Dict[str, List[str]],
    gmm_mono: Dict,
    dict_transkrip: Dict[str, str],
    lexicon: Dict,
    alpha_range: List[float] = VTLN_ALPHA_RANGE,
) -> Dict[str, float]:
    """
    [NEW-1] Grid search VTLN alpha per speaker.

    Untuk setiap speaker dan setiap alpha:
      1. Ekstrak fitur dengan alpha tersebut (tanpa CMVN agar warp murni)
      2. Hitung log-likelihood terhadap GMM monophone yang ada
      3. Pilih alpha dengan LL tertinggi

    Jika GMM belum tersedia (iterasi pertama), kembalikan alpha=1.0 semua.

    Returns:
        {speaker_id: alpha_optimal}
    """
    if not gmm_mono:
        print("  [VTLN] GMM belum ada → skip search, alpha=1.0 untuk semua")
        return {spk: 1.0 for spk in wav_paths_per_speaker}

    speaker_alpha = {}
    t0 = time.time()
    print(f"  [VTLN] Grid search α ∈ {alpha_range} untuk {len(wav_paths_per_speaker)} speaker ...")

    for spk, paths in wav_paths_per_speaker.items():
        best_alpha, best_ll = 1.0, -np.inf
        for alpha in alpha_range:
            total_ll = 0.0; n_frame = 0
            for wp in paths[:10]:  # batasi 10 file per speaker untuk efisiensi
                aid = os.path.splitext(os.path.basename(wp))[0]
                label = dict_transkrip.get(aid, "")
                if not label: continue
                try:
                    feat = extract_features_alpha(wp, alpha=alpha, cmvn=True)
                except Exception:
                    continue
                fs = _bangun_fonem_seq(label, lexicon)
                if not fs: continue
                # Rata-rata LL semua fonem dalam kalimat
                fonem_unik = list(set(fs))
                for fonem in fonem_unik:
                    if fonem in gmm_mono:
                        ll_vals = gmm_mono[fonem].log_prob_per_frame(feat)
                        total_ll += float(ll_vals.mean())
                        n_frame  += len(ll_vals)
            if n_frame > 0:
                avg_ll = total_ll / n_frame
                if avg_ll > best_ll:
                    best_ll = avg_ll; best_alpha = alpha

        speaker_alpha[spk] = best_alpha

    n_nondefault = sum(1 for a in speaker_alpha.values() if a != 1.0)
    print(f"  [VTLN] Selesai {time.time()-t0:.1f}s | {n_nondefault}/{len(speaker_alpha)} speaker punya α≠1.0")
    return speaker_alpha


def bangun_wav_per_speaker(sampled_root: str, dict_speaker: Dict[str, str]) -> Dict[str, List[str]]:
    """Kumpulkan path wav per speaker_id."""
    wf  = glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True)
    wf += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
    spk2wav = defaultdict(list)
    for wp in wf:
        aid = os.path.splitext(os.path.basename(wp))[0]
        spk = dict_speaker.get(aid, "unknown")
        spk2wav[spk].append(wp)
    return dict(spk2wav)


# =============================================================================
#  GMM
# =============================================================================
class GaussianMixtureModel:
    def __init__(self, n_components=4, n_iter=100, tol=1e-3,
                 reg_covar=COV_REG, random_state=None):
        self.n_components = n_components; self.n_iter = n_iter
        self.tol = tol; self.reg_covar = reg_covar; self.random_state = random_state
        self.weights_ = self.means_ = self.covariances_ = None
        self.converged_ = False; self.n_iter_done_ = 0
        self.log_likelihood_history_: List[float] = []

    def _init_params(self, X):
        rng = np.random.default_rng(self.random_state); K = self.n_components
        idx = rng.choice(X.shape[0], size=K, replace=False)
        self.means_ = X[idx].copy(); self.weights_ = np.full(K, 1.0/K)
        self.covariances_ = np.tile(np.var(X, axis=0) + self.reg_covar, (K, 1))

    def _log_gaussian_all(self, X):
        mu = self.means_; var = self.covariances_
        log_det = np.sum(np.log(var), axis=1)
        diff = X[:,None,:] - mu[None,:,:]
        maha = np.sum(diff**2 / var[None,:,:], axis=2)
        return (-0.5*(X.shape[1]*np.log(2*np.pi)+log_det))[None,:] - 0.5*maha

    def _e_step(self, X):
        lw = np.log(self.weights_+LOG_EPS) + self._log_gaussian_all(X)
        lse = np.max(lw, axis=1, keepdims=True)
        ls = lse.reshape(-1) + np.log(np.sum(np.exp(lw-lse), axis=1))
        return lw - ls[:,None], ls

    def _m_step(self, X, lr):
        r = np.exp(lr); Nk = r.sum(0) + LOG_EPS
        self.weights_ = Nk / X.shape[0]; self.means_ = (r.T @ X) / Nk[:,None]
        diff = X[:,None,:] - self.means_[None,:,:]
        self.covariances_ = (np.sum(r[:,:,None]*diff**2, axis=0)/Nk[:,None] + self.reg_covar)

    def fit(self, X):
        assert X.ndim == 2
        self.n_components = min(self.n_components, max(1, X.shape[0]//4))
        self._init_params(X); self.log_likelihood_history_ = []; prev = -np.inf
        for it in range(self.n_iter):
            lr, ls = self._e_step(X); ll = float(ls.mean())
            self.log_likelihood_history_.append(ll)
            if abs(ll - prev) < self.tol:
                self.converged_ = True; self.n_iter_done_ = it+1; break
            prev = ll; self._m_step(X, lr)
        else:
            self.converged_ = False; self.n_iter_done_ = self.n_iter
        return self

    def log_prob_per_frame(self, X) -> np.ndarray:
        assert X.ndim == 2
        lw = np.log(self.weights_+LOG_EPS) + self._log_gaussian_all(X)
        lse = np.max(lw, axis=1, keepdims=True)
        return lse.reshape(-1) + np.log(np.sum(np.exp(lw-lse), axis=1))


# =============================================================================
#  [NEW-2] SAT / fMLLR (Feature-space Maximum Likelihood Linear Regression)
# =============================================================================
class FMLLRTransform:
    """
    [NEW-2] Feature-space MLLR (fMLLR / CMLLR).

    Estimasi transform affine: x̂ = A · x + b
    di mana A ∈ ℝ^(D×D), b ∈ ℝ^D.

    Penggabungan: W = [A | b], x_ext = [x; 1] → x̂ = W · x_ext

    Estimasi via auxiliary function maximization (Gales 1998):
        Q(W) = Σ_t Σ_k γ_tk [ -0.5 · (x̂_t - μ_k)^T Σ_k^{-1} (x̂_t - μ_k) ]

    Optimasi closed-form per baris W_i menggunakan cofactor expansion
    untuk menjaga det(A) > 0.
    """

    def __init__(self, feat_dim: int = FEAT_DIM):
        D = feat_dim
        self.D = D
        # Inisialisasi sebagai identity transform
        self.A = np.eye(D, dtype=np.float64)
        self.b = np.zeros(D, dtype=np.float64)

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Terapkan transform: X_hat = X @ A.T + b"""
        return (X.astype(np.float64) @ self.A.T + self.b).astype(np.float32)

    def fit(self, X: np.ndarray, gmm_per_fonem: Dict,
            fonem_labels: Optional[np.ndarray] = None,
            n_iters: int = FMLLR_ITERS) -> "FMLLRTransform":
        """
        [NEW-2] Estimasi A, b dari data X dan GMM.

        Algoritma:
          1. E-step: hitung posterior γ_tk per frame t dan per komponen k
             dari GMM fonem yang sesuai (atau semua GMM jika label tidak ada)
          2. Hitung statistik sufisien: G_i, k_i per dimensi i
          3. M-step: update W_i = k_i @ G_i^{-1} per baris

        fonem_labels: array int (T,) label fonem per frame. Jika None,
                      gunakan rata-rata semua GMM.
        """
        if X.shape[0] < FMLLR_MIN_FRAME:
            return self  # tidak cukup data, pertahankan identity

        D = self.D
        X_d = X.astype(np.float64)
        T = X_d.shape[0]

        # Kumpulkan semua komponen GMM
        all_means  = []  # (K_total, D)
        all_vars   = []  # (K_total, D)
        all_weights = [] # (K_total,)

        for fonem, gmm in gmm_per_fonem.items():
            if gmm.means_ is None: continue
            all_means.append(gmm.means_)
            all_vars.append(gmm.covariances_)
            all_weights.append(gmm.weights_)

        if not all_means:
            return self

        means   = np.vstack(all_means)   # (K, D)
        vars_   = np.vstack(all_vars)    # (K, D) diagonal covariance
        weights = np.concatenate(all_weights)  # (K,)
        weights = weights / (weights.sum() + LOG_EPS)
        K_total = means.shape[0]

        for _ in range(n_iters):
            # Terapkan transform saat ini untuk hitung posterior
            X_hat = X_d @ self.A.T + self.b  # (T, D)

            # E-step: posterior γ (T, K)
            log_gamma = np.log(weights[None,:] + LOG_EPS)  # (1, K)
            diff = X_hat[:,None,:] - means[None,:,:]        # (T, K, D)
            log_gamma -= 0.5 * np.sum(
                np.log(vars_[None,:,:] + COV_REG) +
                diff**2 / (vars_[None,:,:] + COV_REG),
                axis=2
            )  # (T, K)
            # Numerically stable softmax
            log_gamma -= log_gamma.max(axis=1, keepdims=True)
            gamma = np.exp(log_gamma)
            gamma /= gamma.sum(axis=1, keepdims=True) + LOG_EPS  # (T, K)

            # Statistik sufisien
            # G_i = Σ_t Σ_k γ_tk / σ_{ki}^2 · x_ext_t x_ext_t^T   (D+1 × D+1)
            # k_i = Σ_t Σ_k γ_tk / σ_{ki}^2 · μ_{ki} · x_ext_t^T  (D+1,)

            X_ext = np.hstack([X_d, np.ones((T, 1))])  # (T, D+1)
            W_new = np.zeros((D, D+1), dtype=np.float64)

            for i in range(D):
                # Bobot efektif per frame dari semua komponen untuk dimensi i
                # weight_t = Σ_k γ_tk / (σ_{ki}^2)
                inv_var_i = 1.0 / (vars_[:, i] + COV_REG)  # (K,)
                w_t = gamma @ inv_var_i  # (T,)  Σ_k γ_tk / σ_{ki}^2

                # G_i = Σ_t w_t · x_ext_t x_ext_t^T
                G_i = (X_ext * w_t[:,None]).T @ X_ext  # (D+1, D+1)
                G_i += np.eye(D+1) * 1e-8  # regularisasi

                # k_i = Σ_t Σ_k γ_tk μ_{ki} / σ_{ki}^2 · x_ext_t
                # = Σ_t [ x_ext_t · Σ_k γ_tk μ_{ki} / σ_{ki}^2 ]
                mu_weighted_i = gamma @ (means[:, i] * inv_var_i)  # (T,)
                k_i = (X_ext * mu_weighted_i[:,None]).sum(axis=0)   # (D+1,)

                # Update W_i = k_i @ G_i^{-1}
                try:
                    W_new[i] = np.linalg.solve(G_i.T, k_i)
                except np.linalg.LinAlgError:
                    W_new[i, :D] = self.A[i]
                    W_new[i, D]  = self.b[i]

            # Ekstrak A, b dari W_new
            A_new = W_new[:, :D]
            b_new = W_new[:, D]

            # Cek det > 0 untuk menjaga invertibilitas transform
            if np.linalg.det(A_new) > 1e-10:
                self.A = A_new
                self.b = b_new
            # else: pertahankan A, b sebelumnya (transform lama lebih stabil)

        return self


def estimasi_fmllr_per_speaker(
    cache_mfcc: Dict[str, np.ndarray],
    dict_speaker: Dict[str, str],
    gmm_per_fonem: Dict,
    alignment_cache: Optional[Dict[str, np.ndarray]] = None,
    audio_ids: Optional[List[str]] = None,
) -> Dict[str, FMLLRTransform]:
    """
    [NEW-2] Estimasi fMLLR transform per speaker.

    Mengumpulkan semua frame satu speaker, lalu fit FMLLRTransform.
    Jika alignment_cache tersedia, label fonem per frame diteruskan.

    Returns:
        {speaker_id: FMLLRTransform}
    """
    if not dict_speaker:
        print("  [SAT] Tidak ada info speaker → skip fMLLR")
        return {}

    ids = audio_ids or list(cache_mfcc.keys())
    spk2feat: Dict[str, List[np.ndarray]] = defaultdict(list)
    spk2align: Dict[str, List[np.ndarray]] = defaultdict(list)

    for aid in ids:
        feat = cache_mfcc.get(aid)
        if feat is None: continue
        spk = dict_speaker.get(aid, "unknown")
        spk2feat[spk].append(feat)
        if alignment_cache and aid in alignment_cache:
            spk2align[spk].append(alignment_cache[aid])

    fmllr_per_spk: Dict[str, FMLLRTransform] = {}
    print(f"  [SAT] Estimasi fMLLR untuk {len(spk2feat)} speaker ...")
    t0 = time.time()

    for spk, feats in spk2feat.items():
        X_spk = np.vstack(feats)  # (T_total, D)
        aligns = spk2align.get(spk, [])
        labels = np.concatenate(aligns) if aligns else None

        tr = FMLLRTransform(feat_dim=X_spk.shape[1])
        tr.fit(X_spk, gmm_per_fonem, fonem_labels=labels)
        fmllr_per_spk[spk] = tr

    n_nontrivial = sum(
        1 for tr in fmllr_per_spk.values()
        if not np.allclose(tr.A, np.eye(tr.D), atol=1e-4)
    )
    print(f"  [SAT] Selesai {time.time()-t0:.1f}s | {n_nontrivial}/{len(fmllr_per_spk)} speaker punya transform non-trivial")
    return fmllr_per_spk


def apply_fmllr_cache(
    cache_mfcc: Dict[str, np.ndarray],
    fmllr_per_spk: Dict[str, "FMLLRTransform"],
    dict_speaker: Dict[str, str],
) -> Dict[str, np.ndarray]:
    """
    [NEW-2] Terapkan fMLLR transform ke seluruh cache MFCC.
    Audio tanpa info speaker menggunakan identity transform.
    """
    if not fmllr_per_spk:
        return cache_mfcc
    result = {}
    for aid, feat in cache_mfcc.items():
        spk = dict_speaker.get(aid, "unknown")
        tr  = fmllr_per_spk.get(spk)
        result[aid] = tr.transform(feat) if tr is not None else feat
    return result


def apply_fmllr_single(feat: np.ndarray, spk: str,
                        fmllr_per_spk: Dict[str, "FMLLRTransform"]) -> np.ndarray:
    """[NEW-2] Apply fMLLR pada satu utterance (untuk test-time adaptation)."""
    tr = fmllr_per_spk.get(spk)
    return tr.transform(feat) if tr is not None else feat


# =============================================================================
#  ALIGNMENT — VITERBI VEKTORISASI [FIX-v3-4]
# =============================================================================
def alignment_fonem_viterbi(fitur: np.ndarray, fonem_seq: list,
                             gmm_per_fonem: Dict) -> np.ndarray:
    """[FIX-v3-4] Viterbi alignment dengan vektorisasi NumPy."""
    T = fitur.shape[0]; n = len(fonem_seq)
    if n == 0 or T == 0: return np.full(T, SIL_ID, dtype=np.int32)

    skor = np.full((T, n), LOG_ZERO, dtype=np.float64)
    for j, fonem in enumerate(fonem_seq):
        if fonem in gmm_per_fonem:
            skor[:, j] = gmm_per_fonem[fonem].log_prob_per_frame(fitur)
        elif fonem == "SIL":
            skor[:, j] = 0.0

    dp = np.full((T+1, n+1), LOG_ZERO, dtype=np.float64)
    bp = np.zeros((T+1, n+1), dtype=np.int8)
    dp[0, 0] = 0.0

    for t in range(1, T+1):
        e = skor[t-1]
        stay    = dp[t-1, :n] + e
        advance = np.full(n, LOG_ZERO)
        advance[1:] = dp[t-1, :n-1] + e[1:]
        use_adv = advance > stay
        dp[t, :n] = np.where(use_adv, advance, stay)
        bp[t, :n] = np.where(use_adv, 1, 0)
        dp[t, :n] = np.where(dp[t, :n] > LOG_ZERO, dp[t, :n], LOG_ZERO)

    valid = [(dp[T, j], j) for j in range(n) if dp[T, j] > LOG_ZERO]
    j_end = valid[-1][1] if valid else n-1
    path  = np.zeros(T, dtype=int); j = j_end
    for t in range(T, 0, -1):
        path[t-1] = j
        if bp[t, j] == 1: j = max(j-1, 0)

    return np.array([FONEM2ID.get(fonem_seq[k], SIL_ID) for k in path], dtype=np.int32)


def alignment_triphone_viterbi(fitur: np.ndarray, fonem_seq: list,
                                gmm_tri: Dict, gmm_mono: Dict) -> np.ndarray:
    triphone_seq = buat_triphone_seq(fonem_seq)
    T = fitur.shape[0]; n = len(triphone_seq)
    if n == 0 or T == 0: return np.full(T, SIL_ID, dtype=np.int32)

    skor = np.full((T, n), LOG_ZERO, dtype=np.float64)
    for j, tri in enumerate(triphone_seq):
        if tri in gmm_tri:
            skor[:, j] = gmm_tri[tri].log_prob_per_frame(fitur)
        else:
            center = triphone_center(tri)
            if center in gmm_mono: skor[:, j] = gmm_mono[center].log_prob_per_frame(fitur)
            elif center == "SIL":  skor[:, j] = 0.0

    dp = np.full((T+1, n+1), LOG_ZERO, dtype=np.float64)
    bp = np.zeros((T+1, n+1), dtype=np.int8)
    dp[0, 0] = 0.0
    for t in range(1, T+1):
        e = skor[t-1]; stay = dp[t-1, :n] + e
        advance = np.full(n, LOG_ZERO)
        advance[1:] = dp[t-1, :n-1] + e[1:]
        use_adv = advance > stay
        dp[t, :n] = np.where(use_adv, advance, stay)
        bp[t, :n] = np.where(use_adv, 1, 0)
        dp[t, :n] = np.where(dp[t, :n] > LOG_ZERO, dp[t, :n], LOG_ZERO)

    valid  = [(dp[T, j], j) for j in range(n) if dp[T, j] > LOG_ZERO]
    j_end  = valid[-1][1] if valid else n-1
    path   = np.zeros(T, dtype=int); j = j_end
    for t in range(T, 0, -1):
        path[t-1] = j
        if bp[t, j] == 1: j = max(j-1, 0)

    centers = [triphone_center(triphone_seq[k]) for k in path]
    return np.array([FONEM2ID.get(c, SIL_ID) for c in centers], dtype=np.int32)


def kumpulkan_frame_fonem_naif(fitur: np.ndarray, fonem_seq: list) -> dict:
    T, n = fitur.shape[0], len(fonem_seq)
    if n == 0: return {}
    nf = np.ones(n, dtype=int) * (T // n)
    nf[:T - nf.sum()] += 1
    hasil = {}; idx = 0
    for i, f in enumerate(fonem_seq):
        seg = fitur[idx:idx+nf[i]]
        if seg.shape[0] > 0:
            hasil[f] = np.vstack([hasil[f], seg]) if f in hasil else seg.copy()
        idx += nf[i]
    return hasil


def kumpulkan_frame_per_fonem(cache_mfcc, dict_transkrip, lexicon,
                               gmm_mono=None, gmm_tri=None,
                               audio_ids=None) -> Dict[str, np.ndarray]:
    use_tri  = USE_TRIPHONE and gmm_tri  is not None and len(gmm_tri)  > 0
    use_mono = gmm_mono is not None and len(gmm_mono) > 0
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)
    ids = audio_ids if audio_ids else list(cache_mfcc.keys())

    for audio_id in ids:
        fitur = cache_mfcc.get(audio_id); label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label: continue
        fs = _bangun_fonem_seq(label, lexicon)
        if not fs: continue
        if use_tri:
            lbls = alignment_triphone_viterbi(fitur, fs, gmm_tri, gmm_mono or {})
            for fid in np.unique(lbls):
                buf[FONEM_JAWA[fid]].append(fitur[lbls == fid])
        elif use_mono:
            lbls = alignment_fonem_viterbi(fitur, fs, gmm_mono)
            for fid in np.unique(lbls):
                buf[FONEM_JAWA[fid]].append(fitur[lbls == fid])
        else:
            for fonem, arr in kumpulkan_frame_fonem_naif(fitur, fs).items():
                buf[fonem].append(arr)

    return {f: np.vstack(parts) for f, parts in buf.items()}


def kumpulkan_frame_per_triphone(cache_mfcc, dict_transkrip, lexicon,
                                  gmm_mono, audio_ids=None) -> Dict[str, np.ndarray]:
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)
    ids = audio_ids if audio_ids else list(cache_mfcc.keys())
    for audio_id in ids:
        fitur = cache_mfcc.get(audio_id); label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label: continue
        fs   = _bangun_fonem_seq(label, lexicon)
        if not fs: continue
        tris = buat_triphone_seq(fs); T = fitur.shape[0]
        n_tri = len(tris); seg_len = max(1, T // n_tri) if n_tri > 0 else T
        idx = 0
        for k, tri in enumerate(tris):
            end = idx + seg_len if k < n_tri-1 else T
            seg = fitur[idx:end]
            if seg.shape[0] > 0: buf[tri].append(seg)
            idx = end
    return {tri: np.vstack(parts) for tri, parts in buf.items()
            if np.vstack(parts).shape[0] >= TRIPHONE_MIN_FRAME}


def latih_gmm_per_fonem(frame_per_fonem, n_components=N_COMPONENTS_GMM,
                         n_iter=N_ITER_GMM, tol=TOL_GMM, min_frame=MIN_FRAME_GMM,
                         verbose=True) -> Dict:
    gmm = {}; t0 = time.time()
    for i, (fonem, X) in enumerate(sorted(frame_per_fonem.items())):
        if X.shape[0] < min_frame:
            if verbose: print(f"    SKIP {fonem}: {X.shape[0]} frame")
            continue
        g = GaussianMixtureModel(n_components=n_components, n_iter=n_iter, tol=tol,
                                  reg_covar=COV_REG, random_state=hash(fonem) % 100_000)
        g.fit(X); gmm[fonem] = g
        if verbose:
            ll = round(g.log_likelihood_history_[-1], 3)
            print(f"    [{i+1:2d}] {fonem:4s}: {X.shape[0]:6d} frame, "
                  f"K={g.n_components}, iter={g.n_iter_done_}, conv={g.converged_}, LL={ll}")
    print(f"  GMM: {len(gmm)} model dalam {time.time()-t0:.1f}s")
    return gmm


def latih_gmm_iteratif(
    cache_train: Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon: Dict,
    n_iters: int = GMM_EM_ITERS,
) -> Tuple[Dict, Dict, Dict[str, np.ndarray]]:
    """
    [BUG-FIX-1] Sekarang me-return (gmm_mono, gmm_tri, frame_per_fonem)
    agar frame_per_fonem bisa digunakan untuk statistik GMM di main()
    tanpa variabel global.
    """
    gmm_mono = {}; frame_per_fonem = {}
    print("="*60); print("FASE 1: Iterative monophone GMM")
    for it in range(n_iters):
        print(f"\n  Iter {it+1}/{n_iters} ...")
        frame_per_fonem = kumpulkan_frame_per_fonem(
            cache_train, dict_transkrip, lexicon,
            gmm_mono=gmm_mono if it > 0 else None
        )
        gmm_mono = latih_gmm_per_fonem(frame_per_fonem, verbose=(it == n_iters-1))
        print(f"  → {len(gmm_mono)} model monophone")

    gmm_tri = {}
    if USE_TRIPHONE:
        print("\n" + "="*60); print("FASE 2: Triphone GMM")
        fpt = kumpulkan_frame_per_triphone(cache_train, dict_transkrip, lexicon, gmm_mono)
        gmm_tri = latih_gmm_per_fonem(
            fpt, n_components=max(2, N_COMPONENTS_GMM//2),
            min_frame=TRIPHONE_MIN_FRAME, verbose=False
        )
        print(f"  → {len(gmm_tri)} model triphone")

    # [BUG-FIX-1] Return frame_per_fonem
    return gmm_mono, gmm_tri, frame_per_fonem


def hitung_alignment_corpus(audio_ids, cache_mfcc, dict_transkrip,
                             lexicon, gmm_mono, gmm_tri=None) -> Dict[str, np.ndarray]:
    use_tri = USE_TRIPHONE and gmm_tri is not None and len(gmm_tri) > 0
    ac = {}
    for aid in audio_ids:
        fitur = cache_mfcc.get(aid); label = dict_transkrip.get(aid, "")
        if fitur is None or not label: continue
        fs = _bangun_fonem_seq(label, lexicon)
        if not fs: continue
        if use_tri:
            ac[aid] = alignment_triphone_viterbi(fitur, fs, gmm_tri, gmm_mono)
        else:
            ac[aid] = alignment_fonem_viterbi(fitur, fs, gmm_mono)
    return ac


# =============================================================================
#  DATASET — [FIX-v3-6]
# =============================================================================
class FrameLevelDataset(Dataset):
    """
    [FIX-v3-6] Gunakan alignment_cache yang sudah dihitung.
    [NEW-2] Fitur yang masuk sudah melalui fMLLR transform.
    """
    def __init__(self, audio_ids, cache_mfcc, alignment_cache):
        fp, lp = [], []
        for aid in audio_ids:
            fitur  = cache_mfcc.get(aid)
            aligns = alignment_cache.get(aid)
            if fitur is None or aligns is None: continue
            fp.append(fitur); lp.append(aligns)
        if fp:
            self.frames = torch.from_numpy(np.vstack(fp))
            self.labels = torch.from_numpy(np.concatenate(lp).astype(np.int64))
        else:
            self.frames = torch.zeros((0, FEAT_DIM), dtype=torch.float32)
            self.labels = torch.zeros(0, dtype=torch.long)
        print(f"  Dataset: {len(self.frames):,} frame")

    def __len__(self): return len(self.frames)
    def __getitem__(self, i): return self.frames[i], self.labels[i]


# =============================================================================
#  TDNN — [FIX-v3-1]
# =============================================================================
class TDNNLayer(nn.Module):
    def __init__(self, input_dim, output_dim, context, dropout=0.0):
        super().__init__()
        self.context = sorted(context)
        self.linear  = nn.Linear(input_dim * len(context), output_dim)
        self.bn      = nn.BatchNorm1d(output_dim)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x):
        sq = x.dim() == 2
        if sq: x = x.unsqueeze(1)
        B, T, _ = x.shape
        cmin = min(self.context); cmax = max(self.context)
        xp = F.pad(x.permute(0,2,1), (max(0,-cmin), max(0,cmax))).permute(0,2,1)
        ctx = [xp[:, c-cmin:c-cmin+T, :] for c in self.context]
        sp  = torch.cat(ctx, dim=-1)
        out = self.linear(sp)
        out = self.bn(out.permute(0,2,1)).permute(0,2,1)
        out = F.relu(out); out = self.drop(out)
        return out.squeeze(1) if sq else out


class TDNNAcousticModel(nn.Module):
    def __init__(self, input_dim=FEAT_DIM, n_classes=N_FONEM,
                 hidden_dim=TDNN_HIDDEN, contexts=None, dropout=TDNN_DROPOUT):
        super().__init__()
        if contexts is None: contexts = TDNN_CONTEXTS
        layers = []; in_dim = input_dim
        for ctx in contexts:
            layers.append(TDNNLayer(in_dim, hidden_dim, ctx, dropout))
            in_dim = hidden_dim
        self.tdnn_layers = nn.ModuleList(layers)
        self.classifier  = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        for layer in self.tdnn_layers: x = layer(x)
        return F.log_softmax(self.classifier(x), dim=-1)

    def get_posteriors_sequence(self, x):
        with torch.no_grad(): return self.forward(x)[0]


# =============================================================================
#  TRAINING — [FIX-v3-1, FIX-v3-3]
# =============================================================================
class LabelSmoothingNLLLoss(nn.Module):
    def __init__(self, n_classes, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing; self.n_classes = n_classes; self.weight = weight

    def forward(self, log_probs, targets):
        with torch.no_grad():
            smooth_val  = self.smoothing / self.n_classes
            smooth_dist = torch.full_like(log_probs, smooth_val)
            smooth_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing + smooth_val)
        loss = -(smooth_dist * log_probs)
        if self.weight is not None:
            loss = loss * self.weight.unsqueeze(0)
        return loss.sum(dim=1).mean()


def latih_tdnn_frame_level(
    train_ids, val_ids, cache_mfcc, alignment_train, alignment_val,
    dir_save, epochs=TDNN_EPOCHS, lr=TDNN_LR, batch_size=TDNN_BATCH_SIZE,
    hidden_dim=TDNN_HIDDEN, contexts=None, dropout=TDNN_DROPOUT,
    warmup_epochs=TDNN_WARMUP_EPOCH,
) -> TDNNAcousticModel:
    """
    [NEW-2] cache_mfcc di sini sudah melalui fMLLR transform (speaker-normalized).
    Training TDNN pada fitur speaker-invariant → SAT.
    """
    if contexts is None: contexts = TDNN_CONTEXTS
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = os.path.join(dir_save, "model_tdnn_fonem_sat.pt")
    os.makedirs(dir_save, exist_ok=True)

    model = TDNNAcousticModel(input_dim=FEAT_DIM, n_classes=N_FONEM,
                               hidden_dim=hidden_dim, contexts=contexts,
                               dropout=dropout).to(device)

    if os.path.isfile(model_path):
        try:
            model.load_state_dict(torch.load(model_path, map_location=device))
            print(f"  Checkpoint dimuat: {model_path}"); return model
        except Exception:
            os.remove(model_path)

    print("  Membangun dataset (fMLLR-transformed feats) ...")
    train_ds = FrameLevelDataset(train_ids, cache_mfcc, alignment_train)
    val_ds   = FrameLevelDataset(val_ids,   cache_mfcc, alignment_val)
    if len(train_ds) == 0:
        print("  [ERROR] Train dataset kosong!"); return model

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                               num_workers=0, pin_memory=False)

    label_counts = torch.bincount(train_ds.labels, minlength=N_FONEM).float()
    label_counts = torch.clamp(label_counts, min=1.0)
    weights = (1.0 / torch.sqrt(label_counts))
    weights = (weights / weights.sum()).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                   weight_decay=TDNN_WEIGHT_DECAY)

    def lr_lambda(ep):
        return float(ep+1) / float(warmup_epochs) if ep < warmup_epochs else 1.0

    warmup_sched = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs - warmup_epochs, 1), eta_min=TDNN_LR_MIN)

    loss_fn = LabelSmoothingNLLLoss(N_FONEM, smoothing=TDNN_LABEL_SMOOTH, weight=weights)

    riwayat = []; best_val = float("inf"); t0 = time.time()
    print(f"\n  Device: {device} | Train: {len(train_ds):,} | Val: {len(val_ds):,}")
    print(f"  [SAT] TDNN dilatih pada fitur fMLLR-transformed (speaker-invariant)")
    print(f"  dropout={dropout}, weight_decay={TDNN_WEIGHT_DECAY}, label_smooth={TDNN_LABEL_SMOOTH}")

    for epoch in range(1, epochs+1):
        model.train(); tl = nc = nt = 0
        for feats, lbls in train_loader:
            feats = feats.to(device); lbls = lbls.to(device)
            optimizer.zero_grad()
            lp = model(feats); loss = loss_fn(lp, lbls)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            tl += loss.item()*len(lbls); nc += (lp.argmax(1)==lbls).sum().item(); nt += len(lbls)

        model.eval(); vl = vc = vt = 0
        with torch.no_grad():
            for feats, lbls in val_loader:
                feats = feats.to(device); lbls = lbls.to(device)
                lp = model(feats); vl += loss_fn(lp, lbls).item()*len(lbls)
                vc += (lp.argmax(1)==lbls).sum().item(); vt += len(lbls)

        if epoch <= warmup_epochs: warmup_sched.step()
        else: cosine_sched.step()

        atr = tl/max(nt,1); avl = vl/max(vt,1)
        actr = 100*nc/max(nt,1); acvl = 100*vc/max(vt,1)
        cur_lr = optimizer.param_groups[0]['lr']
        riwayat.append({"Epoch": epoch, "LR": round(cur_lr,7),
                         "Train Loss": round(atr,5), "Val Loss": round(avl,5),
                         "Train Acc (%)": round(actr,2), "Val Acc (%)": round(acvl,2)})
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs} | Loss {atr:.4f}/{avl:.4f} | "
                  f"Acc {actr:.1f}/{acvl:.1f}% | LR {cur_lr:.2e}")

        if avl < best_val:
            best_val = avl
            torch.save(model.state_dict(), model_path)

    if os.path.isfile(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"  Best model dimuat (val_loss={best_val:.5f})")

    pd.DataFrame(riwayat).to_csv(
        os.path.join(dir_save, "HasilTrainingTDNN_SAT.csv"), index=False)
    return model


# =============================================================================
#  [FIX-v3-5] DECODER
# =============================================================================
class TDNNHMMDecoder:
    """
    [FIX-v3-5] Decoder efisien: argmax + CTC collapse + fonem→kata.
    [NEW-2] Test-time adaptation: apply fMLLR sebelum decode jika ada.
    """
    def __init__(self, model, lexicon, ngram_model, gmm_per_fonem,
                 device=None, n_hmm_states=N_HMM_STATES, beam_width=BEAM_WIDTH,
                 fmllr_per_spk: Optional[Dict] = None,
                 dict_speaker: Optional[Dict] = None):
        self.model = model; self.lexicon = lexicon; self.ngram = ngram_model
        self.gmm_per_fonem = gmm_per_fonem
        self.device = device or torch.device("cpu")
        self.n_hmm_states = n_hmm_states; self.beam_width = beam_width
        # [NEW-2] test-time adaptation
        self.fmllr_per_spk = fmllr_per_spk or {}
        self.dict_speaker   = dict_speaker  or {}

        self.rev_lex: Dict[Tuple, List[str]] = defaultdict(list)
        for kata, fonems in lexicon.items():
            self.rev_lex[tuple(fonems)].append(kata)
        lengths = [len(v) for v in lexicon.values()]
        self.avg_fpw = float(np.mean(lengths)) if lengths else 3.0
        self.model.to(self.device); self.model.eval()

    def _adapt_test(self, fitur: np.ndarray, audio_id: str) -> np.ndarray:
        """
        [NEW-2] Test-time speaker adaptation:
        1. Cari speaker dari dict_speaker (jika ada)
        2. Jika ada fMLLR cached → gunakan langsung
        3. Jika tidak ada speaker info → estimasi fMLLR dari utterance ini sendiri
           (unsupervised adaptation, 1 iterasi)
        """
        if not self.fmllr_per_spk:
            return fitur
        spk = self.dict_speaker.get(audio_id, None)
        if spk and spk in self.fmllr_per_spk:
            return self.fmllr_per_spk[spk].transform(fitur)
        # Estimasi dari utterance tunggal (jika cukup frame)
        if fitur.shape[0] >= FMLLR_MIN_FRAME:
            tr = FMLLRTransform(feat_dim=fitur.shape[1])
            tr.fit(fitur, self.gmm_per_fonem, n_iters=1)
            return tr.transform(fitur)
        return fitur

    def _get_posteriors(self, fitur: np.ndarray) -> np.ndarray:
        t = torch.from_numpy(fitur).unsqueeze(0).to(self.device)
        return self.model.get_posteriors_sequence(t).cpu().numpy()

    def _ctc_collapse(self, fonem_ids: np.ndarray) -> List[str]:
        result = []; i = 0
        while i < len(fonem_ids):
            fid = fonem_ids[i]; j = i+1
            while j < len(fonem_ids) and fonem_ids[j] == fid: j += 1
            dur = j - i
            if dur >= self.n_hmm_states and fid != SIL_ID:
                result.append(FONEM_JAWA[fid])
            i = j
        return result

    def _fonem_ke_kata(self, fonem_seq: list, T: int) -> list:
        n = len(fonem_seq)
        if n == 0: return [[]]
        max_words = max(1, T // AVG_FRAMES_PER_WORD)
        dp = [[] for _ in range(n+1)]; dp[0] = [(0.0, [])]
        for i in range(n):
            if not dp[i]: continue
            for j in range(i+1, min(n+1, i+MAX_FONEM_PER_KATA+1)):
                sub = tuple(fonem_seq[i:j])
                if sub not in self.rev_lex: continue
                for kata in self.rev_lex[sub]:
                    for score, words in dp[i]:
                        if len(words) >= max_words: continue
                        new_words = words + [kata]
                        new_score = score + self.ngram.log_prob(
                            kata, tuple(words[-(self.ngram.n-1):]))
                        dp[j].append((new_score, new_words))
                if len(dp[j]) > self.beam_width * 3:
                    dp[j].sort(key=lambda x: x[0], reverse=True)
                    dp[j] = dp[j][:self.beam_width]
        if dp[n]:
            dp[n].sort(key=lambda x: x[0], reverse=True)
            return [w for _, w in dp[n][:self.beam_width]]
        return [fonem_seq]

    def dekode(self, fitur: np.ndarray, audio_id: str = "") -> list:
        if fitur.shape[0] == 0: return []
        # [NEW-2] Test-time adaptation
        fitur_adapted = self._adapt_test(fitur, audio_id)
        log_post  = self._get_posteriors(fitur_adapted)
        fonem_ids = np.argmax(log_post, axis=1)
        fonem_seq = self._ctc_collapse(fonem_ids)
        if not fonem_seq: return []
        kata_cand = self._fonem_ke_kata(fonem_seq, fitur.shape[0])
        if not kata_cand: return fonem_seq
        return self.ngram.rescore(kata_cand)


# =============================================================================
#  N-GRAM LM
# =============================================================================
class NGramModel:
    def __init__(self, n=3, discount=0.75):
        self.n = n; self.discount = discount
        self.ngram_count    = defaultdict(int)
        self.context_count  = defaultdict(int)
        self.unigram_count  = defaultdict(int)
        self.vocab          = set()
        self.N_total        = 0
        self._ctx_div       = {}

    def _tok(self, s):
        t = normalisasi(s).split()
        return ["<s>"] * (self.n-1) + t + ["</s>"]

    def fit(self, korpus):
        cw = defaultdict(set)
        for s in korpus:
            t = self._tok(s); self.vocab.update(t); self.N_total += len(t)
            for tok in t: self.unigram_count[tok] += 1
            for i in range(len(t) - self.n + 1):
                ng = tuple(t[i:i+self.n]); c = ng[:-1]
                self.ngram_count[ng] += 1
                self.context_count[c] += 1
                cw[c].add(ng[-1])
        self._ctx_div = {c: len(w) for c, w in cw.items()}
        return self

    def log_prob(self, kata, ctx):
        ng  = ctx + (kata,); Cng = self.ngram_count.get(ng, 0)
        Cc  = self.context_count.get(ctx, 0)
        Cu  = self.unigram_count.get(kata, 0)
        puni = (Cu + 1) / (self.N_total + len(self.vocab))
        if Cc == 0: return math.log(puni + LOG_EPS)
        lam = self.discount * self._ctx_div.get(ctx, 0) / Cc
        return math.log(max(Cng - self.discount, 0) / Cc + lam*puni + LOG_EPS)

    def score_sequence(self, words):
        t = ["<s>"] * (self.n-1) + words + ["</s>"]; lp = 0.0
        for i in range(self.n-1, len(t)):
            lp += self.log_prob(t[i], tuple(t[i-self.n+1:i]))
        return lp

    def rescore(self, kandidat):
        return max(kandidat, key=self.score_sequence) if kandidat else []


# =============================================================================
#  WER
# =============================================================================
def compute_wer(ref, hyp):
    R, H = len(ref), len(hyp)
    dp = [[0]*(H+1) for _ in range(R+1)]
    ops = [[""]*(H+1) for _ in range(R+1)]
    for i in range(R+1): dp[i][0] = i; ops[i][0] = "D"*i
    for j in range(H+1): dp[0][j] = j; ops[0][j] = "I"*j
    for i in range(1, R+1):
        for j in range(1, H+1):
            if ref[i-1] == hyp[j-1]: dp[i][j] = dp[i-1][j-1]; ops[i][j] = "M"
            else:
                sub = dp[i-1][j-1]+1; dlt = dp[i-1][j]+1; ins = dp[i][j-1]+1
                best = min(sub, dlt, ins); dp[i][j] = best
                ops[i][j] = "S" if best==sub else ("D" if best==dlt else "I")
    S = D = I = 0; i, j = R, H
    while i > 0 or j > 0:
        op = ops[i][j]
        if   op == "M": i-=1; j-=1
        elif op == "S": S+=1; i-=1; j-=1
        elif op == "D": D+=1; i-=1
        elif op == "I": I+=1; j-=1
        else:
            if i > 0: D+=1; i-=1
            else:     I+=1; j-=1
    return {"wer_persen": round((S+D+I)/R*100, 2) if R > 0 else 0.0,
            "S": S, "D": D, "I": I, "N": R}


def dekode_semua_audio(cache_mfcc, audio_ids, dict_transkrip, decoder, verbose=True):
    hasil = []; t0 = time.time()
    for idx, aid in enumerate(audio_ids, 1):
        fitur = cache_mfcc.get(aid); label = dict_transkrip.get(aid, "")
        if fitur is None or not label: continue
        ref = normalisasi(label).split()
        if not ref: continue
        try:
            hyp = decoder.dekode(fitur, audio_id=aid)  # [NEW-2] pass audio_id
        except Exception as e:
            if verbose: print(f"    [ERROR] {aid}: {e}"); continue
        wr = compute_wer(ref, hyp)
        hasil.append({"Id": aid, "Referensi": " ".join(ref), "Hipotesis": " ".join(hyp),
                       "WER (%)": wr["wer_persen"], "S": wr["S"], "D": wr["D"],
                       "I": wr["I"], "N": wr["N"]})
        if verbose and idx % 100 == 0:
            print(f"    [{idx}/{len(audio_ids)}] {time.time()-t0:.1f}s")
    return hasil


def ringkasan_wer(df: pd.DataFrame, label: str = "") -> dict:
    if df.empty: return {}
    S = int(df.S.sum()); D = int(df.D.sum()); I = int(df.I.sum()); N = int(df.N.sum())
    wc = (S+D+I)/N*100 if N > 0 else 100.0
    print(f"\n  === WER: {label} ===")
    print(f"  Audio: {len(df)} | WER corpus: {wc:.2f}% | Mean: {df['WER (%)'].mean():.2f}%")
    print(f"  S={S} D={D} I={I} N={N}")
    return {"wer_corpus": round(wc,2), "wer_mean": round(df['WER (%)'].mean(),2),
            "wer_median": round(df['WER (%)'].median(),2),
            "wer_std": round(df['WER (%)'].std(),2),
            "total_S": S, "total_D": D, "total_I": I, "total_N": N}


# =============================================================================
#  MAIN
# =============================================================================
def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    random.seed(42); np.random.seed(42)
    t_start = time.time()

    # ── [1] Load data ─────────────────────────────────────────────────────────
    print("\n[1/10] Membaca CSV ...")
    dict_transkrip, dict_speaker = baca_transkrip_csv(CSV_PATH)

    # ── [2] MFCC awal (alpha=1.0, tanpa VTLN) ─────────────────────────────────
    print("\n[2/10] Ekstraksi MFCC awal (alpha=1.0) ...")
    cache_mfcc_raw, meta = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    pd.DataFrame(meta).to_csv(f"{DIR_SAVE}/HasilMFCC_raw.csv", index=False)

    # Speaker CMVN
    cache_mfcc = apply_cmvn_speaker(cache_mfcc_raw, dict_speaker)

    # ── [3] Split ──────────────────────────────────────────────────────────────
    print("\n[3/10] Split train/val ...")
    all_ids = list(cache_mfcc.keys())
    if SPEAKER_SPLIT and dict_speaker:
        train_ids, val_ids = split_by_speaker(all_ids, dict_speaker, VAL_RATIO)
    else:
        random.shuffle(all_ids); n_val = max(1, int(len(all_ids)*VAL_RATIO))
        train_ids, val_ids = all_ids[n_val:], all_ids[:n_val]
        print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")

    # ── [4] Lexicon + LM ──────────────────────────────────────────────────────
    print("\n[4/10] Lexicon dan N-gram LM ...")
    semua_transkrip = list(dict_transkrip.values())
    lexicon = buat_lexicon(semua_transkrip)
    pd.DataFrame([{"Kata": k, "Fonem": " ".join(v)}
                  for k, v in lexicon.items()]).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)
    lm = NGramModel(n=N_GRAM)
    lm.fit([dict_transkrip[a] for a in train_ids if a in dict_transkrip])
    print(f"  Lexicon: {len(lexicon)} kata | LM vocab: {len(lm.vocab)}")

    # ── [5] GMM awal (pada fitur raw) ─────────────────────────────────────────
    print("\n[5/10] GMM awal (fitur raw, sebelum VTLN/SAT) ...")
    cache_train_raw = {k: cache_mfcc[k] for k in train_ids if k in cache_mfcc}
    # [BUG-FIX-1] gmm_iteratif return 3 nilai
    gmm_mono, gmm_tri, frame_per_fonem = latih_gmm_iteratif(
        cache_train_raw, dict_transkrip, lexicon)

    # Simpan statistik GMM raw
    gmm_stat = [
        {"Fonem": f, "Frames": frame_per_fonem.get(f, np.empty((0,1))).shape[0],
         "K": g.n_components, "Iter": g.n_iter_done_, "Conv": g.converged_,
         "LL": round(g.log_likelihood_history_[-1], 4) if g.log_likelihood_history_ else None}
        for f, g in gmm_mono.items()
    ]
    pd.DataFrame(gmm_stat).to_csv(f"{DIR_SAVE}/HasilGMM_raw.csv", index=False)

    # ── [6] VTLN: cari alpha per speaker ──────────────────────────────────────
    speaker_alpha_map = {}
    if USE_VTLN and dict_speaker:
        print("\n[6/10] VTLN: grid search alpha per speaker ...")
        wav_per_spk = bangun_wav_per_speaker(SAMPLED_ROOT, dict_speaker)
        speaker_alpha_map = cari_vtln_alpha_per_speaker(
            wav_per_spk, gmm_mono, dict_transkrip, lexicon, VTLN_ALPHA_RANGE)
        pd.DataFrame([{"Speaker": spk, "VTLN_alpha": alpha}
                      for spk, alpha in speaker_alpha_map.items()]
                     ).to_csv(f"{DIR_SAVE}/HasilVTLN_alpha.csv", index=False)
        # Rebuild cache MFCC dengan alpha per speaker
        print("  Rebuild MFCC cache dengan VTLN alpha ...")
        cache_mfcc_raw, meta_vtln = build_mfcc_cache(
            SAMPLED_ROOT, dict_transkrip, speaker_alpha_map, dict_speaker)
        cache_mfcc = apply_cmvn_speaker(cache_mfcc_raw, dict_speaker)
        pd.DataFrame(meta_vtln).to_csv(f"{DIR_SAVE}/HasilMFCC_VTLN.csv", index=False)
        print(f"  [VTLN] Cache MFCC diperbarui dengan warp per speaker")
    else:
        print("\n[6/10] VTLN dinonaktifkan (USE_VTLN=False atau tidak ada speaker info) — skip")

    # ── [7] SAT: iterasi fMLLR + GMM ─────────────────────────────────────────
    cache_mfcc_sat = cache_mfcc  # awal: belum di-transform
    fmllr_per_spk  = {}
    gmm_mono_sat   = gmm_mono
    gmm_tri_sat    = gmm_tri

    if USE_SAT and dict_speaker:
        print(f"\n[7/10] SAT: {SAT_ITERS} iterasi fMLLR + GMM ...")
        for sat_it in range(1, SAT_ITERS + 1):
            print(f"\n  === SAT iterasi {sat_it}/{SAT_ITERS} ===")

            # Alignment pada cache saat ini
            align_sat = hitung_alignment_corpus(
                train_ids, cache_mfcc_sat, dict_transkrip, lexicon,
                gmm_mono_sat, gmm_tri_sat)

            # Estimasi fMLLR per speaker
            fmllr_per_spk = estimasi_fmllr_per_speaker(
                cache_mfcc_sat, dict_speaker, gmm_mono_sat,
                alignment_cache=align_sat, audio_ids=train_ids)

            # Terapkan fMLLR ke seluruh cache
            cache_mfcc_sat = apply_fmllr_cache(cache_mfcc, fmllr_per_spk, dict_speaker)

            # Re-train GMM pada fitur fMLLR-transformed
            cache_train_sat = {k: cache_mfcc_sat[k] for k in train_ids if k in cache_mfcc_sat}
            gmm_mono_sat, gmm_tri_sat, frame_per_fonem = latih_gmm_iteratif(
                cache_train_sat, dict_transkrip, lexicon, n_iters=2)
            print(f"  SAT iter {sat_it}: GMM diperbarui ({len(gmm_mono_sat)} mono, {len(gmm_tri_sat)} tri)")

        # Simpan statistik GMM SAT akhir
        gmm_stat_sat = [
            {"Fonem": f, "Frames": frame_per_fonem.get(f, np.empty((0,1))).shape[0],
             "K": g.n_components, "Iter": g.n_iter_done_, "Conv": g.converged_,
             "LL": round(g.log_likelihood_history_[-1], 4) if g.log_likelihood_history_ else None}
            for f, g in gmm_mono_sat.items()
        ]
        pd.DataFrame(gmm_stat_sat).to_csv(f"{DIR_SAVE}/HasilGMM_SAT.csv", index=False)

        # Simpan fMLLR parameters
        fmllr_records = [{"Speaker": spk, "DetA": float(np.linalg.det(tr.A)),
                           "NormB": float(np.linalg.norm(tr.b))}
                          for spk, tr in fmllr_per_spk.items()]
        pd.DataFrame(fmllr_records).to_csv(f"{DIR_SAVE}/HasilfMLLR_params.csv", index=False)
    else:
        print("\n[7/10] SAT dinonaktifkan (USE_SAT=False atau tidak ada speaker info) — skip")

    # Gunakan hasil terbaik (dengan atau tanpa SAT)
    cache_final   = cache_mfcc_sat
    gmm_mono_final = gmm_mono_sat
    gmm_tri_final  = gmm_tri_sat

    # ── [8] Alignment akhir ───────────────────────────────────────────────────
    print("\n[8/10] Alignment akhir pada fitur final ...")
    align_train = hitung_alignment_corpus(
        train_ids, cache_final, dict_transkrip, lexicon, gmm_mono_final, gmm_tri_final)
    align_val   = hitung_alignment_corpus(
        val_ids,   cache_final, dict_transkrip, lexicon, gmm_mono_final, gmm_tri_final)
    print(f"  Train: {len(align_train)} | Val: {len(align_val)}")

    # ── [9] TDNN pada fitur SAT ────────────────────────────────────────────────
    print("\n[9/10] Training TDNN pada fitur fMLLR-SAT ...")
    model = latih_tdnn_frame_level(
        train_ids, val_ids, cache_final, align_train, align_val, DIR_SAVE)

    # ── [10] Decode + evaluasi ────────────────────────────────────────────────
    print("\n[10/10] Dekode dan evaluasi ...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    decoder = TDNNHMMDecoder(
        model=model, lexicon=lexicon, ngram_model=lm,
        gmm_per_fonem=gmm_mono_final, device=device,
        n_hmm_states=N_HMM_STATES, beam_width=BEAM_WIDTH,
        fmllr_per_spk=fmllr_per_spk,      # [NEW-2] test-time adaptation
        dict_speaker=dict_speaker,
    )

    print("  Dekode train ...")
    hasil_train = dekode_semua_audio(cache_mfcc_raw, train_ids, dict_transkrip, decoder)
    print("  Dekode val ...")
    hasil_val   = dekode_semua_audio(cache_mfcc_raw, val_ids,   dict_transkrip, decoder)

    df_tr = pd.DataFrame(hasil_train); df_vl = pd.DataFrame(hasil_val)
    df_tr.to_csv(f"{DIR_SAVE}/HasilWER_Train.csv", index=False)
    df_vl.to_csv(f"{DIR_SAVE}/HasilWER_Val.csv",   index=False)

    r_tr = ringkasan_wer(df_tr, "Train"); r_vl = ringkasan_wer(df_vl, "Val")
    pd.DataFrame([{"set": "train", **r_tr}, {"set": "val", **r_vl}]
                 ).to_csv(f"{DIR_SAVE}/RingkasanWER.csv", index=False)

    # Ringkasan VTLN
    if speaker_alpha_map:
        alpha_vals = list(speaker_alpha_map.values())
        print(f"\n  [VTLN] Distribusi alpha: min={min(alpha_vals):.2f}, "
              f"max={max(alpha_vals):.2f}, mean={np.mean(alpha_vals):.3f}")

    # Ringkasan fMLLR
    if fmllr_per_spk:
        det_vals = [float(np.linalg.det(tr.A)) for tr in fmllr_per_spk.values()]
        print(f"  [SAT] fMLLR det(A): min={min(det_vals):.4f}, "
              f"max={max(det_vals):.4f}, mean={np.mean(det_vals):.4f}")

    print("\n" + "="*60)
    print(f"  HASIL AKHIR")
    print(f"  Train WER : {r_tr.get('wer_corpus','N/A')}%")
    print(f"  Val   WER : {r_vl.get('wer_corpus','N/A')}%  "
          f"(mean: {r_vl.get('wer_mean','N/A')}%)")
    print(f"  VTLN      : {'aktif, alpha per speaker' if USE_VTLN else 'nonaktif'}")
    print(f"  SAT/fMLLR : {'aktif, ' + str(SAT_ITERS) + ' iterasi' if USE_SAT else 'nonaktif'}")
    print(f"  Total     : {(time.time()-t_start)/60:.1f} menit")
    print("="*60)


if __name__ == "__main__":
    main()


[1/10] Membaca CSV ...
  CSV: 1749 audio | 0 speaker unik

[2/10] Ekstraksi MFCC awal (alpha=1.0) ...
  Ekstrak MFCC dari 1749 file ...
    [200/1749] 2s
    [400/1749] 3s
    [600/1749] 5s
    [800/1749] 8s
    [1000/1749] 11s
    [1200/1749] 14s
    [1400/1749] 17s
    [1600/1749] 18s
  Selesai: 1749 audio
  [FIX-v3-2] Tidak ada kolom speaker → skip speaker CMVN

[3/10] Split train/val ...
  Train: 1400 | Val: 349

[4/10] Lexicon dan N-gram LM ...
  Lexicon: 5006 kata | LM vocab: 4274

[5/10] GMM awal (fitur raw, sebelum VTLN/SAT) ...
FASE 1: Iterative monophone GMM

  Iter 1/3 ...
  GMM: 26 model dalam 355.0s
  → 26 model monophone

  Iter 2/3 ...
  GMM: 27 model dalam 224.7s
  → 27 model monophone

  Iter 3/3 ...
    [ 1] SIL :    431 frame, K=8, iter=41, conv=True, LL=-46.81
    [ 2] a   :  91505 frame, K=8, iter=42, conv=True, LL=-51.151
    [ 3] b   :  13613 frame, K=8, iter=23, conv=True, LL=-51.768
    [ 4] c   :  19999 frame, K=8, iter=69, conv=True, LL=-42.76
    [ 5] d   :

In [8]:
# =============================================================================
#  HYBRID TDNN-HMM ASR — Bahasa Jawa — v5 (Semua Bug Diperbaiki)
#
#  PERBAIKAN DARI v4:
#
#  [FIX-Bug-1] normalisasi() menghapus diakritik é è â ê ô û →
#              Regex diganti agar karakter Unicode Jawa dipertahankan.
#              Sebelum: re.sub(r"[^a-z\s]","",...)
#              Sesudah: re.sub dengan allowlist Unicode diacritics
#
#  [FIX-Bug-2] Fonem "eu" dan "ae" tidak bisa diproduksi G2P →
#              Ditambahkan ke _G2P_DIGRAF sehingga G2P bisa memetakannya.
#
#  [FIX-Bug-3] dekode_semua_audio() memakai cache_mfcc_raw (pra-CMVN)
#              padahal TDNN dilatih pada cache_final (pasca fMLLR) →
#              Evaluasi sekarang menggunakan cache_final konsisten.
#
#  [FIX-Bug-4] Path scipy memakai log10, path librosa memakai log →
#              Disamakan ke natural log (np.log) di scipy path.
#
#  [FIX-Bug-5] build_vtln_filterbank() memakai VTLN_FHIGH (7200 Hz)
#              sebagai batas warp, tapi filterbank hanya sampai _FMAX
#              (4000 Hz) → vtln_warp_freq sekarang menerima f_high dari
#              parameter fungsi, bukan dari konstanta global.
#
#  [FIX-Bug-6] FMLLRTransform.fit() menerima fonem_labels tapi tidak
#              memakainya → GMM sekarang difilter berdasarkan fonem aktif
#              pada utterance jika fonem_labels tersedia.
#
#  [FIX-Bug-7] LabelSmoothingNLLLoss mengalikan class weight ke seluruh
#              distribusi smooth (non-standar) → Weight sekarang hanya
#              diterapkan pada kelas target (sesuai PyTorch NLLLoss).
#
#  [FIX-Bug-8] GaussianMixtureModel diam-diam mengurangi n_components
#              tanpa peringatan → Sekarang mencetak WARNING eksplisit.
#
#  [FIX-Bug-9] cari_vtln_alpha_per_speaker() tidak memeriksa apakah
#              ada file valid per speaker → Sekarang ada pengecekan
#              dan peringatan jika tidak ada data valid.
# =============================================================================

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\hybrid\src\TDNN-HMM\out_v5"
)

COL_SPEAKER = "speaker_id"

# ── GMM ───────────────────────────────────────────────────────────────────────
N_COMPONENTS_GMM   = 8
N_ITER_GMM         = 100
TOL_GMM            = 1e-4
MIN_FRAME_GMM      = 20
GMM_EM_ITERS       = 3

# ── TDNN ──────────────────────────────────────────────────────────────────────
TDNN_HIDDEN        = 256
TDNN_DROPOUT       = 0.3
TDNN_WEIGHT_DECAY  = 1e-3
TDNN_LABEL_SMOOTH  = 0.1
TDNN_EPOCHS        = 60
TDNN_LR            = 5e-4
TDNN_LR_MIN        = 1e-6
TDNN_BATCH_SIZE    = 512
TDNN_WARMUP_EPOCH  = 5
TDNN_CONTEXTS      = [
    [-2, -1, 0, 1, 2],
    [-1,  0, 1],
    [-1,  0, 1],
    [-3,  0, 3],
    [-3,  0, 3],
    [-6,  0, 6],
]

# ── HMM / Decode ─────────────────────────────────────────────────────────────
N_HMM_STATES        = 4
BEAM_WIDTH          = 8
AVG_FRAMES_PER_WORD = 30
MIN_FONEM_DUR       = 2
MAX_FONEM_PER_KATA  = 8

# ── LM ───────────────────────────────────────────────────────────────────────
N_GRAM         = 3
VAL_RATIO      = 0.2
SPEAKER_SPLIT  = True

# ── Triphone ─────────────────────────────────────────────────────────────────
USE_TRIPHONE       = True
TRIPHONE_MIN_FRAME = 30

# ── VTLN ─────────────────────────────────────────────────────────────────────
USE_VTLN           = True
VTLN_ALPHA_RANGE   = [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20]
VTLN_FBREAK        = 4000.0   # Hz — titik patah warp piecewise linear
# [FIX-Bug-5] VTLN_FHIGH sekarang hanya dipakai sebagai default fallback;
#             build_vtln_filterbank() meneruskan f_high-nya sendiri.
VTLN_FHIGH         = 8000.0   # diselaraskan dengan Nyquist 16kHz

# ── SAT / fMLLR ──────────────────────────────────────────────────────────────
USE_SAT            = True
FMLLR_ITERS        = 3
FMLLR_MIN_FRAME    = 50
SAT_ITERS          = 2

LOG_ZERO = -1e15
LOG_EPS  = 1e-300
COV_REG  = 1e-6
FEAT_EPS = 1e-8

# =============================================================================
#  IMPORT
# =============================================================================
import os, re, glob, math, time, random, warnings, unicodedata
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    _TORCH = True
except ImportError:
    raise ImportError("PyTorch diperlukan: pip install torch")

try:
    import librosa; _LIBROSA = True
except ImportError:
    _LIBROSA = False

try:
    from scipy.io import wavfile as _wavfile; _SCIPY = True
except ImportError:
    _SCIPY = False

warnings.filterwarnings("ignore")

# =============================================================================
#  FONEM
# =============================================================================
FONEM_JAWA: List[str] = [
    "a","i","u","e","o","eu","ae",
    "p","b","t","d","k","g","dh","th",
    "m","n","ng","ny",
    "s","h","f",
    "j","c",
    "l","r",
    "w","y",
    "SIL",
]
FONEM2ID: Dict[str, int] = {f: i for i, f in enumerate(FONEM_JAWA)}
N_FONEM: int = len(FONEM_JAWA)
SIL_ID: int  = FONEM2ID["SIL"]

# [FIX-Bug-2] Tambahkan "eu" dan "ae" ke digraf agar G2P bisa memetakannya.
# Sebelumnya kedua fonem ini ada di FONEM_JAWA tapi tidak bisa diproduksi,
# menyebabkan 2 neuron output layer TDNN tidak pernah mendapat training signal.
_G2P_DIGRAF = {
    "dh": "dh",
    "th": "th",
    "ng": "ng",
    "ny": "ny",
    "eu": "eu",   # [FIX-Bug-2] ditambahkan
    "ae": "ae",   # [FIX-Bug-2] ditambahkan
}

_kf_cache: Dict[str, Tuple[str,...]] = {}
_norm_cache: Dict[str, str] = {}


def kata_ke_fonem(kata: str) -> Tuple[str,...]:
    if kata in _kf_cache: return _kf_cache[kata]
    fonem, i = [], 0
    while i < len(kata):
        if i+1 < len(kata) and kata[i:i+2] in _G2P_DIGRAF:
            fonem.append(_G2P_DIGRAF[kata[i:i+2]]); i += 2
        elif kata[i] in FONEM2ID:
            fonem.append(kata[i]); i += 1
        else:
            i += 1
    result = tuple(fonem) if fonem else ("SIL",)
    _kf_cache[kata] = result
    return result


# [FIX-Bug-1] Fungsi normalisasi diperbaiki agar TIDAK menghapus diakritik
# Jawa (é, è, â, ê, ô, û). Regex lama [^a-z\s] menghapus semua karakter
# non-ASCII sehingga "anggoné" → "anggon" (kata terpotong).
# Perbaikan: NFC normalization + allowlist Unicode diacritics.
_ALLOWED_DIACRITICS = (
    "\u00e9"  # é
    "\u00e8"  # è
    "\u00e2"  # â
    "\u00ea"  # ê
    "\u00f4"  # ô
    "\u00fb"  # û
    "\u00e0"  # à (untuk kata serapan)
    "\u00f3"  # ó (untuk kata serapan)
)
_RE_NORM = re.compile("[^a-z " + _ALLOWED_DIACRITICS + "]")

def normalisasi(teks: str) -> str:
    """
    [FIX-Bug-1] Normalisasi teks dengan mempertahankan diakritik Jawa.
    Sebelumnya: re.sub('[^a-z ]', '') menghapus e-aksen, dll.
    Sesudahnya: hanya tanda baca dan angka yang dihapus.
    """
    if teks in _norm_cache: return _norm_cache[teks]
    # NFC: satukan karakter + combining mark menjadi satu code point
    t = unicodedata.normalize("NFC", teks.lower().strip())
    # Buang tanda baca & angka, pertahankan huruf latin + diakritik Jawa
    t = _RE_NORM.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    _norm_cache[teks] = t
    return t


def buat_lexicon(transkrip_list):
    lex = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lex:
                lex[kata] = list(kata_ke_fonem(kata))
    return lex


def _bangun_fonem_seq(label, lexicon):
    fs = []
    for kata in normalisasi(label).split():
        fs.extend(lexicon.get(kata, list(kata_ke_fonem(kata))))
    return fs

# =============================================================================
#  TRIPHONE
# =============================================================================
def buat_triphone_seq(fonem_seq):
    if not fonem_seq: return []
    p = ["SIL"] + list(fonem_seq) + ["SIL"]
    return [f"{p[i-1]}-{p[i]}+{p[i+1]}" for i in range(1, len(p)-1)]

def triphone_center(tri):
    try: return tri.split("-")[1].split("+")[0]
    except: return tri

# =============================================================================
#  CSV & SPLIT
# =============================================================================
def baca_transkrip_csv(csv_path, col_speaker=COL_SPEAKER):
    if not os.path.isfile(csv_path): raise FileNotFoundError(csv_path)
    df = None
    for enc in ("utf-8","utf-8-sig","latin-1","cp1252"):
        try: df = pd.read_csv(csv_path, encoding=enc); break
        except: continue
    if df is None: raise ValueError(f"Gagal baca CSV: {csv_path}")
    df.columns = [c.strip() for c in df.columns]
    col_path = next((c for c in ["Path_Data","path_data","path","audio_path","filename"] if c in df.columns), None)
    col_text = next((c for c in ["sentences","sentence","text","transcription","label"] if c in df.columns), None)
    if not col_path: raise ValueError(f"Kolom path tidak ada. Kolom: {list(df.columns)}")
    if not col_text: raise ValueError(f"Kolom teks tidak ada. Kolom: {list(df.columns)}")
    dt, ds = {}, {}
    has_spk = bool(col_speaker and col_speaker in df.columns)
    for _, row in df.iterrows():
        pv = str(row[col_path]).strip(); tv = str(row[col_text]).strip()
        if pv in ("nan","") or tv in ("nan",""): continue
        aid = os.path.splitext(os.path.basename(pv))[0]
        dt[aid] = tv.lower()
        if has_spk: ds[aid] = str(row[col_speaker]).strip()
    n_spk = len(set(ds.values())) if ds else 0
    print(f"  CSV: {len(dt)} audio | {n_spk} speaker unik")
    return dt, ds


def split_by_speaker(audio_ids, dict_speaker, val_ratio=VAL_RATIO):
    ids = list(audio_ids)
    if not dict_speaker:
        random.shuffle(ids); n_val = max(1, int(len(ids)*val_ratio))
        return ids[n_val:], ids[:n_val]
    spk2ids = defaultdict(list)
    for aid in ids: spk2ids[dict_speaker.get(aid,"unknown")].append(aid)
    speakers = list(spk2ids.keys()); random.shuffle(speakers)
    n_val_spk = max(1, int(len(speakers)*val_ratio))
    val_spk = set(speakers[:n_val_spk])
    train_ids, val_ids = [], []
    for spk, sids in spk2ids.items():
        (val_ids if spk in val_spk else train_ids).extend(sids)
    print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")
    return train_ids, val_ids

# =============================================================================
#  MFCC & VTLN
# =============================================================================
_SR = 16000; _N_MFCC = 13; _N_FFT = 512; _HOP_LENGTH = 160
_WIN_LENGTH = 400; _N_MELS = 26; _PRE_EMPH = 0.97
_FMIN = 0.0; _FMAX = 8000.0; _DELTA_W = 2
FEAT_DIM = _N_MFCC * 3


def vtln_warp_freq(freq_hz: float, alpha: float,
                   f_low: float = _FMIN,
                   f_high: float = VTLN_FHIGH,
                   f_break: float = VTLN_FBREAK) -> float:
    """
    VTLN piecewise-linear warping function (Wegmann & Petrone 1996).
    [FIX-Bug-5] f_high sekarang diteruskan dari parameter, bukan dari
    konstanta global VTLN_FHIGH, sehingga konsisten dengan filterbank.
    """
    if alpha == 1.0:
        return freq_hz
    f_break_warped = f_break / alpha
    if freq_hz <= f_break_warped:
        return alpha * freq_hz
    else:
        scale = (f_high - f_break) / max(f_high - f_break_warped, 1e-8)
        return f_break + (freq_hz - f_break_warped) * scale


def build_vtln_filterbank(n_mels: int, n_fft: int, sr: int, alpha: float,
                           f_low: float = _FMIN,
                           f_high: float = _FMAX) -> np.ndarray:
    """
    Bangun mel filterbank dengan VTLN warping.
    [FIX-Bug-5] vtln_warp_freq sekarang menerima f_high dari parameter
    fungsi ini (bukan dari VTLN_FHIGH global), sehingga batas atas warp
    konsisten dengan batas atas filterbank.
    Sebelumnya: vtln_warp_freq(f, alpha, f_low, VTLN_FHIGH, VTLN_FBREAK)
    Sesudahnya: vtln_warp_freq(f, alpha, f_low, f_high,     VTLN_FBREAK)
    """
    hz2mel = lambda f: 1127.0 * np.log1p(f / 700.0)
    mel2hz = lambda m: 700.0 * (np.exp(m / 1127.0) - 1.0)

    mel_pts = np.linspace(hz2mel(f_low), hz2mel(f_high), n_mels + 2)
    hz_pts  = mel2hz(mel_pts)

    # [FIX-Bug-5] Teruskan f_high dari parameter, bukan VTLN_FHIGH global
    hz_warped = np.array([
        vtln_warp_freq(f, alpha, f_low, f_high, VTLN_FBREAK)
        for f in hz_pts
    ])
    hz_warped = np.clip(hz_warped, f_low, sr / 2)

    bins = np.floor((n_fft + 1) * hz_warped / sr).astype(int)
    bins = np.clip(bins, 0, n_fft // 2)

    fb = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)
    for m in range(1, n_mels + 1):
        lo, mid, hi = bins[m-1], bins[m], bins[m+1]
        for k in range(lo, mid):
            fb[m-1, k] = (k - lo) / max(mid - lo, 1)
        for k in range(mid, hi):
            fb[m-1, k] = (hi - k) / max(hi - mid, 1)
    return fb


def apply_cmvn(f: np.ndarray) -> np.ndarray:
    return (f - f.mean(0, keepdims=True)) / (f.std(0, keepdims=True) + FEAT_EPS)


def apply_cmvn_speaker(cache_mfcc: dict, dict_speaker: dict) -> dict:
    if not dict_speaker:
        print("  [INFO] Tidak ada kolom speaker → skip speaker CMVN")
        return cache_mfcc

    unique_spks = set(dict_speaker.values())
    if len(unique_spks) <= 1:
        print(f"  [INFO] Hanya {len(unique_spks)} speaker → skip speaker CMVN")
        return cache_mfcc

    spk_frames = defaultdict(list)
    for aid, feat in cache_mfcc.items():
        spk_frames[dict_speaker.get(aid, "unknown")].append(feat)

    spk_mean, spk_std = {}, {}
    for spk, frames in spk_frames.items():
        all_f = np.vstack(frames)
        spk_mean[spk] = all_f.mean(0)
        spk_std[spk]  = all_f.std(0) + FEAT_EPS

    result = {}
    for aid, feat in cache_mfcc.items():
        spk = dict_speaker.get(aid, "unknown")
        result[aid] = apply_cmvn((feat - spk_mean[spk]) / spk_std[spk])
    print(f"  Speaker CMVN diterapkan: {len(unique_spks)} speaker")
    return result


def _delta_numpy(feat: np.ndarray, w: int = _DELTA_W) -> np.ndarray:
    T = feat.shape[0]; den = 2.0 * sum(n*n for n in range(1, w+1))
    pad = np.pad(feat, ((w,w),(0,0)), mode="edge"); d = np.zeros_like(feat)
    for n in range(1, w+1):
        d += n * (pad[w+n:w+n+T] - pad[w-n:w-n+T])
    return d / den


def _mfcc_librosa_alpha(wav_path: str, alpha: float = 1.0) -> np.ndarray:
    """Ekstraksi MFCC dengan VTLN alpha via librosa."""
    y, _ = librosa.load(wav_path, sr=_SR, mono=True)
    if len(y) == 0: raise ValueError(f"Audio kosong: {wav_path}")
    y = np.append(y[0], y[1:] - _PRE_EMPH * y[:-1])

    stft = np.abs(librosa.stft(y, n_fft=_N_FFT, hop_length=_HOP_LENGTH,
                                 win_length=_WIN_LENGTH, window="hamming")) ** 2

    # [FIX-Bug-5] build_vtln_filterbank dipanggil dengan f_high=_FMAX
    # sehingga batas atas warp konsisten dengan batas filterbank
    fb = build_vtln_filterbank(_N_MELS, _N_FFT, _SR, alpha, _FMIN, _FMAX)
    mel_spec = np.maximum(fb @ stft, 1e-10)

    # Gunakan natural log (np.log) — konsisten dengan scipy path [FIX-Bug-4]
    log_mel = np.log(mel_spec)

    N = _N_MELS
    dct = np.cos(np.pi * np.arange(_N_MFCC)[:, None] *
                 (np.arange(N) + 0.5)[None, :] / N)
    mfcc = (log_mel.T @ dct.T) * np.sqrt(2.0 / N)

    d1 = librosa.feature.delta(mfcc.T, width=_DELTA_W*2+1).T
    d2 = librosa.feature.delta(mfcc.T, order=2, width=_DELTA_W*2+1).T
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


def _mfcc_scipy_alpha(wav_path: str, alpha: float = 1.0) -> np.ndarray:
    """
    Ekstraksi MFCC + VTLN fallback (scipy).
    [FIX-Bug-4] Disamakan ke natural log (np.log) agar konsisten dengan
    path librosa. Sebelumnya memakai log10 yang menghasilkan skala
    berbeda ~2.303× dari path librosa.
    """
    sr, audio = _wavfile.read(wav_path)
    if audio.dtype == np.int16:   audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32: audio = audio.astype(np.float32) / 2147483648.0
    else:                          audio = audio.astype(np.float32)
    if audio.ndim > 1: audio = audio.mean(1)
    if sr != _SR:
        n_new = int(len(audio) * _SR / sr)
        audio = np.interp(np.linspace(0, len(audio)-1, n_new),
                          np.arange(len(audio)), audio)

    y = np.append(audio[0], audio[1:] - _PRE_EMPH * audio[:-1])
    nf = max(1, 1 + (len(y) - _WIN_LENGTH) // _HOP_LENGTH)
    idx = np.arange(_WIN_LENGTH)[None,:] + _HOP_LENGTH * np.arange(nf)[:,None]
    idx = np.clip(idx, 0, len(y)-1)
    fr  = y[idx].astype(np.float32) * np.hamming(_WIN_LENGTH).astype(np.float32)
    mag = np.abs(np.fft.rfft(fr, n=_N_FFT)) ** 2

    # [FIX-Bug-5] f_high=_FMAX diteruskan eksplisit
    fb = build_vtln_filterbank(_N_MELS, _N_FFT, _SR, alpha, _FMIN, _FMAX)

    # [FIX-Bug-4] Ganti log10 → np.log agar konsisten dengan librosa path
    # Sebelumnya: log_mel = np.log10(np.maximum(mag @ fb.T, 1e-10))
    log_mel = np.log(np.maximum(mag @ fb.T, 1e-10))

    N = _N_MELS
    dct = np.cos(np.pi * np.arange(_N_MFCC)[:, None] *
                 (np.arange(N) + 0.5)[None, :] / N)
    mfcc = (log_mel @ dct.T) * np.sqrt(2.0 / N)
    d1 = _delta_numpy(mfcc); d2 = _delta_numpy(d1)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


def extract_features_alpha(wav_path: str, alpha: float = 1.0,
                            cmvn: bool = True) -> np.ndarray:
    if not os.path.isfile(wav_path): raise FileNotFoundError(wav_path)
    feat = (_mfcc_librosa_alpha(wav_path, alpha) if _LIBROSA
            else _mfcc_scipy_alpha(wav_path, alpha))
    return apply_cmvn(feat) if cmvn else feat


def build_mfcc_cache(sampled_root: str, dict_transkrip: dict,
                     alpha_map: Optional[Dict[str, float]] = None,
                     dict_speaker: Optional[Dict[str, str]] = None) -> Tuple[dict, list]:
    wf  = glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True)
    wf += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
    cache, meta = {}, []
    print(f"  Ekstrak MFCC dari {len(wf)} file ...")
    t0 = time.time()
    for idx, wp in enumerate(wf, 1):
        aid = os.path.splitext(os.path.basename(wp))[0]
        if not dict_transkrip.get(aid, ""): continue
        try:
            alpha = 1.0
            if alpha_map and dict_speaker:
                spk = dict_speaker.get(aid, "unknown")
                alpha = alpha_map.get(spk, 1.0)
            feat = extract_features_alpha(wp, alpha=alpha, cmvn=True)
            cache[aid] = feat
            meta.append({"Id": aid, "Frames": feat.shape[0],
                          "Dim": feat.shape[1], "VTLN_alpha": alpha})
        except Exception as e:
            print(f"    [SKIP] {os.path.basename(wp)}: {e}")
        if idx % 200 == 0:
            print(f"    [{idx}/{len(wf)}] {time.time()-t0:.0f}s")
    print(f"  Selesai: {len(cache)} audio")
    return cache, meta


# =============================================================================
#  VTLN ALPHA SEARCH
# =============================================================================
def cari_vtln_alpha_per_speaker(
    wav_paths_per_speaker: Dict[str, List[str]],
    gmm_mono: Dict,
    dict_transkrip: Dict[str, str],
    lexicon: Dict,
    alpha_range: List[float] = VTLN_ALPHA_RANGE,
) -> Dict[str, float]:
    """
    Grid search VTLN alpha per speaker.
    [FIX-Bug-9] Ditambahkan pengecekan speaker tanpa data valid.
    Sebelumnya: speaker dengan 0 file valid langsung default alpha=1.0
    tanpa peringatan, menyulitkan debug.
    """
    if not gmm_mono:
        print("  [VTLN] GMM belum ada → skip search, alpha=1.0 untuk semua")
        return {spk: 1.0 for spk in wav_paths_per_speaker}

    speaker_alpha = {}
    t0 = time.time()
    print(f"  [VTLN] Grid search α ∈ {alpha_range} untuk {len(wav_paths_per_speaker)} speaker ...")

    for spk, paths in wav_paths_per_speaker.items():
        best_alpha, best_ll = 1.0, -np.inf

        for alpha in alpha_range:
            total_ll = 0.0; n_frame = 0
            for wp in paths[:10]:
                aid = os.path.splitext(os.path.basename(wp))[0]
                label = dict_transkrip.get(aid, "")
                if not label: continue
                try:
                    feat = extract_features_alpha(wp, alpha=alpha, cmvn=True)
                except Exception:
                    continue
                # [FIX-Bug-9] Cek feat kosong sebelum hitung LL
                if feat.shape[0] == 0:
                    continue
                fs = _bangun_fonem_seq(label, lexicon)
                if not fs: continue
                fonem_unik = list(set(fs))
                for fonem in fonem_unik:
                    if fonem in gmm_mono:
                        ll_vals = gmm_mono[fonem].log_prob_per_frame(feat)
                        total_ll += float(ll_vals.mean())
                        n_frame  += len(ll_vals)

            if n_frame > 0:
                avg_ll = total_ll / n_frame
                if avg_ll > best_ll:
                    best_ll = avg_ll; best_alpha = alpha

        # [FIX-Bug-9] Peringatan eksplisit jika tidak ada data valid
        if best_ll == -np.inf:
            print(f"  [VTLN WARNING] Speaker {spk}: tidak ada file valid "
                  f"→ default alpha=1.0")

        speaker_alpha[spk] = best_alpha

    n_nondefault = sum(1 for a in speaker_alpha.values() if a != 1.0)
    print(f"  [VTLN] Selesai {time.time()-t0:.1f}s | "
          f"{n_nondefault}/{len(speaker_alpha)} speaker punya α≠1.0")
    return speaker_alpha


def bangun_wav_per_speaker(sampled_root: str,
                            dict_speaker: Dict[str, str]) -> Dict[str, List[str]]:
    wf  = glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True)
    wf += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
    spk2wav = defaultdict(list)
    for wp in wf:
        aid = os.path.splitext(os.path.basename(wp))[0]
        spk = dict_speaker.get(aid, "unknown")
        spk2wav[spk].append(wp)
    return dict(spk2wav)


# =============================================================================
#  GMM
# =============================================================================
class GaussianMixtureModel:
    def __init__(self, n_components=4, n_iter=100, tol=1e-3,
                 reg_covar=COV_REG, random_state=None):
        self.n_components = n_components; self.n_iter = n_iter
        self.tol = tol; self.reg_covar = reg_covar; self.random_state = random_state
        self.weights_ = self.means_ = self.covariances_ = None
        self.converged_ = False; self.n_iter_done_ = 0
        self.log_likelihood_history_: List[float] = []

    def _init_params(self, X):
        rng = np.random.default_rng(self.random_state); K = self.n_components
        idx = rng.choice(X.shape[0], size=K, replace=False)
        self.means_ = X[idx].copy(); self.weights_ = np.full(K, 1.0/K)
        self.covariances_ = np.tile(np.var(X, axis=0) + self.reg_covar, (K, 1))

    def _log_gaussian_all(self, X):
        mu = self.means_; var = self.covariances_
        log_det = np.sum(np.log(var), axis=1)
        diff = X[:,None,:] - mu[None,:,:]
        maha = np.sum(diff**2 / var[None,:,:], axis=2)
        return (-0.5*(X.shape[1]*np.log(2*np.pi)+log_det))[None,:] - 0.5*maha

    def _e_step(self, X):
        lw = np.log(self.weights_+LOG_EPS) + self._log_gaussian_all(X)
        lse = np.max(lw, axis=1, keepdims=True)
        ls = lse.reshape(-1) + np.log(np.sum(np.exp(lw-lse), axis=1))
        return lw - ls[:,None], ls

    def _m_step(self, X, lr):
        r = np.exp(lr); Nk = r.sum(0) + LOG_EPS
        self.weights_ = Nk / X.shape[0]; self.means_ = (r.T @ X) / Nk[:,None]
        diff = X[:,None,:] - self.means_[None,:,:]
        self.covariances_ = (np.sum(r[:,:,None]*diff**2, axis=0)/Nk[:,None]
                             + self.reg_covar)

    def fit(self, X):
        assert X.ndim == 2
        original_k = self.n_components
        self.n_components = min(self.n_components, max(1, X.shape[0]//4))

        # [FIX-Bug-8] Cetak WARNING eksplisit saat n_components dikurangi.
        # Sebelumnya pengurangan terjadi diam-diam tanpa notifikasi apapun.
        if self.n_components < original_k:
            print(f"    [WARNING] GMM: n_components {original_k}→{self.n_components} "
                  f"karena hanya {X.shape[0]} frame tersedia "
                  f"(min 4 frame/komponen)")

        self._init_params(X); self.log_likelihood_history_ = []; prev = -np.inf
        for it in range(self.n_iter):
            lr, ls = self._e_step(X); ll = float(ls.mean())
            self.log_likelihood_history_.append(ll)
            if abs(ll - prev) < self.tol:
                self.converged_ = True; self.n_iter_done_ = it+1; break
            prev = ll; self._m_step(X, lr)
        else:
            self.converged_ = False; self.n_iter_done_ = self.n_iter
        return self

    def log_prob_per_frame(self, X) -> np.ndarray:
        assert X.ndim == 2
        lw = np.log(self.weights_+LOG_EPS) + self._log_gaussian_all(X)
        lse = np.max(lw, axis=1, keepdims=True)
        return lse.reshape(-1) + np.log(np.sum(np.exp(lw-lse), axis=1))


# =============================================================================
#  SAT / fMLLR
# =============================================================================
class FMLLRTransform:
    """
    Feature-space MLLR (fMLLR / CMLLR).
    [FIX-Bug-6] fonem_labels sekarang benar-benar digunakan untuk
    memfilter GMM — hanya GMM fonem yang aktif dalam utterance yang
    digunakan untuk estimasi transform, meningkatkan akurasi fMLLR.
    """

    def __init__(self, feat_dim: int = FEAT_DIM):
        D = feat_dim
        self.D = D
        self.A = np.eye(D, dtype=np.float64)
        self.b = np.zeros(D, dtype=np.float64)

    def transform(self, X: np.ndarray) -> np.ndarray:
        return (X.astype(np.float64) @ self.A.T + self.b).astype(np.float32)

    def fit(self, X: np.ndarray, gmm_per_fonem: Dict,
            fonem_labels: Optional[np.ndarray] = None,
            n_iters: int = FMLLR_ITERS) -> "FMLLRTransform":
        """
        [FIX-Bug-6] fonem_labels sekarang digunakan untuk memfilter GMM.
        Sebelumnya parameter ini dideklarasikan tapi tidak pernah dipakai
        di dalam fungsi, sehingga semua GMM selalu digunakan tanpa
        mempertimbangkan fonem mana yang aktif di utterance ini.

        Jika fonem_labels tersedia: hanya GMM fonem yang muncul di
        alignment yang diikutkan dalam estimasi statistik sufisien.
        Jika fonem_labels None: gunakan semua GMM (perilaku lama).
        """
        if X.shape[0] < FMLLR_MIN_FRAME:
            return self

        D = self.D
        X_d = X.astype(np.float64)
        T = X_d.shape[0]

        # [FIX-Bug-6] Filter GMM berdasarkan fonem aktif di utterance ini
        if fonem_labels is not None:
            active_ids = set(int(fid) for fid in np.unique(fonem_labels)
                             if fid < len(FONEM_JAWA))
            active_names = {FONEM_JAWA[fid] for fid in active_ids}
            gmm_used = {f: g for f, g in gmm_per_fonem.items()
                        if f in active_names and g.means_ is not None}
            if not gmm_used:
                # Fallback ke semua GMM jika filter menghasilkan set kosong
                gmm_used = gmm_per_fonem
        else:
            gmm_used = gmm_per_fonem

        all_means, all_vars, all_weights = [], [], []
        for fonem, gmm in gmm_used.items():
            if gmm.means_ is None: continue
            all_means.append(gmm.means_)
            all_vars.append(gmm.covariances_)
            all_weights.append(gmm.weights_)

        if not all_means:
            return self

        means   = np.vstack(all_means)
        vars_   = np.vstack(all_vars)
        weights = np.concatenate(all_weights)
        weights = weights / (weights.sum() + LOG_EPS)

        for _ in range(n_iters):
            X_hat = X_d @ self.A.T + self.b

            log_gamma = np.log(weights[None,:] + LOG_EPS)
            diff = X_hat[:,None,:] - means[None,:,:]
            log_gamma -= 0.5 * np.sum(
                np.log(vars_[None,:,:] + COV_REG) +
                diff**2 / (vars_[None,:,:] + COV_REG),
                axis=2
            )
            log_gamma -= log_gamma.max(axis=1, keepdims=True)
            gamma = np.exp(log_gamma)
            gamma /= gamma.sum(axis=1, keepdims=True) + LOG_EPS

            X_ext = np.hstack([X_d, np.ones((T, 1))])
            W_new = np.zeros((D, D+1), dtype=np.float64)

            for i in range(D):
                inv_var_i = 1.0 / (vars_[:, i] + COV_REG)
                w_t = gamma @ inv_var_i
                G_i = (X_ext * w_t[:,None]).T @ X_ext
                G_i += np.eye(D+1) * 1e-8
                mu_weighted_i = gamma @ (means[:, i] * inv_var_i)
                k_i = (X_ext * mu_weighted_i[:,None]).sum(axis=0)
                try:
                    W_new[i] = np.linalg.solve(G_i.T, k_i)
                except np.linalg.LinAlgError:
                    W_new[i, :D] = self.A[i]
                    W_new[i, D]  = self.b[i]

            A_new = W_new[:, :D]
            b_new = W_new[:, D]
            if np.linalg.det(A_new) > 1e-10:
                self.A = A_new
                self.b = b_new

        return self


def estimasi_fmllr_per_speaker(
    cache_mfcc: Dict[str, np.ndarray],
    dict_speaker: Dict[str, str],
    gmm_per_fonem: Dict,
    alignment_cache: Optional[Dict[str, np.ndarray]] = None,
    audio_ids: Optional[List[str]] = None,
) -> Dict[str, "FMLLRTransform"]:
    if not dict_speaker:
        print("  [SAT] Tidak ada info speaker → skip fMLLR")
        return {}

    ids = audio_ids or list(cache_mfcc.keys())
    spk2feat: Dict[str, List[np.ndarray]] = defaultdict(list)
    spk2align: Dict[str, List[np.ndarray]] = defaultdict(list)

    for aid in ids:
        feat = cache_mfcc.get(aid)
        if feat is None: continue
        spk = dict_speaker.get(aid, "unknown")
        spk2feat[spk].append(feat)
        if alignment_cache and aid in alignment_cache:
            spk2align[spk].append(alignment_cache[aid])

    fmllr_per_spk: Dict[str, FMLLRTransform] = {}
    print(f"  [SAT] Estimasi fMLLR untuk {len(spk2feat)} speaker ...")
    t0 = time.time()

    for spk, feats in spk2feat.items():
        X_spk = np.vstack(feats)
        aligns = spk2align.get(spk, [])
        # [FIX-Bug-6] labels diteruskan ke fit() dan sekarang benar-benar dipakai
        labels = np.concatenate(aligns) if aligns else None

        tr = FMLLRTransform(feat_dim=X_spk.shape[1])
        tr.fit(X_spk, gmm_per_fonem, fonem_labels=labels)
        fmllr_per_spk[spk] = tr

    n_nontrivial = sum(
        1 for tr in fmllr_per_spk.values()
        if not np.allclose(tr.A, np.eye(tr.D), atol=1e-4)
    )
    print(f"  [SAT] Selesai {time.time()-t0:.1f}s | "
          f"{n_nontrivial}/{len(fmllr_per_spk)} speaker punya transform non-trivial")
    return fmllr_per_spk


def apply_fmllr_cache(
    cache_mfcc: Dict[str, np.ndarray],
    fmllr_per_spk: Dict[str, "FMLLRTransform"],
    dict_speaker: Dict[str, str],
) -> Dict[str, np.ndarray]:
    if not fmllr_per_spk:
        return cache_mfcc
    result = {}
    for aid, feat in cache_mfcc.items():
        spk = dict_speaker.get(aid, "unknown")
        tr  = fmllr_per_spk.get(spk)
        result[aid] = tr.transform(feat) if tr is not None else feat
    return result


def apply_fmllr_single(feat: np.ndarray, spk: str,
                        fmllr_per_spk: Dict[str, "FMLLRTransform"]) -> np.ndarray:
    tr = fmllr_per_spk.get(spk)
    return tr.transform(feat) if tr is not None else feat


# =============================================================================
#  ALIGNMENT — VITERBI VEKTORISASI
# =============================================================================
def alignment_fonem_viterbi(fitur: np.ndarray, fonem_seq: list,
                             gmm_per_fonem: Dict) -> np.ndarray:
    T = fitur.shape[0]; n = len(fonem_seq)
    if n == 0 or T == 0: return np.full(T, SIL_ID, dtype=np.int32)

    skor = np.full((T, n), LOG_ZERO, dtype=np.float64)
    for j, fonem in enumerate(fonem_seq):
        if fonem in gmm_per_fonem:
            skor[:, j] = gmm_per_fonem[fonem].log_prob_per_frame(fitur)
        elif fonem == "SIL":
            skor[:, j] = 0.0

    dp = np.full((T+1, n+1), LOG_ZERO, dtype=np.float64)
    bp = np.zeros((T+1, n+1), dtype=np.int8)
    dp[0, 0] = 0.0

    for t in range(1, T+1):
        e = skor[t-1]
        stay    = dp[t-1, :n] + e
        advance = np.full(n, LOG_ZERO)
        advance[1:] = dp[t-1, :n-1] + e[1:]
        use_adv = advance > stay
        dp[t, :n] = np.where(use_adv, advance, stay)
        bp[t, :n] = np.where(use_adv, 1, 0)
        dp[t, :n] = np.where(dp[t, :n] > LOG_ZERO, dp[t, :n], LOG_ZERO)

    valid = [(dp[T, j], j) for j in range(n) if dp[T, j] > LOG_ZERO]
    j_end = valid[-1][1] if valid else n-1
    path  = np.zeros(T, dtype=int); j = j_end
    for t in range(T, 0, -1):
        path[t-1] = j
        if bp[t, j] == 1: j = max(j-1, 0)

    return np.array([FONEM2ID.get(fonem_seq[k], SIL_ID) for k in path],
                    dtype=np.int32)


def alignment_triphone_viterbi(fitur: np.ndarray, fonem_seq: list,
                                gmm_tri: Dict, gmm_mono: Dict) -> np.ndarray:
    triphone_seq = buat_triphone_seq(fonem_seq)
    T = fitur.shape[0]; n = len(triphone_seq)
    if n == 0 or T == 0: return np.full(T, SIL_ID, dtype=np.int32)

    skor = np.full((T, n), LOG_ZERO, dtype=np.float64)
    for j, tri in enumerate(triphone_seq):
        if tri in gmm_tri:
            skor[:, j] = gmm_tri[tri].log_prob_per_frame(fitur)
        else:
            center = triphone_center(tri)
            if center in gmm_mono: skor[:, j] = gmm_mono[center].log_prob_per_frame(fitur)
            elif center == "SIL":  skor[:, j] = 0.0

    dp = np.full((T+1, n+1), LOG_ZERO, dtype=np.float64)
    bp = np.zeros((T+1, n+1), dtype=np.int8)
    dp[0, 0] = 0.0
    for t in range(1, T+1):
        e = skor[t-1]; stay = dp[t-1, :n] + e
        advance = np.full(n, LOG_ZERO)
        advance[1:] = dp[t-1, :n-1] + e[1:]
        use_adv = advance > stay
        dp[t, :n] = np.where(use_adv, advance, stay)
        bp[t, :n] = np.where(use_adv, 1, 0)
        dp[t, :n] = np.where(dp[t, :n] > LOG_ZERO, dp[t, :n], LOG_ZERO)

    valid  = [(dp[T, j], j) for j in range(n) if dp[T, j] > LOG_ZERO]
    j_end  = valid[-1][1] if valid else n-1
    path   = np.zeros(T, dtype=int); j = j_end
    for t in range(T, 0, -1):
        path[t-1] = j
        if bp[t, j] == 1: j = max(j-1, 0)

    centers = [triphone_center(triphone_seq[k]) for k in path]
    return np.array([FONEM2ID.get(c, SIL_ID) for c in centers], dtype=np.int32)


def kumpulkan_frame_fonem_naif(fitur: np.ndarray, fonem_seq: list) -> dict:
    T, n = fitur.shape[0], len(fonem_seq)
    if n == 0: return {}
    nf = np.ones(n, dtype=int) * (T // n)
    nf[:T - nf.sum()] += 1
    hasil = {}; idx = 0
    for i, f in enumerate(fonem_seq):
        seg = fitur[idx:idx+nf[i]]
        if seg.shape[0] > 0:
            hasil[f] = np.vstack([hasil[f], seg]) if f in hasil else seg.copy()
        idx += nf[i]
    return hasil


def kumpulkan_frame_per_fonem(cache_mfcc, dict_transkrip, lexicon,
                               gmm_mono=None, gmm_tri=None,
                               audio_ids=None) -> Dict[str, np.ndarray]:
    use_tri  = USE_TRIPHONE and gmm_tri  is not None and len(gmm_tri)  > 0
    use_mono = gmm_mono is not None and len(gmm_mono) > 0
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)
    ids = audio_ids if audio_ids else list(cache_mfcc.keys())

    for audio_id in ids:
        fitur = cache_mfcc.get(audio_id); label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label: continue
        fs = _bangun_fonem_seq(label, lexicon)
        if not fs: continue
        if use_tri:
            lbls = alignment_triphone_viterbi(fitur, fs, gmm_tri, gmm_mono or {})
            for fid in np.unique(lbls):
                buf[FONEM_JAWA[fid]].append(fitur[lbls == fid])
        elif use_mono:
            lbls = alignment_fonem_viterbi(fitur, fs, gmm_mono)
            for fid in np.unique(lbls):
                buf[FONEM_JAWA[fid]].append(fitur[lbls == fid])
        else:
            for fonem, arr in kumpulkan_frame_fonem_naif(fitur, fs).items():
                buf[fonem].append(arr)

    return {f: np.vstack(parts) for f, parts in buf.items()}


def kumpulkan_frame_per_triphone(cache_mfcc, dict_transkrip, lexicon,
                                  gmm_mono, audio_ids=None) -> Dict[str, np.ndarray]:
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)
    ids = audio_ids if audio_ids else list(cache_mfcc.keys())
    for audio_id in ids:
        fitur = cache_mfcc.get(audio_id); label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label: continue
        fs   = _bangun_fonem_seq(label, lexicon)
        if not fs: continue
        tris = buat_triphone_seq(fs); T = fitur.shape[0]
        n_tri = len(tris); seg_len = max(1, T // n_tri) if n_tri > 0 else T
        idx = 0
        for k, tri in enumerate(tris):
            end = idx + seg_len if k < n_tri-1 else T
            seg = fitur[idx:end]
            if seg.shape[0] > 0: buf[tri].append(seg)
            idx = end
    return {tri: np.vstack(parts) for tri, parts in buf.items()
            if np.vstack(parts).shape[0] >= TRIPHONE_MIN_FRAME}


def latih_gmm_per_fonem(frame_per_fonem, n_components=N_COMPONENTS_GMM,
                         n_iter=N_ITER_GMM, tol=TOL_GMM, min_frame=MIN_FRAME_GMM,
                         verbose=True) -> Dict:
    gmm = {}; t0 = time.time()
    for i, (fonem, X) in enumerate(sorted(frame_per_fonem.items())):
        if X.shape[0] < min_frame:
            if verbose: print(f"    SKIP {fonem}: {X.shape[0]} frame")
            continue
        g = GaussianMixtureModel(n_components=n_components, n_iter=n_iter, tol=tol,
                                  reg_covar=COV_REG,
                                  random_state=hash(fonem) % 100_000)
        g.fit(X); gmm[fonem] = g
        if verbose:
            ll = round(g.log_likelihood_history_[-1], 3)
            print(f"    [{i+1:2d}] {fonem:4s}: {X.shape[0]:6d} frame, "
                  f"K={g.n_components}, iter={g.n_iter_done_}, "
                  f"conv={g.converged_}, LL={ll}")
    print(f"  GMM: {len(gmm)} model dalam {time.time()-t0:.1f}s")
    return gmm


def latih_gmm_iteratif(
    cache_train: Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon: Dict,
    n_iters: int = GMM_EM_ITERS,
) -> Tuple[Dict, Dict, Dict[str, np.ndarray]]:
    gmm_mono = {}; frame_per_fonem = {}
    print("="*60); print("FASE 1: Iterative monophone GMM")
    for it in range(n_iters):
        print(f"\n  Iter {it+1}/{n_iters} ...")
        frame_per_fonem = kumpulkan_frame_per_fonem(
            cache_train, dict_transkrip, lexicon,
            gmm_mono=gmm_mono if it > 0 else None
        )
        gmm_mono = latih_gmm_per_fonem(frame_per_fonem,
                                        verbose=(it == n_iters-1))
        print(f"  → {len(gmm_mono)} model monophone")

    gmm_tri = {}
    if USE_TRIPHONE:
        print("\n" + "="*60); print("FASE 2: Triphone GMM")
        fpt = kumpulkan_frame_per_triphone(cache_train, dict_transkrip,
                                            lexicon, gmm_mono)
        gmm_tri = latih_gmm_per_fonem(
            fpt, n_components=max(2, N_COMPONENTS_GMM//2),
            min_frame=TRIPHONE_MIN_FRAME, verbose=False
        )
        print(f"  → {len(gmm_tri)} model triphone")

    return gmm_mono, gmm_tri, frame_per_fonem


def hitung_alignment_corpus(audio_ids, cache_mfcc, dict_transkrip,
                             lexicon, gmm_mono, gmm_tri=None) -> Dict[str, np.ndarray]:
    use_tri = USE_TRIPHONE and gmm_tri is not None and len(gmm_tri) > 0
    ac = {}
    for aid in audio_ids:
        fitur = cache_mfcc.get(aid); label = dict_transkrip.get(aid, "")
        if fitur is None or not label: continue
        fs = _bangun_fonem_seq(label, lexicon)
        if not fs: continue
        if use_tri:
            ac[aid] = alignment_triphone_viterbi(fitur, fs, gmm_tri, gmm_mono)
        else:
            ac[aid] = alignment_fonem_viterbi(fitur, fs, gmm_mono)
    return ac


# =============================================================================
#  DATASET
# =============================================================================
class FrameLevelDataset(Dataset):
    def __init__(self, audio_ids, cache_mfcc, alignment_cache):
        fp, lp = [], []
        for aid in audio_ids:
            fitur  = cache_mfcc.get(aid)
            aligns = alignment_cache.get(aid)
            if fitur is None or aligns is None: continue
            fp.append(fitur); lp.append(aligns)
        if fp:
            self.frames = torch.from_numpy(np.vstack(fp))
            self.labels = torch.from_numpy(
                np.concatenate(lp).astype(np.int64))
        else:
            self.frames = torch.zeros((0, FEAT_DIM), dtype=torch.float32)
            self.labels = torch.zeros(0, dtype=torch.long)
        print(f"  Dataset: {len(self.frames):,} frame")

    def __len__(self): return len(self.frames)
    def __getitem__(self, i): return self.frames[i], self.labels[i]


# =============================================================================
#  TDNN
# =============================================================================
class TDNNLayer(nn.Module):
    def __init__(self, input_dim, output_dim, context, dropout=0.0):
        super().__init__()
        self.context = sorted(context)
        self.linear  = nn.Linear(input_dim * len(context), output_dim)
        self.bn      = nn.BatchNorm1d(output_dim)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x):
        sq = x.dim() == 2
        if sq: x = x.unsqueeze(1)
        B, T, _ = x.shape
        cmin = min(self.context); cmax = max(self.context)
        xp = F.pad(x.permute(0,2,1),
                   (max(0,-cmin), max(0,cmax))).permute(0,2,1)
        ctx = [xp[:, c-cmin:c-cmin+T, :] for c in self.context]
        sp  = torch.cat(ctx, dim=-1)
        out = self.linear(sp)
        out = self.bn(out.permute(0,2,1)).permute(0,2,1)
        out = F.relu(out); out = self.drop(out)
        return out.squeeze(1) if sq else out


class TDNNAcousticModel(nn.Module):
    def __init__(self, input_dim=FEAT_DIM, n_classes=N_FONEM,
                 hidden_dim=TDNN_HIDDEN, contexts=None, dropout=TDNN_DROPOUT):
        super().__init__()
        if contexts is None: contexts = TDNN_CONTEXTS
        layers = []; in_dim = input_dim
        for ctx in contexts:
            layers.append(TDNNLayer(in_dim, hidden_dim, ctx, dropout))
            in_dim = hidden_dim
        self.tdnn_layers = nn.ModuleList(layers)
        self.classifier  = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        for layer in self.tdnn_layers: x = layer(x)
        return F.log_softmax(self.classifier(x), dim=-1)

    def get_posteriors_sequence(self, x):
        with torch.no_grad(): return self.forward(x)[0]


# =============================================================================
#  TRAINING
# =============================================================================
class LabelSmoothingNLLLoss(nn.Module):
    """
    [FIX-Bug-7] Class weight sekarang diterapkan hanya pada kelas target,
    bukan pada seluruh distribusi smooth.
    Sebelumnya: loss = -(smooth_dist * log_probs) * weight  (semua kelas)
    Sesudahnya: loss_per_sample * weight[target]  (hanya kelas target)

    Ini sesuai dengan perilaku standar PyTorch nn.NLLLoss(weight=...) di mana
    weight hanya mengatur kontribusi loss dari sample dengan label tertentu.
    """
    def __init__(self, n_classes, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.n_classes = n_classes
        self.weight = weight

    def forward(self, log_probs, targets):
        with torch.no_grad():
            smooth_val  = self.smoothing / self.n_classes
            smooth_dist = torch.full_like(log_probs, smooth_val)
            smooth_dist.scatter_(1, targets.unsqueeze(1),
                                 1.0 - self.smoothing + smooth_val)

        # Hitung loss per sample (sum atas kelas)
        loss_per_sample = -(smooth_dist * log_probs).sum(dim=1)

        # [FIX-Bug-7] Terapkan weight berdasarkan kelas TARGET saja,
        # bukan dikalikan ke seluruh matriks distribusi
        if self.weight is not None:
            target_weight = self.weight[targets]   # (B,) — weight kelas target
            loss_per_sample = loss_per_sample * target_weight

        return loss_per_sample.mean()


def latih_tdnn_frame_level(
    train_ids, val_ids, cache_mfcc, alignment_train, alignment_val,
    dir_save, epochs=TDNN_EPOCHS, lr=TDNN_LR, batch_size=TDNN_BATCH_SIZE,
    hidden_dim=TDNN_HIDDEN, contexts=None, dropout=TDNN_DROPOUT,
    warmup_epochs=TDNN_WARMUP_EPOCH,
) -> TDNNAcousticModel:
    if contexts is None: contexts = TDNN_CONTEXTS
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = os.path.join(dir_save, "model_tdnn_fonem_v5.pt")
    os.makedirs(dir_save, exist_ok=True)

    model = TDNNAcousticModel(input_dim=FEAT_DIM, n_classes=N_FONEM,
                               hidden_dim=hidden_dim, contexts=contexts,
                               dropout=dropout).to(device)

    if os.path.isfile(model_path):
        try:
            model.load_state_dict(torch.load(model_path, map_location=device))
            print(f"  Checkpoint dimuat: {model_path}"); return model
        except Exception:
            os.remove(model_path)

    print("  Membangun dataset ...")
    train_ds = FrameLevelDataset(train_ids, cache_mfcc, alignment_train)
    val_ds   = FrameLevelDataset(val_ids,   cache_mfcc, alignment_val)
    if len(train_ds) == 0:
        print("  [ERROR] Train dataset kosong!"); return model

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                               num_workers=0, pin_memory=False)

    label_counts = torch.bincount(train_ds.labels, minlength=N_FONEM).float()
    label_counts = torch.clamp(label_counts, min=1.0)
    weights = (1.0 / torch.sqrt(label_counts))
    weights = (weights / weights.sum()).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                   weight_decay=TDNN_WEIGHT_DECAY)

    def lr_lambda(ep):
        return float(ep+1) / float(warmup_epochs) if ep < warmup_epochs else 1.0

    warmup_sched = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs - warmup_epochs, 1), eta_min=TDNN_LR_MIN)

    # [FIX-Bug-7] Loss function dengan weight yang diperbaiki
    loss_fn = LabelSmoothingNLLLoss(N_FONEM,
                                     smoothing=TDNN_LABEL_SMOOTH,
                                     weight=weights)

    riwayat = []; best_val = float("inf"); t0 = time.time()
    print(f"\n  Device: {device} | Train: {len(train_ds):,} | Val: {len(val_ds):,}")

    for epoch in range(1, epochs+1):
        model.train(); tl = nc = nt = 0
        for feats, lbls in train_loader:
            feats = feats.to(device); lbls = lbls.to(device)
            optimizer.zero_grad()
            lp = model(feats); loss = loss_fn(lp, lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            tl += loss.item()*len(lbls)
            nc += (lp.argmax(1)==lbls).sum().item()
            nt += len(lbls)

        model.eval(); vl = vc = vt = 0
        with torch.no_grad():
            for feats, lbls in val_loader:
                feats = feats.to(device); lbls = lbls.to(device)
                lp = model(feats)
                vl += loss_fn(lp, lbls).item()*len(lbls)
                vc += (lp.argmax(1)==lbls).sum().item()
                vt += len(lbls)

        if epoch <= warmup_epochs: warmup_sched.step()
        else: cosine_sched.step()

        atr = tl/max(nt,1); avl = vl/max(vt,1)
        actr = 100*nc/max(nt,1); acvl = 100*vc/max(vt,1)
        cur_lr = optimizer.param_groups[0]['lr']
        riwayat.append({"Epoch": epoch, "LR": round(cur_lr,7),
                         "Train Loss": round(atr,5), "Val Loss": round(avl,5),
                         "Train Acc (%)": round(actr,2),
                         "Val Acc (%)": round(acvl,2)})
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs} | "
                  f"Loss {atr:.4f}/{avl:.4f} | "
                  f"Acc {actr:.1f}/{acvl:.1f}% | LR {cur_lr:.2e}")

        if avl < best_val:
            best_val = avl
            torch.save(model.state_dict(), model_path)

    if os.path.isfile(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"  Best model dimuat (val_loss={best_val:.5f})")

    pd.DataFrame(riwayat).to_csv(
        os.path.join(dir_save, "HasilTrainingTDNN_v5.csv"), index=False)
    return model


# =============================================================================
#  DECODER
# =============================================================================
class TDNNHMMDecoder:
    """
    [FIX-Bug-3] Decoder sekarang menerima cache yang sudah dinormalisasi
    (post-CMVN + post-fMLLR) agar konsisten dengan fitur yang dipakai
    saat training TDNN. Test-time fMLLR adaptation tetap tersedia untuk
    speaker yang tidak ada di training.
    """
    def __init__(self, model, lexicon, ngram_model, gmm_per_fonem,
                 device=None, n_hmm_states=N_HMM_STATES, beam_width=BEAM_WIDTH,
                 fmllr_per_spk: Optional[Dict] = None,
                 dict_speaker: Optional[Dict] = None):
        self.model = model; self.lexicon = lexicon; self.ngram = ngram_model
        self.gmm_per_fonem = gmm_per_fonem
        self.device = device or torch.device("cpu")
        self.n_hmm_states = n_hmm_states; self.beam_width = beam_width
        self.fmllr_per_spk = fmllr_per_spk or {}
        self.dict_speaker   = dict_speaker  or {}

        self.rev_lex: Dict[Tuple, List[str]] = defaultdict(list)
        for kata, fonems in lexicon.items():
            self.rev_lex[tuple(fonems)].append(kata)
        lengths = [len(v) for v in lexicon.values()]
        self.avg_fpw = float(np.mean(lengths)) if lengths else 3.0
        self.model.to(self.device); self.model.eval()

    def _adapt_test(self, fitur: np.ndarray, audio_id: str) -> np.ndarray:
        """
        [FIX-Bug-3] Test-time adaptation hanya untuk speaker yang TIDAK ada
        di training (fmllr_per_spk tidak punya entry untuk speaker ini).
        Untuk speaker training, fitur sudah di-transform sebelum masuk
        ke dekode_semua_audio() sehingga tidak perlu re-transform di sini.
        """
        if not self.fmllr_per_spk:
            return fitur
        spk = self.dict_speaker.get(audio_id, None)
        if spk and spk in self.fmllr_per_spk:
            # Speaker training: transform sudah diterapkan di cache_final
            # Tidak perlu transform ulang
            return fitur
        # Speaker baru (tidak ada di training): estimasi fMLLR dari utterance
        if fitur.shape[0] >= FMLLR_MIN_FRAME:
            tr = FMLLRTransform(feat_dim=fitur.shape[1])
            tr.fit(fitur, self.gmm_per_fonem, n_iters=1)
            return tr.transform(fitur)
        return fitur

    def _get_posteriors(self, fitur: np.ndarray) -> np.ndarray:
        t = torch.from_numpy(fitur).unsqueeze(0).to(self.device)
        return self.model.get_posteriors_sequence(t).cpu().numpy()

    def _ctc_collapse(self, fonem_ids: np.ndarray) -> List[str]:
        result = []; i = 0
        while i < len(fonem_ids):
            fid = fonem_ids[i]; j = i+1
            while j < len(fonem_ids) and fonem_ids[j] == fid: j += 1
            dur = j - i
            if dur >= self.n_hmm_states and fid != SIL_ID:
                result.append(FONEM_JAWA[fid])
            i = j
        return result

    def _fonem_ke_kata(self, fonem_seq: list, T: int) -> list:
        n = len(fonem_seq)
        if n == 0: return [[]]
        max_words = max(1, T // AVG_FRAMES_PER_WORD)
        dp = [[] for _ in range(n+1)]; dp[0] = [(0.0, [])]
        for i in range(n):
            if not dp[i]: continue
            for j in range(i+1, min(n+1, i+MAX_FONEM_PER_KATA+1)):
                sub = tuple(fonem_seq[i:j])
                if sub not in self.rev_lex: continue
                for kata in self.rev_lex[sub]:
                    for score, words in dp[i]:
                        if len(words) >= max_words: continue
                        new_words = words + [kata]
                        new_score = score + self.ngram.log_prob(
                            kata, tuple(words[-(self.ngram.n-1):]))
                        dp[j].append((new_score, new_words))
                if len(dp[j]) > self.beam_width * 3:
                    dp[j].sort(key=lambda x: x[0], reverse=True)
                    dp[j] = dp[j][:self.beam_width]
        if dp[n]:
            dp[n].sort(key=lambda x: x[0], reverse=True)
            return [w for _, w in dp[n][:self.beam_width]]
        return [fonem_seq]

    def dekode(self, fitur: np.ndarray, audio_id: str = "") -> list:
        if fitur.shape[0] == 0: return []
        fitur_adapted = self._adapt_test(fitur, audio_id)
        log_post  = self._get_posteriors(fitur_adapted)
        fonem_ids = np.argmax(log_post, axis=1)
        fonem_seq = self._ctc_collapse(fonem_ids)
        if not fonem_seq: return []
        kata_cand = self._fonem_ke_kata(fonem_seq, fitur.shape[0])
        if not kata_cand: return fonem_seq
        return self.ngram.rescore(kata_cand)


# =============================================================================
#  N-GRAM LM
# =============================================================================
class NGramModel:
    def __init__(self, n=3, discount=0.75):
        self.n = n; self.discount = discount
        self.ngram_count    = defaultdict(int)
        self.context_count  = defaultdict(int)
        self.unigram_count  = defaultdict(int)
        self.vocab          = set()
        self.N_total        = 0
        self._ctx_div       = {}

    def _tok(self, s):
        t = normalisasi(s).split()
        return ["<s>"] * (self.n-1) + t + ["</s>"]

    def fit(self, korpus):
        cw = defaultdict(set)
        for s in korpus:
            t = self._tok(s); self.vocab.update(t)
            self.N_total += len(t)
            for tok in t: self.unigram_count[tok] += 1
            for i in range(len(t) - self.n + 1):
                ng = tuple(t[i:i+self.n]); c = ng[:-1]
                self.ngram_count[ng] += 1
                self.context_count[c] += 1
                cw[c].add(ng[-1])
        self._ctx_div = {c: len(w) for c, w in cw.items()}
        return self

    def log_prob(self, kata, ctx):
        ng  = ctx + (kata,); Cng = self.ngram_count.get(ng, 0)
        Cc  = self.context_count.get(ctx, 0)
        Cu  = self.unigram_count.get(kata, 0)
        puni = (Cu + 1) / (self.N_total + len(self.vocab))
        if Cc == 0: return math.log(puni + LOG_EPS)
        lam = self.discount * self._ctx_div.get(ctx, 0) / Cc
        return math.log(max(Cng - self.discount, 0) / Cc + lam*puni + LOG_EPS)

    def score_sequence(self, words):
        t = ["<s>"] * (self.n-1) + words + ["</s>"]; lp = 0.0
        for i in range(self.n-1, len(t)):
            lp += self.log_prob(t[i], tuple(t[i-self.n+1:i]))
        return lp

    def rescore(self, kandidat):
        return max(kandidat, key=self.score_sequence) if kandidat else []


# =============================================================================
#  WER
# =============================================================================
def compute_wer(ref, hyp):
    R, H = len(ref), len(hyp)
    dp = [[0]*(H+1) for _ in range(R+1)]
    ops = [[""]*(H+1) for _ in range(R+1)]
    for i in range(R+1): dp[i][0] = i; ops[i][0] = "D"*i
    for j in range(H+1): dp[0][j] = j; ops[0][j] = "I"*j
    for i in range(1, R+1):
        for j in range(1, H+1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]; ops[i][j] = "M"
            else:
                sub = dp[i-1][j-1]+1; dlt = dp[i-1][j]+1; ins = dp[i][j-1]+1
                best = min(sub, dlt, ins); dp[i][j] = best
                ops[i][j] = ("S" if best==sub else
                              ("D" if best==dlt else "I"))
    S = D = I = 0; i, j = R, H
    while i > 0 or j > 0:
        op = ops[i][j]
        if   op == "M": i-=1; j-=1
        elif op == "S": S+=1; i-=1; j-=1
        elif op == "D": D+=1; i-=1
        elif op == "I": I+=1; j-=1
        else:
            if i > 0: D+=1; i-=1
            else:     I+=1; j-=1
    return {"wer_persen": round((S+D+I)/R*100, 2) if R > 0 else 0.0,
            "S": S, "D": D, "I": I, "N": R}


def dekode_semua_audio(cache_mfcc, audio_ids, dict_transkrip, decoder,
                        verbose=True):
    """
    [FIX-Bug-3] Fungsi ini sekarang menerima cache yang sudah melalui
    transformasi lengkap (CMVN + VTLN + fMLLR). Di main(), dipanggil
    dengan cache_final, bukan cache_mfcc_raw.
    """
    hasil = []; t0 = time.time()
    for idx, aid in enumerate(audio_ids, 1):
        fitur = cache_mfcc.get(aid); label = dict_transkrip.get(aid, "")
        if fitur is None or not label: continue
        ref = normalisasi(label).split()
        if not ref: continue
        try:
            hyp = decoder.dekode(fitur, audio_id=aid)
        except Exception as e:
            if verbose: print(f"    [ERROR] {aid}: {e}"); continue
        wr = compute_wer(ref, hyp)
        hasil.append({"Id": aid,
                       "Referensi": " ".join(ref),
                       "Hipotesis": " ".join(hyp),
                       "WER (%)": wr["wer_persen"],
                       "S": wr["S"], "D": wr["D"],
                       "I": wr["I"], "N": wr["N"]})
        if verbose and idx % 100 == 0:
            print(f"    [{idx}/{len(audio_ids)}] {time.time()-t0:.1f}s")
    return hasil


def ringkasan_wer(df: pd.DataFrame, label: str = "") -> dict:
    if df.empty: return {}
    S = int(df.S.sum()); D = int(df.D.sum())
    I = int(df.I.sum()); N = int(df.N.sum())
    wc = (S+D+I)/N*100 if N > 0 else 100.0
    print(f"\n  === WER: {label} ===")
    print(f"  Audio: {len(df)} | WER corpus: {wc:.2f}% | "
          f"Mean: {df['WER (%)'].mean():.2f}%")
    print(f"  S={S} D={D} I={I} N={N}")
    return {"wer_corpus": round(wc,2),
            "wer_mean":   round(df['WER (%)'].mean(),2),
            "wer_median": round(df['WER (%)'].median(),2),
            "wer_std":    round(df['WER (%)'].std(),2),
            "total_S": S, "total_D": D, "total_I": I, "total_N": N}


# =============================================================================
#  MAIN
# =============================================================================
def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    random.seed(42); np.random.seed(42)
    t_start = time.time()

    # ── [1] Load data ─────────────────────────────────────────────────────────
    print("\n[1/10] Membaca CSV ...")
    dict_transkrip, dict_speaker = baca_transkrip_csv(CSV_PATH)

    # ── [2] MFCC awal (alpha=1.0) ─────────────────────────────────────────────
    print("\n[2/10] Ekstraksi MFCC awal (alpha=1.0) ...")
    cache_mfcc_raw, meta = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    pd.DataFrame(meta).to_csv(f"{DIR_SAVE}/HasilMFCC_raw.csv", index=False)

    cache_mfcc = apply_cmvn_speaker(cache_mfcc_raw, dict_speaker)

    # ── [3] Split ──────────────────────────────────────────────────────────────
    print("\n[3/10] Split train/val ...")
    all_ids = list(cache_mfcc.keys())
    if SPEAKER_SPLIT and dict_speaker:
        train_ids, val_ids = split_by_speaker(all_ids, dict_speaker, VAL_RATIO)
    else:
        random.shuffle(all_ids); n_val = max(1, int(len(all_ids)*VAL_RATIO))
        train_ids, val_ids = all_ids[n_val:], all_ids[:n_val]
        print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")

    # ── [4] Lexicon + LM ──────────────────────────────────────────────────────
    print("\n[4/10] Lexicon dan N-gram LM ...")
    semua_transkrip = list(dict_transkrip.values())
    lexicon = buat_lexicon(semua_transkrip)
    pd.DataFrame([{"Kata": k, "Fonem": " ".join(v)}
                  for k, v in lexicon.items()]
                 ).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)
    lm = NGramModel(n=N_GRAM)
    lm.fit([dict_transkrip[a] for a in train_ids if a in dict_transkrip])
    print(f"  Lexicon: {len(lexicon)} kata | LM vocab: {len(lm.vocab)}")

    # ── [5] GMM awal ──────────────────────────────────────────────────────────
    print("\n[5/10] GMM awal ...")
    cache_train_raw = {k: cache_mfcc[k] for k in train_ids if k in cache_mfcc}
    gmm_mono, gmm_tri, frame_per_fonem = latih_gmm_iteratif(
        cache_train_raw, dict_transkrip, lexicon)

    gmm_stat = [
        {"Fonem": f,
         "Frames": frame_per_fonem.get(f, np.empty((0,1))).shape[0],
         "K": g.n_components, "Iter": g.n_iter_done_,
         "Conv": g.converged_,
         "LL": round(g.log_likelihood_history_[-1], 4)
               if g.log_likelihood_history_ else None}
        for f, g in gmm_mono.items()
    ]
    pd.DataFrame(gmm_stat).to_csv(f"{DIR_SAVE}/HasilGMM_raw.csv", index=False)

    # ── [6] VTLN ──────────────────────────────────────────────────────────────
    speaker_alpha_map = {}
    if USE_VTLN and dict_speaker:
        print("\n[6/10] VTLN: grid search alpha per speaker ...")
        wav_per_spk = bangun_wav_per_speaker(SAMPLED_ROOT, dict_speaker)
        # [FIX-Bug-9] Warning untuk speaker tanpa data valid ada di dalam fungsi
        speaker_alpha_map = cari_vtln_alpha_per_speaker(
            wav_per_spk, gmm_mono, dict_transkrip, lexicon, VTLN_ALPHA_RANGE)
        pd.DataFrame([{"Speaker": spk, "VTLN_alpha": alpha}
                      for spk, alpha in speaker_alpha_map.items()]
                     ).to_csv(f"{DIR_SAVE}/HasilVTLN_alpha.csv", index=False)
        print("  Rebuild MFCC cache dengan VTLN alpha ...")
        cache_mfcc_raw, meta_vtln = build_mfcc_cache(
            SAMPLED_ROOT, dict_transkrip, speaker_alpha_map, dict_speaker)
        cache_mfcc = apply_cmvn_speaker(cache_mfcc_raw, dict_speaker)
        pd.DataFrame(meta_vtln).to_csv(
            f"{DIR_SAVE}/HasilMFCC_VTLN.csv", index=False)
    else:
        print("\n[6/10] VTLN dinonaktifkan — skip")

    # ── [7] SAT ───────────────────────────────────────────────────────────────
    cache_mfcc_sat = cache_mfcc
    fmllr_per_spk  = {}
    gmm_mono_sat   = gmm_mono
    gmm_tri_sat    = gmm_tri

    if USE_SAT and dict_speaker:
        print(f"\n[7/10] SAT: {SAT_ITERS} iterasi fMLLR + GMM ...")
        for sat_it in range(1, SAT_ITERS + 1):
            print(f"\n  === SAT iterasi {sat_it}/{SAT_ITERS} ===")
            align_sat = hitung_alignment_corpus(
                train_ids, cache_mfcc_sat, dict_transkrip, lexicon,
                gmm_mono_sat, gmm_tri_sat)
            # [FIX-Bug-6] alignment diteruskan dan benar-benar dipakai di fit()
            fmllr_per_spk = estimasi_fmllr_per_speaker(
                cache_mfcc_sat, dict_speaker, gmm_mono_sat,
                alignment_cache=align_sat, audio_ids=train_ids)
            cache_mfcc_sat = apply_fmllr_cache(
                cache_mfcc, fmllr_per_spk, dict_speaker)
            cache_train_sat = {k: cache_mfcc_sat[k]
                               for k in train_ids if k in cache_mfcc_sat}
            gmm_mono_sat, gmm_tri_sat, frame_per_fonem = latih_gmm_iteratif(
                cache_train_sat, dict_transkrip, lexicon, n_iters=2)
            print(f"  SAT iter {sat_it}: GMM diperbarui "
                  f"({len(gmm_mono_sat)} mono, {len(gmm_tri_sat)} tri)")

        gmm_stat_sat = [
            {"Fonem": f,
             "Frames": frame_per_fonem.get(f, np.empty((0,1))).shape[0],
             "K": g.n_components, "Iter": g.n_iter_done_,
             "Conv": g.converged_,
             "LL": round(g.log_likelihood_history_[-1], 4)
                   if g.log_likelihood_history_ else None}
            for f, g in gmm_mono_sat.items()
        ]
        pd.DataFrame(gmm_stat_sat).to_csv(
            f"{DIR_SAVE}/HasilGMM_SAT.csv", index=False)
        fmllr_records = [
            {"Speaker": spk,
             "DetA": float(np.linalg.det(tr.A)),
             "NormB": float(np.linalg.norm(tr.b))}
            for spk, tr in fmllr_per_spk.items()
        ]
        pd.DataFrame(fmllr_records).to_csv(
            f"{DIR_SAVE}/HasilfMLLR_params.csv", index=False)
    else:
        print("\n[7/10] SAT dinonaktifkan — skip")

    cache_final    = cache_mfcc_sat
    gmm_mono_final = gmm_mono_sat
    gmm_tri_final  = gmm_tri_sat

    # ── [8] Alignment akhir ───────────────────────────────────────────────────
    print("\n[8/10] Alignment akhir pada fitur final ...")
    align_train = hitung_alignment_corpus(
        train_ids, cache_final, dict_transkrip, lexicon,
        gmm_mono_final, gmm_tri_final)
    align_val   = hitung_alignment_corpus(
        val_ids, cache_final, dict_transkrip, lexicon,
        gmm_mono_final, gmm_tri_final)
    print(f"  Train: {len(align_train)} | Val: {len(align_val)}")

    # ── [9] TDNN ──────────────────────────────────────────────────────────────
    print("\n[9/10] Training TDNN pada fitur fMLLR-SAT ...")
    model = latih_tdnn_frame_level(
        train_ids, val_ids, cache_final,
        align_train, align_val, DIR_SAVE)

    # ── [10] Decode + evaluasi ────────────────────────────────────────────────
    print("\n[10/10] Dekode dan evaluasi ...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    decoder = TDNNHMMDecoder(
        model=model, lexicon=lexicon, ngram_model=lm,
        gmm_per_fonem=gmm_mono_final, device=device,
        n_hmm_states=N_HMM_STATES, beam_width=BEAM_WIDTH,
        fmllr_per_spk=fmllr_per_spk,
        dict_speaker=dict_speaker,
    )

    # [FIX-Bug-3] Gunakan cache_final (bukan cache_mfcc_raw) untuk evaluasi
    # agar domain fitur konsisten dengan yang dipakai saat training TDNN.
    print("  Dekode train ...")
    hasil_train = dekode_semua_audio(
        cache_final, train_ids, dict_transkrip, decoder)
    print("  Dekode val ...")
    hasil_val   = dekode_semua_audio(
        cache_final, val_ids,   dict_transkrip, decoder)

    df_tr = pd.DataFrame(hasil_train); df_vl = pd.DataFrame(hasil_val)
    df_tr.to_csv(f"{DIR_SAVE}/HasilWER_Train.csv", index=False)
    df_vl.to_csv(f"{DIR_SAVE}/HasilWER_Val.csv",   index=False)

    r_tr = ringkasan_wer(df_tr, "Train")
    r_vl = ringkasan_wer(df_vl, "Val")
    pd.DataFrame([{"set": "train", **r_tr}, {"set": "val", **r_vl}]
                 ).to_csv(f"{DIR_SAVE}/RingkasanWER.csv", index=False)

    if speaker_alpha_map:
        alpha_vals = list(speaker_alpha_map.values())
        print(f"\n  [VTLN] Distribusi alpha: min={min(alpha_vals):.2f}, "
              f"max={max(alpha_vals):.2f}, mean={np.mean(alpha_vals):.3f}")

    if fmllr_per_spk:
        det_vals = [float(np.linalg.det(tr.A))
                    for tr in fmllr_per_spk.values()]
        print(f"  [SAT] fMLLR det(A): min={min(det_vals):.4f}, "
              f"max={max(det_vals):.4f}, mean={np.mean(det_vals):.4f}")

    print("\n" + "="*60)
    print(f"  HASIL AKHIR v5")
    print(f"  Train WER : {r_tr.get('wer_corpus','N/A')}%")
    print(f"  Val   WER : {r_vl.get('wer_corpus','N/A')}%  "
          f"(mean: {r_vl.get('wer_mean','N/A')}%)")
    print(f"  VTLN      : {'aktif' if USE_VTLN else 'nonaktif'}")
    print(f"  SAT/fMLLR : {'aktif, ' + str(SAT_ITERS) + ' iterasi' if USE_SAT else 'nonaktif'}")
    print(f"  Total     : {(time.time()-t_start)/60:.1f} menit")
    print("="*60)
    print("\n  Bug yang diperbaiki dari v4:")
    print("  [FIX-Bug-1] normalisasi() mempertahankan diakritik é è â ê ô û")
    print("  [FIX-Bug-2] Fonem eu/ae ditambahkan ke _G2P_DIGRAF")
    print("  [FIX-Bug-3] Evaluasi menggunakan cache_final (bukan cache_mfcc_raw)")
    print("  [FIX-Bug-4] scipy path: log10 → np.log (konsisten dengan librosa)")
    print("  [FIX-Bug-5] VTLN warp: f_high dari parameter, bukan VTLN_FHIGH global")
    print("  [FIX-Bug-6] fMLLR: fonem_labels benar-benar digunakan untuk filter GMM")
    print("  [FIX-Bug-7] LabelSmoothing: weight hanya pada kelas target")
    print("  [FIX-Bug-8] GMM: WARNING eksplisit saat n_components dikurangi")
    print("  [FIX-Bug-9] VTLN: WARNING jika speaker tidak punya file valid")


if __name__ == "__main__":
    main()


[1/10] Membaca CSV ...
  CSV: 1749 audio | 0 speaker unik

[2/10] Ekstraksi MFCC awal (alpha=1.0) ...
  Ekstrak MFCC dari 1749 file ...
    [200/1749] 3s
    [400/1749] 5s
    [600/1749] 8s
    [800/1749] 9s
    [1000/1749] 11s
    [1200/1749] 13s
    [1400/1749] 15s
    [1600/1749] 17s
  Selesai: 1749 audio
  [INFO] Tidak ada kolom speaker → skip speaker CMVN

[3/10] Split train/val ...
  Train: 1400 | Val: 349

[4/10] Lexicon dan N-gram LM ...
  Lexicon: 5070 kata | LM vocab: 4328

[5/10] GMM awal ...
FASE 1: Iterative monophone GMM

  Iter 1/3 ...
  GMM: 28 model dalam 313.1s
  → 28 model monophone

  Iter 2/3 ...
  GMM: 29 model dalam 220.3s
  → 29 model monophone

  Iter 3/3 ...
    [ 1] SIL :    429 frame, K=8, iter=87, conv=True, LL=-44.376
    [ 2] a   : 104935 frame, K=8, iter=63, conv=True, LL=-51.112
    [ 3] ae  :    324 frame, K=8, iter=88, conv=True, LL=-41.063
    [ 4] b   :  17168 frame, K=8, iter=58, conv=True, LL=-46.867
    [ 5] c   :  17556 frame, K=8, iter=72, con